# PEMFC Cathode -- Parameter-Sensitivity Check: RH_in = 1.0 (baseline RH_in = 0.0, fully dry)

**Purpose.** In response to the reviewer critique ("lack of parameter-sensitivity / experimental
validation"), this notebook tests whether, starting from the paper's baseline
($\tau_{brug}=1.1$, $L_x=0.2$mm, $RH_{in}=0.0$) and raising the inlet gas relative humidity from the baseline's fully dry (RH_in=0.0) to 1.0 (fully humidified), the second and third
branches still exist, and if so, how their fold locations shift relative to the baseline.

**Now uses exactly the same recipe as the baseline.** An earlier version of this notebook could
not find `MinusDirection_Fields.ipynb` (the "-" direction eigenvector-perturbation engine that
actually discovered the second and third branches) and substituted the "+" direction instead.
That notebook (`PEMFC_Tau1_1_Lx0_2mm_MinusDirection_Fields_032.ipynb`, a copy already configured
for fraction=0.032) has since been located, so this version reuses the **exact same
(idx, fraction=0.032, direction="-") recipe as the baseline** -- `idx=2` matches the baseline's
second-branch seed, `idx=0` matches the third-branch seed, exactly.

**Where this notebook's code comes from**: the physics/numerics are taken verbatim from
`HighEta_Stability.ipynb` (Stage 1 checkpoint extraction + Part A eigenvalue computation) and
`MinusDirection_Fields_032.ipynb` (Part C, "-" direction perturbation). No new FEM code was
written. What changed is a single `OVERRIDES` dict (`OVERRIDES = dict(tau_brug=1.1, Lx=0.2e-3, RH_in=1.0)`) and the output/Drive-backup
path names, so results never mix with the baseline's.

**Structure**:
- Step 1: fresh continuation from $\eta=0.02$ at the new parameter value, extracting
  $\eta=1.24/1.27/1.28$ checkpoints
- Part A: linear stability (eigenvalue) check at each checkpoint -- the number/direction of
  unstable eigenvectors may differ from the baseline's 3, since the physics has changed. **If
  `idx=2` doesn't exist** (fewer than 3 unstable eigenvalues at this parameter point), the
  idx=2 portions of Part C/D below will fail with a clear error -- that's not a bug, just a
  valid outcome, and those cells can simply be skipped.
- Part B: nonlinear time-marching relaxation (sanity check)
- Part C: perturbs both `idx=0` and `idx=2`, "-" direction, fraction=0.032 -- exactly the
  baseline's second/third-branch seed recipe
- Part D: continues each (idx, fraction=0.032, "-") plateau from $\eta=0.02$ to $1.60$ --
  directly checks whether a second/third-branch-like state exists and where its fold sits

**Outputs**: `/content/pemfc_sensitivity_rh1p0_stability_results` (Stage 1/Part A/Part B), `/content/pemfc_sensitivity_rh1p0_minusdir_results` (Part C),
`/content/pemfc_sensitivity_rh1p0_cont_results` (Part D) -- automatically backed up to Drive at `/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup` /
`/content/drive/MyDrive/pemfc_sensitivity_rh1p0_minusdir_backup` / `/content/drive/MyDrive/pemfc_sensitivity_rh1p0_cont_backup`. These folders never mix with the baseline's results.


## Step 0: Install FEniCSx 0.11


In [ ]:
# FEniCSx on Google Colab (DOLFINx 0.11.x)
# SeoulTechPSE/fenicsx-colab + OpenMPI 5.x (PRRTE) slot patch

from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    print('Cloning fenicsx-colab...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repository already exists')

# OpenMPI 5.x (PRRTE) slot patch
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
print(f'MPI: {N_PROC} slots on {multiprocessing.cpu_count()} CPUs (oversubscribe enabled)')

DOLFINX_VERSION = '0.11'   # pin to '0.10' for legacy notebooks not yet migrated
USE_COMPLEX = False
USE_CLEAN   = False
ENV_NAME    = None

opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
if ENV_NAME:    opts.append(f'--env-name {ENV_NAME}')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")


### Defensive: clear stale FFCx JIT cache

If a `%%fenicsx` cell is ever interrupted mid-compile (stopped manually,
runtime disconnect, etc.), FFCx can leave a half-written cache file behind
that causes every *subsequent* compile of the same form to fail with
`FileExistsError` → `TimeoutError: JIT compilation timed out, probably due
to a failed previous compile` -- even after fixing whatever code issue
triggered the interruption in the first place. Restarting the runtime
clears this too, but re-running this cell is faster and doesn't lose any
other session state.


In [ ]:
import shutil, os

cache_dir = "/root/.cache/fenics"
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Cleared stale JIT cache: {cache_dir}")
else:
    print("No JIT cache present -- nothing to clear.")


## Step 1: Shared physics module

Writes `pemfc_lib.py` to `/content/` -- provides `DEFAULT_PARAMS`, `Psat_atm`,
`_smooth_outflow`, and `_project_scalar_standalone`, which every `%%fenicsx` cell below imports
and builds its own mesh/weak-form/solver from directly (via `build_problem`/`solve_to_eta`
defined locally in each cell). This notebook does **not** use `pemfc_lib.run_case()` -- that
function's fixed-`eta_list` + per-point bisection doesn't fit this notebook's own adaptive,
VI-constrained continuation, which is written fresh in each compute cell instead.

In [ ]:
%%writefile /content/pemfc_lib.py
# ==================================================================
# FEML / PSE Lab -- PEM Fuel Cell Cathode, M^2 Two-Phase Mixture Model
# Reproducing Abdollahzadeh et al., Energy 68 (2014) 478-494, Table 1,
# cross-checked and corrected against Chang, Chen & Teng, J. Power
# Sources 160 (2006) 268-276 (same M^2 family, fully self-consistent
# closed forms for gamma, jl, Leverett J(s), Bruggeman D_eff).
# DOLFINx 0.11  --  shared physics module (imported from %%fenicsx cells)
# ==================================================================
#
# Governing equations solved (2D cathode GDL cross-section, membrane
# at x=0 -> channel land at x=Lx, interdigitated inlet/rib/outlet on
# the channel side, symmetry top/bottom):
#   Mass + Darcy momentum   -> mixture pressure p         (mass cons., Darcy limit)
#   Water mixture mass fraction C_w  -> transport           (Chang Eq. 2/4, alpha=H2O)
#   Gas O2 mass fraction Cg_O2       -> transport            (Chang Eq. 2/4, alpha=O2)
#   Liquid saturation s = s(C_w)      -> algebraic equilibrium
#   Local current density I(y)        -> Tafel kinetics at the catalyst layer
#   Average current density            -> integration along the catalyst layer
#
# --------------------------------------------------------------------
# NUMERICS -- Newton/SNES on a monolithic (p, C_w, Cg_O2) system
# --------------------------------------------------------------------
# Earlier versions of this notebook used segregated Picard iteration:
# solve pressure, then water, then oxygen, each with frozen coefficients
# from the previous iterate; saturation s was tracked as its own field,
# updated from C_w and fed back (lagged) into the other three equations'
# coefficients. Six different remedies -- fixed relaxation, adaptive
# (bold-driver) relaxation on s alone, the same extended to all three
# fields with a proper atol+rtol convergence check, smoothing the s(C_w)
# kink, adaptive eta step-halving continuation, and finally a loosened
# ~1% convergence tolerance -- were tried in sequence and ALL failed
# once saturation became significant (roughly eta >~ 0.33-0.35 V):
# Picard would run to its iteration cap without converging, at every
# eta step in that range, regardless of step size (bisecting the eta
# increment down to ~1/32 of its original size didn't help either,
# ruling out "too large a continuation step" as the cause) or how loose
# the tolerance was made. That pattern -- consistent failure exactly
# where s becomes significant, independent of damping, smoothness, step
# size, or tolerance -- pointed at the SEGREGATED lag between s and C_w
# itself as the structural problem, not any particular numerical
# remedy's tuning.
#
# Fix: eliminate that lag entirely. s is smooth in C_w already (see
# s_smooth_eps below), so it can be substituted as a differentiable UFL
# EXPRESSION of C_w directly everywhere it's used (mixture density,
# viscosity, Darcy coefficient, capillary diffusion, Bruggeman
# diffusivities, the Tafel BC's (1-s) factor) rather than tracked as a
# separate lagged field. grad(s) then also comes for free via UFL's
# chain rule -- no separate L2 projection step needed. Pressure, water,
# and oxygen are combined into ONE mixed function space and solved
# together as a single nonlinear residual via dolfinx's Newton solver
# (PETSc SNES under the hood), which uses the full analytic Jacobian
# (via UFL automatic differentiation) and genuine quadratic convergence
# near a solution, rather than fixed-point iteration with hand-tuned
# damping. eta is a dolfinx Constant updated between solves (the
# symbolic form is built once, not rebuilt per eta), with the same
# continuation-with-bisection-fallback strategy as before for
# robustness across the flooding-onset transition.
#
# CORRECTIONS applied earlier, after cross-checking Chang et al. (2006),
# which gives fully closed, self-consistent forms the first
# (Abdollahzadeh-only) version of this notebook had to guess or got
# wrong (still in effect; only the SOLUTION METHOD changed above, not
# the physics):
#   * Leverett/capillary-pressure scaling is sqrt(eps/K), not sqrt(K*eps)
#     -- confirmed identically by both papers (Eq. 10 / Eq. 17).
#   * The capillary flux jl (Eq. 14/16/20 in Chang) carries NO extra
#     rho_l prefactor.
#   * The advection correction factor gamma^alpha (Eq. 3/9) is *defined*
#     so that gamma^alpha * rho*C^alpha = rho*(lambda_l*Cl^alpha +
#     lambda_g*Cg^alpha) identically (Eq. 24, 30), with Cl^H2O=1,
#     Cg^H2O=min(C_w, Cg_sat) (smoothed), Cl^O2=0.
#   * Species diffusion is weighted by the GAS density rho_g (Chang
#     Eq. 29), not the mixture density.
#   * D_eff uses the Bruggeman tortuosity exponent tau (Chang Eq. 33:
#     D_eff=[eps(1-s)]^tau * D_g); tau=1.5 (literature default, not
#     given numerically in either excerpt).
#   * Porosity eps multiplies only the *diffusive* terms (Chang Eq. 1-2),
#     never the convective flux divergence; standard Darcy superficial
#     velocity u=-(K/mu)grad(p), no extra eps.
#   * Membrane pressure Neumann BC: kappa*dp/dn = N_w - N_O2 (a
#     difference, O2 and H2O flux point in opposite directions through
#     the membrane plane; cross-checked two independent ways).
#   * Species membrane flux signs: water +N_w, oxygen -N_O2 (re-derived
#     via D*dC/dn = -(physical_flux . n_hat), the sign Fick's law
#     introduces that a naive "flux along the outward normal" shortcut
#     misses).
#   * Psat_atm's correlation (Table 2) needs T in CELSIUS, not Kelvin.
#
# SUPG STABILIZATION -- CURRENTLY DISABLED IN THIS (NEWTON) MODULE.
# Was added after comparing Fig. 8's single-phase vs. two-phase local
# current density profiles: the two-phase curve showed severe
# node-to-node oscillation along the channel that the single-phase one
# never did. Root cause diagnosis: the Bruggeman-shrunk diffusivity
# D_eff=[eps(1-s)]^tau*D at high s (e.g. ~1% of its unshrunk value at
# s=0.9) combined with the membrane's advective flux pushes the local
# Peclet number well past where unstabilized P1 Galerkin is known to
# wiggle. Standard SUPG was added to both the water and oxygen
# equations (tau=1/sqrt((2|a|/h)^2+(4D/h^2)^2), h=cell diameter,
# residual=non-conservative a.grad(u)) and initially seemed to cause
# FFCx JIT compilation to hang for the full 3-field coupled Newton
# system -- but this was never isolated against a no-SUPG baseline, so
# it's not confirmed SUPG was actually the cause rather than the base
# coupling already being this expensive to compile. Removed here to
# get a working baseline; if that baseline itself compiles/runs fine,
# SUPG (or a cheaper alternative) should be re-added and re-tested in
# isolation. The Picard companion notebook still has SUPG (its
# per-equation LinearProblem forms are far simpler to compile than one
# monolithic 3-field nonlinear residual+Jacobian) and isn't showing
# this compile-time problem, so it remains the more reliable source
# for two-phase-regime figures in the meantime.
#
# TEMPERATURE-DEPENDENT EXCHANGE CURRENT DENSITY I0(T) (added after
# checking the Fig. 10-12 temperature sweep against the actual paper
# text, which the notebook now has full access to): I0 was held
# constant at 100 A/m^2 across the whole T sweep, giving I *decreasing*
# monotonically with T at fixed eta -- opposite to the paper. Table 2
# lists I0=100 A/m^2 explicitly "(at 333.0 K)", and Sec. 4.3 states
# plainly: "Increase in operating temperature leads to higher fuel cell
# performance due to increase in rates of oxygen reduction reaction
# through enhancement of kinetics of electrochemical reaction (increase
# in Reference current density)" -- i.e. I0 itself is meant to rise with
# T, not stay fixed; held constant, T only enters through the Tafel
# exponential's own 1/T in the denominator, which decreases with T.
# Fixed with a standard Arrhenius correction, I0(T) = I0_ref *
# exp[-(Ea/R)(1/T - 1/T_ref)], anchored at T_ref=333K per Table 2. The
# paper doesn't give its own Ea numerically (an under-specification, like
# GDL thickness or Bruggeman tau elsewhere in this file), so Ea=72.4
# kJ/mol is used -- a standard, widely-cited literature value for Pt ORR
# activation energy in PEMFC modeling. At eta=0.30 V this comfortably
# flips the net trend the right way (Ea=72.4 kJ/mol dominates the Tafel
# exponential's own alpha_c*F*eta~14.5 kJ/mol "reverse" contribution at
# that eta), consistent with the paper's stated direction, though the
# exact quantitative match isn't guaranteed since the true Ea is unknown.
#
# See the "Modeling assumptions" markdown cell in the notebook for what
# is still assumed rather than confirmed (GDL thickness, U_oc/R_contact
# for a full U-I curve, Bruggeman tau's numeric value).
# ==================================================================

import numpy as np
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
import ufl
import os
import time

PROGRESS_LOG = "/content/pemfc_m2_progress.log"


def _progress_log(line):
    """Append a timestamped line to PROGRESS_LOG, fsync'd so it's visible
    to a *separate* cell/process reading the file while this one is still
    running. %%fenicsx appears to buffer a cell's stdout until the whole
    cell finishes (or errors) rather than streaming it -- print(...,
    flush=True) flushes Python's own buffer but not whatever the magic
    itself is doing upstream of that -- so this is the only way to get
    real-time visibility into a long-running run_case() call from another
    cell without waiting for the first one to finish."""
    try:
        with open(PROGRESS_LOG, "a") as f:
            f.write(f"[{time.strftime('%H:%M:%S')}] {line}\n")
            f.flush()
            os.fsync(f.fileno())
    except OSError:
        pass  # never let logging itself break the actual computation


def _classify_branch_state(eta_val, s_max_val, deta_sign):
    """Heuristic, human-readable label for roughly where a point sits on
    the traced branch -- NOT a rigorous branch identifier (there's no way
    to do that without genuine bifurcation-theoretic bookkeeping across
    the whole history), just enough to make checkpoints and snapshots
    recognizable at a glance (e.g. when deciding whether a resumed run
    picked up where you expect, or browsing saved snapshots for an
    interesting one) without manually cross-checking eta/s_max/direction
    every time."""
    if s_max_val < 0.05:
        phase = "near-single-phase"
    elif s_max_val < 0.3:
        phase = "light flooding"
    elif s_max_val < 0.6:
        phase = "moderate flooding"
    elif s_max_val < 0.9:
        phase = "severe flooding"
    else:
        phase = "near-saturated (s~cap)"
    direction = "eta increasing" if deta_sign > 0 else "eta decreasing"
    return f"{phase}, {direction} (eta={eta_val:.4f}, s_max={s_max_val:.3f})"


def _jacobian_singular_values(problem, n_smallest=6, n_largest=3):
    """Extract the SNES's current Jacobian (the state after its most
    recent solve() call, converged or not) and return its singular value
    spectrum, both raw and after Jacobi (diagonal) equilibration.

    Direct diagnostic for whether a hard region is a genuine fold
    bifurcation: the defining mathematical signature of a fold (turning
    point) is that the Jacobian of the governing system becomes singular
    there -- if the smallest singular value collapses toward zero (while
    the largest stays bounded, i.e. the condition number blows up
    specifically as eta approaches the problem region) that's a fold, not
    merely a hard but regular nonlinearity.

    The RAW spectrum from this monolithic (p, C_w, Cg_O2) system is not
    directly usable for that, though: pressure (~1e5 Pa) and the mass
    fractions (~0.01-1) differ by 5-7 orders of magnitude in natural
    scale, and Dirichlet BC rows are set to exact identity -- both
    contribute a large, roughly CONSTANT (eta-independent) baseline
    condition number that has nothing to do with proximity to a fold and
    would swamp any genuine fold signature if not removed. Jacobi
    (diagonal) equilibration -- rescaling each row/column by
    1/sqrt(|diagonal entry|), a standard, cheap technique -- removes most
    of that scale artifact; the SCALED smallest singular value's trend
    across eta is what actually reflects the underlying system's
    conditioning.

    The system here (~4500x4500 for the default mesh) is small enough
    that a one-off dense SVD (a few seconds) is a reasonable diagnostic
    cost, even though it would be far too expensive to run on every
    iteration of a production sweep."""
    import scipy.sparse as sp
    J_mat = problem.solver.getJacobian()[0]  # (J, P[, callback]) depending on petsc4py version
    n = J_mat.getSize()[0]
    indptr, indices, data = J_mat.getValuesCSR()
    J_dense = sp.csr_matrix((data, indices, indptr), shape=(n, n)).toarray()

    diag = np.abs(np.diag(J_dense))
    scale = 1.0 / np.sqrt(np.maximum(diag, 1e-30))
    J_scaled = (J_dense * scale[None, :]) * scale[:, None]

    svals_raw = np.sort(np.linalg.svd(J_dense, compute_uv=False))
    svals_scaled = np.sort(np.linalg.svd(J_scaled, compute_uv=False))
    return (svals_raw[:n_smallest], svals_raw[-n_largest:],
            svals_scaled[:n_smallest], svals_scaled[-n_largest:])
    return svals[:n_smallest], svals[-n_largest:]

DEFAULT_PARAMS = dict(
    # --- Table 2 (Abdollahzadeh base validation case, T = 330 K) ---
    F=96485.0, R=8.314,
    K_perm=1.2e-12, eps_p=0.30, p_in=1.007e5, p_out=1.0e5,
    mu_g=2.03e-5, D_O2_g=1.775e-5, D_H2O_g=2.56e-5,
    I0=100.0, alpha_c=0.5, alpha_w=0.5, rho_l=1000.0,
    T_ref_I0=330.0,      # Anchored at the actual base-case/validation
                          # operating temperature (Table 2: "Operating
                          # temperature T K 330"), NOT the "333.0 K" label
                          # on the I0 row -- that's most likely just citing
                          # the source measurement's condition, not a
                          # correction meant to apply within the 330K base
                          # case itself. Anchoring here keeps I0=100 exactly
                          # at the validated default (leaving Fig. 4's
                          # calibration and every other already-checked
                          # figure undisturbed) while still giving the
                          # correct increasing trend once T_cell is swept
                          # away from 330K in the Fig. 10-12 study.
    Ea_orr=72400.0,      # J/mol -- ORR activation energy on Pt (72.4 kJ/mol,
                          # a standard literature value; not given explicitly
                          # in the excerpted paper text, but the paper's own
                          # Sec. 4.3 explicitly says higher T raises I0
                          # ("increase in Reference current density") --
                          # holding I0 constant across a T sweep, as earlier
                          # versions of this notebook did, produces the
                          # opposite (decreasing) trend, since T then only
                          # enters through the Tafel exponential's own
                          # denominator
    M_H2O=18.0e-3, M_O2=0.032, M_N2=0.028, mu_l=3.15e-4,
    RH_in=0.0, T_cell=330.0, C_O2_ref=40.0,
    theta_c_deg=91.0, sigma_st=0.0625,
    X_O2_in=0.21, X_N2_in=0.79,
    # --- Table 3 (interdigitated geometry) --------------------------
    w_ch=0.8e-3, w_rb=1.2e-3, Lx=0.30e-3,
    # --- closures confirmed / added after Chang et al. (2006) -------
    n_corey=3.0,        # Eq. 12-13 -- confirmed identical in both papers
    tau_brug=1.5,       # Eq. 33 Bruggeman exponent -- literature default,
                         # not given numerically in either excerpt
    s_max=0.99,
    s_smooth_eps=0.02,  # smoothing width (s-units) for the two-phase onset;
                         # also what lets s be substituted as a differentiable
                         # UFL expression of C_w for the Newton solve
    single_phase=False,   # if True, s is forced to 0 everywhere (reproduces
                            # the paper's "single-phase flow" comparison curves)
    # --- numerics -----------------------------------------------------
    nx=24, ny=60,
    newton_rtol=1e-7, newton_atol=1e-9, newton_max_it=50,
)


def Psat_atm(T):
    """Saturation pressure of water, atm (Table 2 correlation). T in CELSIUS."""
    return 10.0 ** (-2.1794 + 0.02953 * T - 9.1837e-5 * T**2 + 1.4454e-7 * T**3)


def _smooth_outflow(x, eps=1e-5):
    """Smooth (regularized) approximation to max(x, 0), used for clipping
    the outlet boundary's advective flux to its outflow-only part (see
    the outlet-BC stabilization discussion at each F_water/F_oxy
    definition). A HARD ufl.max_value(x, 0.0) was tried first and works
    fine for warm-started continuation (small Newton steps near an
    already-good solution rarely cross the kink), but was confirmed to
    cause SNES divergence for a COLD START from a naive/inlet-like
    initial guess (large early Newton steps repeatedly hit the
    non-smooth kink) -- exactly the situation at the very start of every
    continuation run (eta_start, no checkpoint yet). This smooth version
    removes that kink while remaining numerically equivalent to the hard
    clip away from x=0 (to within eps)."""
    return 0.5 * (x + ufl.sqrt(x**2 + eps**2))


def J_leverett(s, hydrophobic=True):
    """Leverett J-function (reference only -- the Newton solve builds dJds
    directly as a flat UFL expression inline inside run_case). hydrophobic branch
    (theta_c>90 deg, Chang Eq.19 / Abdollahzadeh Eq.10):
    J(s)=1.417s-2.120s^2+1.263s^3. hydrophilic branch (theta_c<90 deg,
    Chang Eq. 18): J(s)=1.417(1-s)-2.120(1-s)^2+1.263(1-s)^3."""
    if hydrophobic:
        return 1.417 * s - 2.120 * s**2 + 1.263 * s**3
    u = 1.0 - s
    return 1.417 * u - 2.120 * u**2 + 1.263 * u**3


def dJ_leverett(s, hydrophobic=True):
    if hydrophobic:
        return 1.417 - 4.240 * s + 3.789 * s**2
    u = 1.0 - s
    return -(1.417 - 4.240 * u + 3.789 * u**2)


def extract_line(coords, arrays, x_target=0.0, tol=1e-7):
    """Pull out a profile along the vertical line x=x_target (default: the
    membrane), sorted by y. `arrays` is a dict of {name: array-over-all-dofs};
    returns a dict with 'y' plus the same keys, restricted/sorted to that line.
    Pure NumPy -- works equally in a %%fenicsx cell or a later plotting cell."""
    x = coords[:, 0]
    mask = np.abs(x - x_target) < tol
    y_sel = coords[mask, 1]
    order = np.argsort(y_sel)
    out = {"y": y_sel[order]}
    for name, arr in arrays.items():
        out[name] = np.asarray(arr)[mask][order]
    return out


def extract_interface(coords, s_arr, threshold=0.02):
    """For each distinct y (along-channel position), find the x (GDL-
    thickness position, membrane=0) where saturation first drops below
    `threshold` moving away from the membrane -- i.e. the boundary between
    the two-phase and single-phase zones (paper Figs. 12/17/20/25). Returns
    (y_values, x_interface). x_interface=0 means no two-phase zone at that y;
    x_interface=Lx means two-phase all the way to the channel."""
    x, y = coords[:, 0], coords[:, 1]
    y_unique = np.unique(np.round(y, 12))
    xi = np.zeros_like(y_unique)
    for i, yv in enumerate(y_unique):
        row = np.abs(y - yv) < 1e-9
        xr, sr = x[row], s_arr[row]
        order = np.argsort(xr)
        xr, sr = xr[order], sr[order]
        below = sr < threshold
        if below[0]:
            xi[i] = 0.0
        elif not np.any(below):
            xi[i] = xr[-1]
        else:
            idx = int(np.argmax(below))
            x0, x1, s0, s1 = xr[idx - 1], xr[idx], sr[idx - 1], sr[idx]
            xi[i] = x1 if s1 == s0 else x0 + (threshold - s0) * (x1 - x0) / (s1 - s0)
    return y_unique, xi


def run_parametric_study(param_key, values, eta_list=None, profile_eta=0.55,
                          verbose=True, **base_overrides):
    """Run run_case once per value of `param_key` on top of `base_overrides`,
    collecting both the full eta sweep (I_avg, s_max) and a field snapshot
    near `profile_eta` (membrane s(y)/I(y)/Cg_O2(y) profiles and the
    two-phase interface position x(y)). Used by every parametric-study
    figure in the paper (temperature, humidity, porosity, pressure
    difference, GDL thickness, permeability, contact angle) so the sweep
    logic lives in exactly one place.

    Returns {value: {eta, I_avg, s_max, iters, membrane_y, membrane_s,
    membrane_I, membrane_CgO2, interface_y, interface_x}}.
    """
    results = {}
    for v in values:
        if verbose:
            print(f"\n--- {param_key} = {v} ---", flush=True)
        overrides = dict(base_overrides)
        overrides[param_key] = v
        r = run_case(overrides=overrides, eta_list=eta_list, verbose=verbose,
                      save_fields_at=(profile_eta,))
        tag = f"eta{profile_eta:.3f}"
        fld = r["fields"].get(tag)
        if fld is None and r["fields"]:
            fld = next(iter(r["fields"].values()))   # fall back to nearest match found
        fld = fld or {}
        results[v] = dict(
            eta=r["eta"], I_avg=r["I_avg"], s_max=r["s_max"], iters=r["iters"],
            membrane_y=fld.get("membrane_y"), membrane_s=fld.get("membrane_s"),
            membrane_I=fld.get("membrane_I"), membrane_CgO2=fld.get("membrane_CgO2"),
            interface_y=fld.get("interface_y"), interface_x=fld.get("interface_x"))
    return results


def save_study_npz(path, param_key, values, results):
    """Flatten a run_parametric_study() result dict into a single .npz
    (npz only supports a flat name->array mapping). Keys are indexed by
    position (v0__eta, v0__I_avg, ...) rather than by the raw float value,
    to sidestep float-to-string formatting ambiguity; `values` gives the
    index -> parameter-value mapping. See load_study_npz (duplicated,
    NumPy-only, in the plotting cells -- they run outside the FEniCSx conda
    env and can't import this dolfinx-dependent module)."""
    payload = {"param_key": np.array([param_key]), "values": np.asarray(values, dtype=float)}
    for i, v in enumerate(values):
        for k, arr in results[v].items():
            if arr is not None:
                payload[f"v{i}__{k}"] = np.asarray(arr)
    np.savez(path, **payload)


def run_case(overrides=None, eta_list=None, verbose=True, save_fields_at=(0.55,),
             jacobian_diagnostics=False):
    """Solve the M^2 two-phase PEMFC cathode model over an eta sweep, using
    a monolithic Newton solve on a mixed (p, C_w, Cg_O2) space at each eta
    (saturation s embedded as a smooth UFL expression of C_w -- see the
    module header for why this replaced segregated Picard iteration).

    Parameters
    ----------
    overrides : dict, optional overrides on top of DEFAULT_PARAMS.
    eta_list  : array of overpotentials (V) to sweep, default 18 pts.
    save_fields_at : eta values (nearest match) to snapshot full 2D
        fields for, in addition to the polarization data.
    jacobian_diagnostics : if True, after every Newton solve attempt
        (converged or not), extract the SNES's current Jacobian and log
        its smallest few and largest singular values -- a direct,
        principled way to check whether a stuck/slow region is a genuine
        fold (turning point) bifurcation (smallest singular value -> 0,
        i.e. Jacobian going singular) versus merely a hard-but-regular
        nonlinear region (Jacobian stays well-conditioned, Newton is just
        slow/needs a better initial guess). Adds real cost (a dense SVD
        of the ~4500x4500 system per solve, several seconds each) so it's
        off by default -- meant for targeted diagnostic sweeps, not the
        production parametric studies.

    Returns
    -------
    dict with 'eta', 'I_avg' (A/m^2), 's_max', 'iters' (Newton iteration
    count actually used, for diagnostics), and 'fields' (a dict
    eta_tag -> {coords, p, Cw, CgO2, s, ...}).
    """
    P = dict(DEFAULT_PARAMS)
    try:
        open(PROGRESS_LOG, "w").close()  # fresh log for this call
    except OSError:
        pass
    _progress_log(f"run_case() started (overrides={overrides}, "
                  f"eta_list size={None if eta_list is None else len(eta_list)})")
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    # Arrhenius-corrected exchange current density (paper Sec. 4.3: higher T
    # raises I0 via faster ORR kinetics -- "increase in Reference current
    # density"). I0_ref is anchored at T_ref_I0=330K (the base-case
    # operating temperature), not the "333.0K" label on Table 2's I0 row.
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]
    newton_rtol, newton_atol, newton_max_it = P["newton_rtol"], P["newton_atol"], P["newton_max_it"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])

    ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    # Scalar space (for output/projection only) + mixed space (for the
    # coupled Newton solve).
    V = fem.functionspace(domain, ("Lagrange", 1))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)

    # Non-dimensionalized pressure unknown: p_hat = (p - p_out)/P_scale,
    # P_scale a fixed O(driving-pressure-difference) constant (NOT
    # recomputed per case, so parametric sweeps over p_in stay on a
    # consistent scale). Raw p (~1e5 Pa) mixed with C_w/Cg_O2 (~O(1) mass
    # fractions) in one monolithic Jacobian meant the pressure block's
    # entries were ~5 orders of magnitude larger than the others purely
    # from unit choice -- confirmed by direct Jacobian SVD diagnostics
    # (see module header) to dominate the *raw* condition number
    # (~1e9-1e10 even in the easy, far-from-any-fold regime); this
    # doesn't touch the genuine fold near the flooding-onset transition
    # (that's a real feature of the underlying system, not a scaling
    # artifact -- confirmed by the SAME diagnostics showing the
    # *equilibrated* smallest singular value itself, not just the raw
    # condition number, collapsing there), but it does remove the
    # constant baseline ill-conditioning everywhere else.
    P_scale = 1000.0  # Pa, O(the typical driving pressure difference)
    p_sol = p_out + P_scale * p_hat_sol  # reconstruct physical pressure; everything
                                          # downstream (u_darcy, F_pres, field saving)
                                          # references p_sol exactly as before

    # --- inlet composition (RH_in-dependent, dry-gas fractions renormalised) ---
    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2   # dry-basis MW for the saturation formula

    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    # --- BCs on the mixed space ---
    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    # Scalar-valued BCs on a mixed-space subspace use the single-space dof
    # form (W.sub(i)) -- the (subspace, collapsed_subspace) tuple form is
    # only needed when the BC value is itself a Function on the collapsed
    # subspace, which isn't the case here (plain constants).
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]
    membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))

    # --- saturation as a smooth (differentiable) UFL expression of C_w ---
    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw   # a genuine UFL zero expression (grad(0.0) would fail; grad(0*Cw) is fine)
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0

    # --- assemble the coupled residual (see module header for the
    # derivation/sign history of every term; only the LAG between s and
    # C_w was eliminated here, the physics itself is unchanged). Every
    # s-dependent quantity is computed EXACTLY ONCE below and reused by
    # plain variable reference from then on -- earlier drafts routed
    # these through nested closures (e.g. Gamma_e() called Dc_e() again
    # internally even though Dc was already computed at top level, and
    # Dc_e() itself re-derived lambda_l/lambda_g/mu_mix/rho_mix from
    # scratch instead of reusing the top-level values), which silently
    # blew up the symbolic expression tree by ~10-20x before FFCx
    # compiled it and UFL symbolically differentiated it for the
    # Jacobian -- almost certainly why the solve hung for tens of
    # minutes instead of the few seconds this mesh size should take.
    s = s_expr(Cw_sol)

    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix

    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)

    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)   # comes for free via UFL's chain rule -- no projection needed

    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)

    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    Weff_w = lam_l + lam_g * CgH2O_local

    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix

    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds(TAG_MEMBRANE))

    # --- SUPG stabilization: DISABLED for now -----------------------------
    # Was added to fix node-to-node oscillation seen in the two-phase
    # regime (Bruggeman-shrunk diffusivity + membrane flux pushing the
    # local Peclet number high), but JIT compilation hung even after
    # simplifying it to the cheap non-conservative a.grad(u) form -- and
    # that was never isolated against a no-SUPG baseline to confirm SUPG
    # itself was the cause (vs. the base 3-field coupling already being
    # this expensive to compile). Removed for now to get a working
    # baseline; re-add (or find a cheaper alternative) once that's
    # confirmed. The Picard companion notebook still has SUPG and isn't
    # showing this compile-time problem, so it remains the more reliable
    # source for two-phase-regime figures in the meantime.
    # ----------------------------------------------------------------------

    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds(TAG_OUTLET)
               - N_w_expr * v_cw * ds(TAG_MEMBRANE))

    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds(TAG_MEMBRANE))

    F_total = F_pres + F_water + F_oxy

    if verbose:
        print("  [building nonlinear form + JIT-compiling residual/Jacobian -- can take a "
              "while the first time; no further output until this finishes]", flush=True)
    _progress_log("building nonlinear form + starting JIT compilation")
    snes_petsc_options = {
        "snes_type": "newtonls",
        "snes_rtol": newton_rtol,
        "snes_atol": newton_atol,
        "snes_max_it": newton_max_it,
        "snes_error_if_not_converged": False,   # handle non-convergence ourselves (bisection fallback)
        "ksp_type": "preonly",
        "pc_type": "lu",
    }
    # Explicit quadrature degree cap: sqrt/min_value/max_value in s_expr()
    # (and, previously, in SUPG's tau) are non-polynomial, so FFCx's
    # automatic degree-estimation heuristic has no exact answer to fall
    # back on and can pick an excessively high degree -- ballooning both
    # the generated C code size and its compile time, independent of how
    # symbolically "redundant" the UFL expression tree itself is. Capping
    # it explicitly avoids that; degree 4 is more than enough accuracy
    # for smooth (if non-polynomial) coefficients on P1 elements.
    form_compiler_options = {"quadrature_degree": 4}
    problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                petsc_options_prefix="pemfc_newton_",
                                form_compiler_options=form_compiler_options)
    if verbose:
        print("  [JIT compilation done -- starting eta sweep]", flush=True)
    _progress_log("JIT compilation done -- starting eta sweep")

    # --- initial guess: uniform inlet-like state ---
    w.sub(0).interpolate(lambda x: np.full(x.shape[1], (p_in - p_out) / P_scale))
    w.sub(1).interpolate(lambda x: np.full(x.shape[1], Y_H2O_in))
    w.sub(2).interpolate(lambda x: np.full(x.shape[1], Y_O2_in))
    w.x.scatter_forward()

    def project_scalar(expr, prefix):
        u_tr, vt = ufl.TrialFunction(V), ufl.TestFunction(V)
        a_p = u_tr * vt * dx
        L_p = expr * vt * dx
        prob = LinearProblem(a_p, L_p, bcs=[],
                              petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
                              petsc_options_prefix=prefix)
        return prob.solve().x.array.copy()

    def checkpoint(): return w.x.array.copy()
    def restore(ckpt): w.x.array[:] = ckpt

    def solve_at_eta(eta_target, eta_from, max_bisect=4):
        """Newton-solve at eta_target, warm-started from the current state
        (converged at eta_from). Falls back to a bisected intermediate eta
        (and retries from there) if Newton fails to converge -- Newton can
        still fail from a poor-enough initial guess even though it no
        longer suffers the segregated-lag oscillation; this keeps the same
        continuation robustness the Picard version had."""
        pending = [eta_target]
        eta_prev = eta_from
        n_bisect_total = 0
        result = None
        while pending:
            eta_now = pending[-1]
            ckpt = checkpoint()
            eta_const.value = eta_now
            _progress_log(f"attempting Newton solve at eta={eta_now:.4f} ...")
            try:
                problem.solve()
                converged_reason = problem.solver.getConvergedReason()
                n_its = problem.solver.getIterationNumber()
                converged = converged_reason > 0
                _progress_log(f"  -> solve returned: converged={converged}, n_its={n_its}")
                if jacobian_diagnostics:
                    try:
                        sv_small_raw, sv_large_raw, sv_small_sc, sv_large_sc = \
                            _jacobian_singular_values(problem)
                        cond_raw = sv_large_raw[-1] / max(sv_small_raw[0], 1e-300)
                        cond_sc = sv_large_sc[-1] / max(sv_small_sc[0], 1e-300)
                        _progress_log(
                            f"  Jacobian @ eta={eta_now:.4f}  "
                            f"RAW: smallest={np.array2string(sv_small_raw, precision=3)} "
                            f"largest={np.array2string(sv_large_raw, precision=3)} "
                            f"cond~{cond_raw:.3e}  ||  "
                            f"SCALED (equilibrated, the trustworthy one): "
                            f"smallest={np.array2string(sv_small_sc, precision=3)} "
                            f"largest={np.array2string(sv_large_sc, precision=3)} "
                            f"cond~{cond_sc:.3e}")
                    except Exception as e:
                        _progress_log(f"  Jacobian diagnostic failed: {e}")
            except RuntimeError as e:
                n_its, converged = newton_max_it, False
                _progress_log(f"  -> RuntimeError: {e}")
            if converged or n_bisect_total >= max_bisect:
                pending.pop()
                eta_prev = eta_now
                result = (n_its, converged)
            else:
                restore(ckpt)
                n_bisect_total += 1
                pending.append(0.5 * (eta_prev + eta_now))
        return result[0], result[1], n_bisect_total

    if eta_list is None:
        eta_list = np.concatenate([np.linspace(0.02, 0.30, 8), np.linspace(0.33, 0.60, 10)])
    eta_list = np.asarray(eta_list)

    # Precompute exactly which eta(s) in eta_list are the single closest
    # match to each requested snapshot target -- NOT "within half the
    # first interval's spacing" (eta_list[1]-eta_list[0]), which breaks
    # down for non-uniformly spaced eta_list (silently matching multiple
    # neighboring points) and crashes outright for a single-point sweep
    # (eta_list[1] out of bounds) -- same fix already applied on the
    # Picard side.
    snapshot_etas = {float(eta_list[int(np.argmin(np.abs(eta_list - target)))])
                      for target in save_fields_at}

    I_avg_list, s_max_list, iters_list, fields = [], [], [], {}
    eta_prev = float(eta_list[0]) - 1e-6
    for eta in eta_list:
        n_its, converged, n_bisect = solve_at_eta(float(eta), eta_prev)
        eta_prev = float(eta)

        s_arr = project_scalar(s, "pemfc_sproj_")
        Jloc_arr = project_scalar(I_local, "pemfc_jlocproj_")
        CgO2_arr = project_scalar(CgO2_sol, "pemfc_cgo2proj_")
        I_avg = float(np.mean(Jloc_arr[membrane_dofs_V]))
        s_max_now = float(s_arr.max())
        CgO2_membrane_min = float(CgO2_arr[membrane_dofs_V].min())

        I_avg_list.append(I_avg)
        s_max_list.append(s_max_now)
        iters_list.append(n_its)
        if verbose:
            flags = []
            if CgO2_membrane_min > Y_O2_in + 1e-6:
                flags.append(" \u26a0 O2 ABOVE INLET AT MEMBRANE")
            if not converged:
                flags.append(" \u26a0 NEWTON DID NOT CONVERGE")
            if n_bisect:
                flags.append(f" [{n_bisect} continuation bisection(s)]")
            line = (f"  eta={eta:6.3f} V   I_avg={I_avg:9.3f} A/m^2 = {I_avg/1e4:6.4f} A/cm^2   "
                    f"s_max={s_max_now:.3f}   Cg_O2,membrane={CgO2_membrane_min:.4f}   "
                    f"Newton iters={n_its}{''.join(flags)}")
            print(line, flush=True)
            _progress_log(line)

        if float(eta) in snapshot_etas:
            coords = V.tabulate_dof_coordinates()[:, :2]
            p_arr = project_scalar(p_sol, "pemfc_pproj_")
            Cw_arr = project_scalar(Cw_sol, "pemfc_cwproj_")
            vx = project_scalar(u_darcy[0], "pemfc_vx_")
            vy = project_scalar(u_darcy[1], "pemfc_vy_")
            # Phase velocities (Table 1 Eq. 14): rho_l*ul = jl + lambda_l*rho*u,
            # rho_g*ug = -jl + lambda_g*rho*u.
            ul_expr = (Jl_vec + lam_l * rho_mix * u_darcy) / rho_l
            ug_expr = (-Jl_vec + lam_g * rho_mix * u_darcy) / rho_g_const
            ulx = project_scalar(ul_expr[0], "pemfc_ulx_")
            uly = project_scalar(ul_expr[1], "pemfc_uly_")
            ugx = project_scalar(ug_expr[0], "pemfc_ugx_")
            ugy = project_scalar(ug_expr[1], "pemfc_ugy_")
            y_line, xi_line = extract_interface(coords, s_arr, threshold=0.02)
            membrane = extract_line(coords, {"s": s_arr, "I_local": Jloc_arr, "CgO2": CgO2_arr})
            fields[f"eta{eta:.3f}"] = dict(
                coords=coords.copy(), p=p_arr, Cw=Cw_arr, CgO2=CgO2_arr, s=s_arr,
                vx=vx, vy=vy, ulx=ulx, uly=uly, ugx=ugx, ugy=ugy, eta=float(eta),
                interface_y=y_line, interface_x=xi_line,
                membrane_y=membrane["y"], membrane_s=membrane["s"],
                membrane_I=membrane["I_local"], membrane_CgO2=membrane["CgO2"])

    return dict(eta=eta_list, I_avg=np.array(I_avg_list), s_max=np.array(s_max_list),
                iters=np.array(iters_list), fields=fields, params=P)


def run_case_arclength(overrides=None, eta_start=0.30, ds=0.01, ds_min=1e-4, ds_max=0.03,
                        n_steps=60, max_newton_it=30, newton_tol=1e-6, verbose=True,
                        resume=False, checkpoint_path="/content/pemfc_m2_arclength_checkpoint.npz",
                        refresh_tangent=False, refresh_step=1e-4, snapshot_dir=None,
                        svd_diagnostic_every=None, svd_out_path=None, init_w_array=None):
    """Trace the (eta, w) solution branch using pseudo-arclength continuation
    (Keller's bordering algorithm) instead of run_case()'s eta-parametrized
    Newton continuation.

    WHY: direct Jacobian SVD diagnostics (run_case(..., jacobian_diagnostics=
    True) through the eta~0.30-0.36 range, base case) showed the *scaled*
    (Jacobi-equilibrated, removing the artificial ill-conditioning from
    mixing ~1e5 Pa pressure with O(1) mass fractions -- see P_scale above)
    smallest singular value collapsing from ~1.6e-5 at eta=0.355 to ~2.9e-10
    at eta=0.360 -- a genuine fold (turning point), not a scaling artifact
    or merely a hard-but-regular nonlinearity. At a fold, eta is *not* a
    valid local coordinate for the solution branch (dI/deta -> infinity, or
    the branch turns back on itself), so run_case()'s approach -- solve
    F(w, eta_target)=0 for w at a FIXED eta_target -- cannot converge near
    one no matter how good the warm start or how finely eta is bisected
    (confirmed empirically: bisecting all the way to eta=0.3563 still gave
    a near-singular Jacobian). Pseudo-arclength continuation instead treats
    eta as an UNKNOWN alongside w, adding an arclength constraint in its
    place -- allowing the path (w(s), eta(s)) to be traced smoothly through
    a fold, with eta itself moving non-monotonically (or even backwards)
    across it if that's what the branch does.

    ALGORITHM (Keller's bordering, avoids needing an augmented (n+1)x(n+1)
    matrix -- reuses the same n x n Jacobian factorization twice per Newton
    iteration instead):
      Predictor:  w_pred = w0 + ds*dw_tan,  eta_pred = eta0 + ds*deta_tan
      Corrector (Newton on the bordered system, iterating on (w_pred, eta_pred)):
        Solve J v1 = -F(w_pred, eta_pred)
        Solve J v2 = -dF/deta(w_pred, eta_pred)
        N = dw_tan.(w_pred-w0) + deta_tan*(eta_pred-eta0) - ds   [arclength residual]
        delta_eta = (-N - dw_tan.v1) / (deta_tan + dw_tan.v2)
        delta_w   = v1 + delta_eta*v2
        w_pred += delta_w; eta_pred += delta_eta; repeat until small
      Tangent update (at the new converged point, reusing J, dF/deta there):
        Solve J z = -dF/deta
        deta_new = 1/(deta_tan_old + dw_tan_old.z);  dw_new = deta_new*z
        normalize (dw_new, deta_new) to unit length; flip sign if it
        reverses direction relative to the previous tangent.

    STATUS: this is a substantially different, substantially less battle-
    tested code path than run_case() -- it duplicates run_case()'s physics
    setup verbatim (deliberately kept separate rather than refactored to
    share code, so there is zero risk of this destabilizing the validated
    run_case() implementation) but replaces the entire solve strategy with
    hand-rolled PETSc calls that have NOT been run successfully end-to-end
    even once. Treat every part of the bordering/tangent logic below as a
    first draft that will likely need real debugging with actual execution
    feedback -- the physics/residual/BC setup is the validated part, the
    continuation algorithm around it is not.

    CHECKPOINTING: after every accepted step, the current point, tangent,
    and full history are saved to `checkpoint_path` (a .npz file) --
    meant for long unattended runs where a Colab disconnect partway
    through would otherwise lose everything. Call with resume=True (same
    `overrides` as the original call -- mesh/physical parameters must
    match for the saved state to mean anything) to continue from the
    last checkpoint instead of re-bootstrapping from eta_start; `n_steps`
    in a resumed call is the number of ADDITIONAL steps to take from
    where the checkpoint left off, not a total. If no checkpoint file
    exists yet, resume=True is silently ignored and a fresh run starts.

    refresh_tangent=True (only meaningful together with resume=True):
    discards the checkpoint's saved tangent and re-bootstraps a fresh one
    via two ordinary Newton solves right at the checkpoint's current
    point, before continuing the main loop. Useful if a run stalls for
    many steps at a fixed eta with no improving trend -- check with
    diagnose_fold_point() first: if the stuck point is NOT substantially
    (many orders of magnitude) worse-conditioned than an ordinary
    reference point, the stall is more likely from tangent drift
    (accumulated floating-point error over hundreds of prior steps) than
    a genuine fold/bifurcation, and a fresh tangent may resolve it
    without needing any change to the continuation algorithm itself.

    Refreshing recomputes the tangent DIRECTLY from the Jacobian at the
    CURRENT point (w_last, eta_last) -- solving J*z = -F_eta (the same
    linear system used for the post-step tangent update) and setting the
    new tangent from z, rather than doing new nonlinear solves at nearby
    eta values. An earlier version bootstrapped via two ordinary Newton
    solves (at eta_last and eta_last +/- a small refresh_step) and took
    their secant -- this turned out to be fundamentally unsound: an
    independent test (solving from two different initial guesses at a
    fixed eta near a stuck point) confirmed the TRUE eta-fixed solution
    there had a dramatically different s_max than the branch being
    traced, meaning s(eta) was locally near-vertical right where a
    refresh was needed -- exactly the situation ordinary (eta-fixed)
    Newton cannot be trusted in, which is the whole reason arclength
    continuation is used in the first place. The direct-Jacobian
    approach never solves a new nonlinear problem, so it cannot jump to
    a different branch; `refresh_step` is kept as a parameter for
    backward compatibility but no longer affects anything.

    Returns dict with 'eta', 'I_avg', 's_max', 'ds_used', 'converged'
    (arrays, one entry per accepted continuation step, INCLUDING any
    prior steps restored from a checkpoint) -- eta may not be monotonic
    if the branch actually turns back across a fold.
    """
    from petsc4py import PETSc

    P = dict(DEFAULT_PARAMS)
    try:
        open(PROGRESS_LOG, "w").close()
    except OSError:
        pass
    _progress_log(f"run_case_arclength() started (overrides={overrides}, eta_start={eta_start}, ds={ds})")
    if snapshot_dir:
        os.makedirs(snapshot_dir, exist_ok=True)
        _progress_log(f"  snapshot_dir={snapshot_dir} (saving full fields every accepted step)")
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]
    newton_rtol, newton_atol, newton_max_it = P["newton_rtol"], P["newton_atol"], P["newton_max_it"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])

    ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)


    # Automatic mesh-resolution scaling for the arclength step sizes: the
    # plain Euclidean norm on the raw dof vector (used throughout this
    # function for ||delta_w||) grows roughly as sqrt(n_dofs) under mesh
    # refinement for a FIXED underlying physical perturbation -- a properly
    # area-weighted L2 norm would be refinement-invariant, but this simpler
    # vector norm isn't. Confirmed empirically: doubling mesh density made
    # eta advance ~sqrt(2)x slower per unit ds at a fixed ds_max. Rescaling
    # ds/ds_max/ds_min/refresh_step by sqrt(n_dofs / n_dofs_reference)
    # keeps the arclength step corresponding to roughly the same physical
    # step size regardless of mesh resolution; this is a no-op (factor
    # exactly 1.0) at the default nx/ny.
    n_dofs_reference = 3 * (DEFAULT_PARAMS["nx"] + 1) * (DEFAULT_PARAMS["ny"] + 1)
    n_dofs_actual = len(w.x.array)
    mesh_scale = float(np.sqrt(n_dofs_actual / n_dofs_reference))
    if abs(mesh_scale - 1.0) > 1e-6:
        _progress_log(f"  mesh resolution differs from reference ({n_dofs_actual} vs "
                      f"{n_dofs_reference} dofs) -- auto-scaling ds/ds_max/ds_min/refresh_step "
                      f"by {mesh_scale:.4f}x")
    ds = ds * mesh_scale
    ds_max = ds_max * mesh_scale
    ds_min = ds_min * mesh_scale
    refresh_step = refresh_step * mesh_scale
    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2

    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]
    bc_dofs_all = np.concatenate([dofs_p_in, dofs_p_out, dofs_cw_in, dofs_cgo2_in])
    membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0

    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix

    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)

    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)

    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)

    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    C_O2_mix = rho_g_const * CgO2_sol / rho_mix  # Fig. 7c: oxygen mixture (rho*C^O2 = rho_g*Cg^O2)
    Weff_w = lam_l + lam_g * CgH2O_local

    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix

    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    # OUTLET STABILIZATION (max_value(...,0.0) clip on the boundary normal
    # flux): the outlet is a natural/Neumann advective-outflow condition,
    # which implicitly assumes genuine outflow there. diagnose_fold_point()
    # + check_outlet_flow() traced the second (non-fold, ~40x-conditioned)
    # stuck point directly to this -- W_O2's normal (y) component CHANGES
    # SIGN inside the outlet segment as flooding intensifies (confirmed at
    # both the original and a refined mesh), meaning part of the "outlet"
    # locally has reversed (inflow) flow. The unclipped weak form has no
    # prescribed composition for that reversed inflow (nothing quantifies
    # what's flowing back IN), which makes its own Jacobian contribution
    # locally indefinite right where the sign flips -- a well known failure
    # mode for natural outflow BCs under flow reversal. Clipping the
    # boundary-integral velocity to its outflow-only part (>=0) removes
    # that sign-indefinite contribution; the interior advection term
    # (-inner(F_conv_w, grad(v_cw))*dx / the CgO2 analogue) is untouched.
    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
               - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
    F_total = F_pres + F_water + F_oxy

    if verbose:
        print("  [building forms + JIT-compiling F, J, dF/deta -- can take a while]", flush=True)
    _progress_log("building forms for arclength continuation")

    fc_opts = {"quadrature_degree": 4}
    F_form = fem.form(F_total, form_compiler_options=fc_opts)
    J_ufl = ufl.derivative(F_total, w)
    J_form = fem.form(J_ufl, form_compiler_options=fc_opts)
    # Note: ufl.derivative(F_total, eta_const, ...) is NOT supported by UFL
    # for a dolfinx Constant ("Invalid coefficient type" -- confirmed by
    # direct testing, not just a missing-direction-argument issue). dF/deta
    # is instead computed via finite differencing below (assemble_Feta_vec),
    # since eta enters F_total only through eta_const in the smooth Tafel
    # exponential -- a well-behaved quantity to finite-difference.

    snes_petsc_options = {
        "snes_type": "newtonls", "snes_rtol": newton_rtol, "snes_atol": newton_atol,
        "snes_max_it": newton_max_it, "snes_error_if_not_converged": False,
        "ksp_type": "preonly", "pc_type": "lu",
    }
    bootstrap_problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                          petsc_options_prefix="pemfc_arclen_boot_",
                                          form_compiler_options=fc_opts)
    # Separate, much more conservative SNES config used ONLY for the
    # refresh_tangent bootstrap (small snes_max_it): that solve starts
    # from an already-good warm start and only needs to move by a tiny
    # eta perturbation, so it should converge in just a few iterations.
    # Confirmed happening in practice: with the SAME generous iteration
    # budget as the from-scratch initial bootstrap, a refresh solve can
    # wander away from the warm start's branch entirely and converge to a
    # different, nearby solution instead (s_max collapsing from 0.306 to
    # 0.012 over a eta step of only 1.5e-4) -- capping iterations here
    # keeps the solve local to the warm start.
    refresh_snes_petsc_options = dict(snes_petsc_options)
    refresh_snes_petsc_options["snes_max_it"] = 8
    refresh_bootstrap_problem = NonlinearProblem(
        F_total, w, bcs=bcs, petsc_options=refresh_snes_petsc_options,
        petsc_options_prefix="pemfc_arclen_refresh_boot_", form_compiler_options=fc_opts)
    if verbose:
        print("  [JIT compilation done]", flush=True)
    _progress_log("JIT compilation done")

    def assemble_F_vec(x_arr, eta_val):
        w.x.array[:] = x_arr
        w.x.scatter_forward()
        eta_const.value = eta_val
        b = fem.petsc.assemble_vector(F_form)
        fem.petsc.apply_lifting(b, [J_form], bcs=[bcs], x0=[w.x.petsc_vec], alpha=-1.0)
        b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
        fem.petsc.set_bc(b, bcs, w.x.petsc_vec, -1.0)
        return b

    def assemble_J_mat():
        A = fem.petsc.assemble_matrix(J_form, bcs=bcs)
        A.assemble()
        return A

    def assemble_Feta_vec(x_arr, eta_val, fd_eps=1e-6):
        """Central-difference approximation of dF/deta at (x_arr, eta_val).
        eta enters F_total only through eta_const in the smooth Tafel
        exponential, so this is a well-behaved quantity to finite-
        difference (no UFL symbolic differentiation w.r.t. a Constant
        needed -- confirmed unsupported by this UFL version)."""
        b_plus = assemble_F_vec(x_arr, eta_val + fd_eps)
        b_minus = assemble_F_vec(x_arr, eta_val - fd_eps)
        b_plus.axpy(-1.0, b_minus)          # b_plus <- b_plus - b_minus
        b_plus.scale(1.0 / (2.0 * fd_eps))  # central difference
        b_minus.destroy()
        arr = b_plus.array_w
        arr[bc_dofs_all] = 0.0  # dF/deta is trivially 0 at Dirichlet dofs (BC values don't depend on eta)
        return b_plus

    def lu_solve(A_mat, rhs_vec):
        ksp = PETSc.KSP().create(comm)
        ksp.setOperators(A_mat)
        ksp.setType("preonly")
        ksp.getPC().setType("lu")
        x_vec = A_mat.createVecRight()
        ksp.solve(rhs_vec, x_vec)
        ksp.destroy()
        return x_vec

    def I_avg_at(x_arr, eta_val):
        w.x.array[:] = x_arr
        w.x.scatter_forward()
        eta_const.value = eta_val
        Jloc_arr = _project_scalar_standalone(I_local, V, dx)
        return float(np.mean(Jloc_arr[membrane_dofs_V]))

    def s_max_at(x_arr):
        w.x.array[:] = x_arr
        w.x.scatter_forward()
        s_arr = _project_scalar_standalone(s, V, dx)
        return float(s_arr.max())

    def svd_diag_at(x_arr, eta_val):
        """Lightweight Jacobian SVD diagnostic at (x_arr, eta_val): just
        the row-norm-equilibrated smallest/largest singular values and
        condition number -- no reference-point comparison, no null
        vectors (see diagnose_fold_point() for the full version). Cheap
        enough to call periodically (every svd_diagnostic_every accepted
        steps) to build a Fig. 3-style dataset covering the WHOLE traced
        branch, not just a one-off pre-fold sweep."""
        import scipy.sparse as sp
        w.x.array[:] = x_arr
        w.x.scatter_forward()
        eta_const.value = eta_val
        J_mat_local = assemble_J_mat()
        n = J_mat_local.getSize()[0]
        indptr, indices, data = J_mat_local.getValuesCSR()
        J_dense = sp.csr_matrix((data, indices, indptr), shape=(n, n)).toarray()
        row_norms = np.sqrt(np.sum(J_dense**2, axis=1))
        scale = 1.0 / np.sqrt(np.maximum(row_norms, 1e-30))
        J_scaled = (J_dense * scale[None, :]) * scale[:, None]
        svals = np.linalg.svd(J_scaled, compute_uv=False)
        sv_min, sv_max = float(svals.min()), float(svals.max())
        J_mat_local.destroy()
        return sv_min, sv_max, sv_max / max(sv_min, 1e-300)

    _snapshot_coords = V.tabulate_dof_coordinates()[:, :2] if snapshot_dir else None

    # Periodic SVD diagnostic tracking (Fig. 3-style dataset covering the
    # whole branch, not just a one-off pre-fold sweep). Independent of
    # the main checkpoint file so it can be inspected/plotted without
    # touching continuation state; persists across resumed sessions by
    # loading whatever's already at svd_out_path.
    svd_eta_hist, svd_smallest_hist, svd_largest_hist, svd_cond_hist = [], [], [], []
    if svd_out_path and os.path.exists(svd_out_path):
        try:
            _svd_prior = np.load(svd_out_path)
            svd_eta_hist = list(_svd_prior["eta"])
            svd_smallest_hist = list(_svd_prior["sv_smallest"])
            svd_largest_hist = list(_svd_prior["sv_largest"])
            svd_cond_hist = list(_svd_prior["cond"])
            _progress_log(f"  loaded {len(svd_eta_hist)} prior SVD diagnostic point(s) "
                          f"from {svd_out_path}")
        except Exception as e:
            _progress_log(f"  could not load prior SVD diagnostics ({e}), starting fresh")

    def maybe_record_svd(x_arr, eta_val, accepted_step_count):
        if not svd_diagnostic_every or accepted_step_count % svd_diagnostic_every != 0:
            return
        sv_min, sv_max, cond = svd_diag_at(x_arr, eta_val)
        svd_eta_hist.append(eta_val)
        svd_smallest_hist.append(sv_min)
        svd_largest_hist.append(sv_max)
        svd_cond_hist.append(cond)
        _progress_log(f"  svd_diag: eta={eta_val:.5f} sv_min={sv_min:.4e} cond={cond:.4e}")
        if svd_out_path:
            try:
                np.savez(svd_out_path, eta=np.array(svd_eta_hist),
                         sv_smallest=np.array(svd_smallest_hist),
                         sv_largest=np.array(svd_largest_hist),
                         cond=np.array(svd_cond_hist))
            except Exception as e:
                _progress_log(f"  svd_diag save failed (continuing anyway): {e}")

    def save_snapshot_now(x_arr, eta_val, step_label, deta_sign=1.0):
        """Project and save the full Fig. 7 field set (s, p, Cw, CgO2,
        CgH2O water vapor, C_O2_mix oxygen mixture) for THIS accepted
        step, if snapshot_dir was provided. Assumes w is already set to
        (x_arr, eta_val) by the caller (I_avg_at/s_max_at just before
        this is always called that way in practice) -- re-sets it
        explicitly anyway to not depend on call order."""
        if snapshot_dir is None:
            return
        try:
            w.x.array[:] = x_arr
            w.x.scatter_forward()
            eta_const.value = eta_val
            s_arr = _project_scalar_standalone(s, V, dx)
            p_arr = _project_scalar_standalone(p_sol, V, dx)
            Cw_arr = _project_scalar_standalone(Cw_sol, V, dx)
            CgO2_arr = _project_scalar_standalone(CgO2_sol, V, dx)
            CgH2O_arr = _project_scalar_standalone(CgH2O_local, V, dx)
            CO2mix_arr = _project_scalar_standalone(C_O2_mix, V, dx)
            branch_state = _classify_branch_state(eta_val, float(s_arr.max()), deta_sign)
            fname = os.path.join(snapshot_dir, f"step_{step_label}_eta_{eta_val:.5f}.npz")
            np.savez(fname, coords=_snapshot_coords, eta=eta_val, s=s_arr, p=p_arr,
                     Cw=Cw_arr, CgO2=CgO2_arr, CgH2O=CgH2O_arr, C_O2_mix=CO2mix_arr,
                     branch_state=branch_state)
        except Exception as e:
            _progress_log(f"  snapshot save FAILED for step {step_label} (continuing anyway): {e}")

    def save_checkpoint(w_last, eta_last, dw_tan, deta_tan, eta_hist, I_hist,
                         smax_hist, ds_hist, cur_ds):
        if not (np.all(np.isfinite(w_last)) and np.isfinite(eta_last)
                and np.all(np.isfinite(dw_tan)) and np.isfinite(deta_tan)):
            _progress_log("  REFUSING to save checkpoint: state contains NaN/Inf -- "
                          "keeping the last good checkpoint on disk instead")
            return
        branch_state = _classify_branch_state(eta_last, float(smax_hist[-1]) if smax_hist else 0.0,
                                               deta_tan)
        try:
            np.savez(checkpoint_path, w_last=w_last, eta_last=eta_last, dw_tan=dw_tan,
                     deta_tan=deta_tan, eta_hist=np.array(eta_hist), I_hist=np.array(I_hist),
                     smax_hist=np.array(smax_hist), ds_hist=np.array(ds_hist), cur_ds=cur_ds,
                     branch_state=branch_state)
        except OSError as e:
            _progress_log(f"  checkpoint save failed (continuing anyway): {e}")

    resumed = False
    if resume and os.path.exists(checkpoint_path):
        try:
            ckpt = np.load(checkpoint_path)
            w_last = ckpt["w_last"]
            eta_last = float(ckpt["eta_last"])
            dw_tan = ckpt["dw_tan"]
            deta_tan = float(ckpt["deta_tan"])
            eta_hist = list(ckpt["eta_hist"])
            I_hist = list(ckpt["I_hist"])
            smax_hist = list(ckpt["smax_hist"])
            ds_hist = list(ckpt["ds_hist"])
            cur_ds = float(ckpt["cur_ds"])
            conv_hist = [True] * len(eta_hist)
            resumed = True
            if "branch_state" in ckpt.files:
                branch_state = str(ckpt["branch_state"])
            else:
                branch_state = _classify_branch_state(eta_last, smax_hist[-1] if smax_hist else 0.0,
                                                       deta_tan)
            _progress_log(f"RESUMED from checkpoint '{checkpoint_path}': "
                          f"{len(eta_hist)} prior step(s), last eta={eta_last:.5f}, "
                          f"cur_ds={cur_ds:.5f}, branch_state=[{branch_state}]")
            if verbose:
                print(f"  [resumed from checkpoint: {len(eta_hist)} prior steps, "
                      f"last eta={eta_last:.5f}, branch_state=[{branch_state}]]", flush=True)

            if refresh_tangent:
                # Direction from recent history (same logic as before):
                # this branch can be heading in EITHER eta direction (e.g.
                # past a fold, tracing the back side where eta decreases).
                if len(eta_hist) >= 2 and eta_hist[-1] < eta_hist[-2]:
                    direction = -1.0
                else:
                    direction = 1.0

                # Recompute the tangent DIRECTLY from the Jacobian at the
                # CURRENT point (w_last, eta_last) -- no new nonlinear
                # solve at a perturbed eta. This replaces an earlier
                # design (two ordinary Newton solves at eta_last and
                # eta_last +/- a small step, then a secant) that turned
                # out to be fundamentally unsound: an independent test
                # (two different initial guesses solved at the same fixed
                # eta near a stuck point) confirmed the "true" eta-fixed
                # solution there has s_max~0.01, NOT the s_max~0.31 the
                # branch was actually tracing -- i.e. exactly the kind of
                # near-vertical s(eta) dependence that motivated using
                # arclength continuation in the first place. Fixing eta
                # and running ordinary Newton is the wrong tool in
                # precisely the regions where a tangent refresh is
                # needed, since those are the delicate regions to begin
                # with. A pure linearization at the current point cannot
                # jump to a different branch (it never solves a new
                # nonlinear problem), and is also cheaper (no extra
                # nonlinear solves at all).
                _progress_log(f"  refresh_tangent=True: recomputing tangent directly via "
                              f"Jacobian at eta={eta_last:.5f} (no new nonlinear solve), "
                              f"direction={direction:+.0f} from recent history")
                if verbose:
                    print(f"  [refreshing tangent directly via Jacobian at eta={eta_last:.5f}, "
                          f"direction={direction:+.0f}]", flush=True)

                F_vec = assemble_F_vec(w_last, eta_last)
                J_mat = assemble_J_mat()
                Feta_vec = assemble_Feta_vec(w_last, eta_last)
                Feta_vec.scale(-1.0)
                z = lu_solve(J_mat, Feta_vec)

                if not np.all(np.isfinite(z.array)):
                    J_mat.destroy(); z.destroy(); F_vec.destroy(); Feta_vec.destroy()
                    raise RuntimeError(
                        f"refresh_tangent: NaN/Inf in direct Jacobian tangent computation "
                        f"at eta={eta_last:.5f}. Checkpoint left UNCHANGED.")

                dw_tan_new = direction * z.array
                deta_tan_new = float(direction)
                norm_t = float(np.sqrt(np.sum(dw_tan_new**2) + deta_tan_new**2))
                if not (np.isfinite(norm_t) and norm_t > 1e-300):
                    J_mat.destroy(); z.destroy(); F_vec.destroy(); Feta_vec.destroy()
                    raise RuntimeError(
                        f"refresh_tangent: degenerate tangent norm ({norm_t}) at "
                        f"eta={eta_last:.5f}. Checkpoint left UNCHANGED.")
                dw_tan = dw_tan_new / norm_t
                deta_tan = deta_tan_new / norm_t
                J_mat.destroy(); z.destroy(); F_vec.destroy(); Feta_vec.destroy()

                # The point itself (w_last, eta_last) is UNCHANGED -- only
                # the tangent was recomputed -- so there's no new history
                # entry or snapshot to add here, just a checkpoint update
                # with the fresh tangent.
                cur_ds = min(cur_ds, ds)
                _progress_log(f"  refresh done: tangent recomputed at eta={eta_last:.5f} "
                              f"(point unchanged)")
                save_checkpoint(w_last, eta_last, dw_tan, deta_tan, eta_hist, I_hist,
                                smax_hist, ds_hist, cur_ds)
        except Exception as e:
            _progress_log(f"  checkpoint load failed ({e}), starting fresh instead")
            resumed = False

    if not resumed:
        # --- bootstrap: two ordinary (SNES) solves to get a starting secant tangent ---
        eta_const.value = eta_start
        if init_w_array is not None:
            # Warm-start from an existing solution (e.g. the main run's
            # current state, reused for a separate segment targeting the
            # same or a nearby eta) instead of a naive/inlet-like guess --
            # skips the warmup ramp entirely below, since we're already
            # starting from a state that's presumably close to converged
            # somewhere in the easy, near-single-phase regime.
            w.x.array[:] = init_w_array
            w.x.scatter_forward()
            _progress_log(f"bootstrap: warm-starting from a supplied init_w_array "
                          f"at eta={eta_start} (skipping the naive-guess warmup ramp)")
        else:
            w.sub(0).interpolate(lambda x: np.full(x.shape[1], (p_in - p_out) / P_scale))
            w.sub(1).interpolate(lambda x: np.full(x.shape[1], Y_H2O_in))
            w.sub(2).interpolate(lambda x: np.full(x.shape[1], Y_O2_in))
            w.x.scatter_forward()

        # Warm-up ramp: solving directly at eta_start (default 0.30) from
        # this naive/inlet-like guess is NOT reliable on its own (confirmed:
        # SNES fails to converge, leaving w in a divergent state with Cw
        # far outside its physical [0,1] range) -- the same issue already
        # found and fixed in fold_svd_sweep(). Ramp up gradually from a
        # much smaller eta first, exactly like that fix, before the actual
        # two-point secant bootstrap below. Skipped entirely if warm-
        # starting from an existing solution (init_w_array) instead.
        if init_w_array is None and eta_start > 0.05:
            warmup = list(np.linspace(0.02, eta_start, 8))[:-1]
            _progress_log(f"bootstrap: warming up from eta=0.02 to eta={eta_start} "
                          f"before the main bootstrap...")
            for eta_now in warmup:
                eta_const.value = eta_now
                bootstrap_problem.solve()
                if bootstrap_problem.solver.getConvergedReason() <= 0:
                    raise RuntimeError(
                        f"run_case_arclength: bootstrap warm-up solve failed to converge "
                        f"at eta={eta_now:.4f} -- cannot safely bootstrap the main "
                        f"continuation from a divergent state.")

        _progress_log(f"bootstrap solve 1 at eta={eta_start}")
        bootstrap_problem.solve()
        if bootstrap_problem.solver.getConvergedReason() <= 0:
            raise RuntimeError(
                f"run_case_arclength: bootstrap solve 1 failed to converge at "
                f"eta={eta_start} even after warm-up. Checkpoint left UNCHANGED.")
        w0_arr = w.x.array.copy()
        eta0 = eta_start

        _progress_log(f"bootstrap solve 2 at eta={eta_start + ds}")
        eta_const.value = eta_start + ds
        bootstrap_problem.solve()
        if bootstrap_problem.solver.getConvergedReason() <= 0:
            raise RuntimeError(
                f"run_case_arclength: bootstrap solve 2 failed to converge at "
                f"eta={eta_start + ds}. Checkpoint left UNCHANGED.")
        w1_arr = w.x.array.copy()
        eta1 = eta_start + ds

        dw_tan = w1_arr - w0_arr
        deta_tan = eta1 - eta0
        secant_len = float(np.sqrt(np.sum(dw_tan**2) + deta_tan**2))
        dw_tan = dw_tan / secant_len
        deta_tan = deta_tan / secant_len
        _progress_log(f"bootstrap done: eta0={eta0}, eta1={eta1}, |secant|={secant_len:.4e}")

        eta_hist = [eta0, eta1]
        I_hist = [I_avg_at(w0_arr, eta0), I_avg_at(w1_arr, eta1)]
        smax_hist = [s_max_at(w0_arr), s_max_at(w1_arr)]
        ds_hist = [0.0, ds]
        conv_hist = [True, True]
        cur_ds = ds
        save_snapshot_now(w0_arr, eta0, "boot0", deta_sign=1.0)
        save_snapshot_now(w1_arr, eta1, "boot1", deta_sign=1.0)
        maybe_record_svd(w0_arr, eta0, 1)
        maybe_record_svd(w1_arr, eta1, 2)

        w_last, eta_last = w1_arr, eta1
        save_checkpoint(w_last, eta_last, dw_tan, deta_tan, eta_hist, I_hist,
                         smax_hist, ds_hist, cur_ds)

    for step in range(n_steps):
        w_pred = w_last + cur_ds * dw_tan
        eta_pred = eta_last + cur_ds * deta_tan
        _progress_log(f"step {step}: predictor eta={eta_pred:.5f}  ds={cur_ds:.5f}")

        x, eta_try = w_pred.copy(), eta_pred
        converged = False
        for it in range(max_newton_it):
            F_vec = assemble_F_vec(x, eta_try)
            J_mat = assemble_J_mat()
            Feta_vec = assemble_Feta_vec(x, eta_try)

            F_vec.scale(-1.0)
            v1 = lu_solve(J_mat, F_vec)
            Feta_vec.scale(-1.0)
            v2 = lu_solve(J_mat, Feta_vec)

            if not (np.all(np.isfinite(v1.array)) and np.all(np.isfinite(v2.array))):
                _progress_log(f"  step {step} it {it}: NaN/Inf in linear solve "
                              f"(Jacobian likely singular here too) -- treating as failed corrector")
                J_mat.destroy(); v1.destroy(); v2.destroy(); F_vec.destroy(); Feta_vec.destroy()
                break

            N_val = float(np.dot(dw_tan, x - w_last) + deta_tan * (eta_try - eta_last) - cur_ds)
            dw_dot_v1 = float(np.dot(dw_tan, v1.array))
            dw_dot_v2 = float(np.dot(dw_tan, v2.array))
            denom = deta_tan + dw_dot_v2
            if abs(denom) < 1e-300:
                _progress_log(f"  step {step} it {it}: bordering denom ~0, aborting corrector")
                J_mat.destroy(); v1.destroy(); v2.destroy(); F_vec.destroy(); Feta_vec.destroy()
                break
            delta_eta = (-N_val - dw_dot_v1) / denom
            delta_x = v1.array + delta_eta * v2.array

            if not (np.all(np.isfinite(delta_x)) and np.isfinite(delta_eta)):
                _progress_log(f"  step {step} it {it}: NaN/Inf in computed update -- "
                              f"treating as failed corrector")
                J_mat.destroy(); v1.destroy(); v2.destroy(); F_vec.destroy(); Feta_vec.destroy()
                break

            x = x + delta_x
            eta_try = eta_try + delta_eta
            res_norm = float(np.sqrt(np.sum(delta_x**2) + delta_eta**2))
            J_mat.destroy(); v1.destroy(); v2.destroy(); F_vec.destroy(); Feta_vec.destroy()
            if res_norm < newton_tol:
                converged = True
                _progress_log(f"  step {step}: corrector converged in {it+1} it, "
                              f"eta={eta_try:.5f}")
                break
        if not converged:
            cur_ds *= 0.5
            _progress_log(f"  step {step}: corrector FAILED, halving ds to {cur_ds:.5f}")
            if cur_ds < ds_min:
                _progress_log("  ds below ds_min, stopping continuation")
                break
            continue

        I_now = I_avg_at(x, eta_try)
        smax_now = s_max_at(x)

        # Branch-jump sanity check (same threshold as the refresh
        # bootstrap's own check): a converged Newton step can still land
        # on a genuinely different, disconnected solution if multiple
        # solutions coexist near this eta -- residual convergence alone
        # doesn't guarantee it's still on the branch being traced.
        # Confirmed to happen in practice: s_max jumped from 0.306 to
        # 0.011 over a single accepted step with eta moving by only
        # ~0.005, right after an outlet-BC stabilization fix widened
        # Newton's basin of attraction enough to make a nearby, simpler
        # (near-single-phase) solution reachable in one step.
        smax_prev = smax_hist[-1] if smax_hist else smax_now
        threshold = 0.05 + 0.1 * smax_prev
        _progress_log(f"  step {step}: branch-jump check: smax_prev={smax_prev:.4f} "
                      f"smax_now={smax_now:.4f} diff={abs(smax_now-smax_prev):.4f} "
                      f"threshold={threshold:.4f} len(smax_hist)={len(smax_hist)}")
        if abs(smax_now - smax_prev) > threshold:
            _progress_log(f"  step {step}: REJECTED as likely branch jump "
                          f"(s_max {smax_prev:.3f} -> {smax_now:.3f} over a single step, "
                          f"eta {eta_last:.5f} -> {eta_try:.5f}) -- halving ds and retrying")
            cur_ds *= 0.5
            if cur_ds < ds_min:
                _progress_log("  ds below ds_min, stopping continuation")
                break
            continue

        eta_hist.append(eta_try); I_hist.append(I_now); smax_hist.append(smax_now)
        ds_hist.append(cur_ds); conv_hist.append(True)
        save_snapshot_now(x, eta_try, f"{len(eta_hist):05d}", deta_sign=deta_tan)
        maybe_record_svd(x, eta_try, len(eta_hist))
        branch_state_now = _classify_branch_state(eta_try, smax_now, deta_tan)
        step_summary = (f"  [arclength] step={step}  eta={eta_try:.4f}  I={I_now/1e4:.4f} A/cm^2  "
                         f"s_max={smax_now:.3f}  ds={cur_ds:.4f}  branch_state=[{branch_state_now}]")
        _progress_log(step_summary)
        if verbose:
            print(step_summary, flush=True)

        # tangent update at the new point
        F_vec = assemble_F_vec(x, eta_try)  # also sets w to (x, eta_try) as a side effect
        J_mat = assemble_J_mat()
        Feta_vec = assemble_Feta_vec(x, eta_try)
        Feta_vec.scale(-1.0)
        z = lu_solve(J_mat, Feta_vec)
        if not np.all(np.isfinite(z.array)):
            _progress_log(f"  step {step}: NaN/Inf in tangent-update solve -- "
                          f"keeping previous tangent unchanged")
        else:
            dw_dot_z = float(np.dot(dw_tan, z.array))
            denom_t = deta_tan + dw_dot_z
            deta_new = 1.0 / denom_t if abs(denom_t) > 1e-300 else deta_tan
            dw_new = deta_new * z.array
            norm_t = float(np.sqrt(np.sum(dw_new**2) + deta_new**2))
            if np.isfinite(norm_t) and norm_t > 1e-300:
                dw_new, deta_new = dw_new / norm_t, deta_new / norm_t
                if np.all(np.isfinite(dw_new)) and np.isfinite(deta_new):
                    if dw_new @ dw_tan + deta_new * deta_tan < 0:
                        dw_new, deta_new = -dw_new, -deta_new
                    dw_tan, deta_tan = dw_new, deta_new
                else:
                    _progress_log(f"  step {step}: normalized tangent has NaN/Inf -- "
                                  f"keeping previous tangent unchanged")
            else:
                _progress_log(f"  step {step}: degenerate tangent norm ({norm_t}) -- "
                              f"keeping previous tangent unchanged")
        J_mat.destroy(); z.destroy(); F_vec.destroy(); Feta_vec.destroy()

        w_last, eta_last = x, eta_try
        cur_ds = min(cur_ds * 1.2, ds_max)
        save_checkpoint(w_last, eta_last, dw_tan, deta_tan, eta_hist, I_hist,
                         smax_hist, ds_hist, cur_ds)

    return dict(eta=np.array(eta_hist), I_avg=np.array(I_hist), s_max=np.array(smax_hist),
                ds_used=np.array(ds_hist), converged=np.array(conv_hist), params=P)


_project_scalar_call_counter = [0]


def _project_scalar_standalone(expr, V, dx):
    """Standalone L2 projection helper for run_case_arclength (which
    doesn't have run_case's nested project_scalar closure available).
    Uses a fresh, unique PETSc options prefix every call (via a simple
    counter) -- reusing the same fixed prefix across many rapid calls
    was suspected of allowing PETSc-level state to leak between calls
    (confirmed symptom: s_max_at() appeared to return a value from a
    DIFFERENT, previous state than the one just requested, e.g. right
    after a tiny eta perturbation that should have left s_max nearly
    unchanged)."""
    _project_scalar_call_counter[0] += 1
    prefix = f"pemfc_arclen_proj_{_project_scalar_call_counter[0]}_"
    u_tr, v_te = ufl.TrialFunction(V), ufl.TestFunction(V)
    a_p = u_tr * v_te * dx
    L_p = expr * v_te * dx
    prob = LinearProblem(a_p, L_p, bcs=[],
                          petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
                          petsc_options_prefix=prefix)
    return prob.solve().x.array.copy()


def diagnose_fold_point(checkpoint_path, overrides=None, reference_eta=0.30,
                         n_smallest=10, verbose=True, save_nullvec_path=None):
    """Diagnose the local bifurcation structure AT the point currently
    saved in checkpoint_path (from run_case_arclength), to distinguish a
    genuine SIMPLE FOLD (limit point) -- which pseudo-arclength
    continuation is designed to trace straight through -- from a higher-
    codimension degeneracy (a bifurcation point, where a second branch
    crosses the one being traced, or a cusp, where two folds coincide).
    Motivated by run_case_arclength stalling for 90+ steps at a fixed eta
    with no improving trend in Newton iteration counts, unlike the FIRST
    fold (near eta~0.36 for the base case), which showed a clear,
    monotonically-improving iteration-count trend and broke through in
    ~6 step-halvings. That qualitative difference is the reason to
    suspect this second stuck point is NOT a simple fold.

    METHODOLOGY (revised after an earlier version of this function gave
    an unreliable result): the Jacobian's singular value spectrum is
    computed at the stuck point AND at a fresh solve of a KNOWN-REGULAR
    reference point (`reference_eta`, default 0.30 -- comfortably far
    from any fold), using row-norm (not diagonal/Jacobi) equilibration,
    and the two are compared directly:

    1. CONDITION NUMBER RATIO (stuck / reference): a genuine singularity
       shows up as the stuck point being MANY orders of magnitude worse-
       conditioned than an ordinary point; a mild or absent gap says the
       stall is more likely algorithmic than mathematical. An earlier
       version of this function instead counted singular values below an
       absolute threshold (1e-3 x the largest) -- this was unreliable:
       FEM discretizations of elliptic-type PDEs naturally have singular
       value spectra spanning many orders of magnitude even at perfectly
       regular points (mesh-induced conditioning, unrelated to any
       bifurcation), so an uncalibrated absolute cutoff flagged most of
       the ~4500-dof spectrum as "small" regardless of what was actually
       happening. Comparing against a live reference point removes that
       problem. Row-norm scaling (rather than pure diagonal/Jacobi
       scaling) was also substituted for the same reason: diagonal
       scaling blows up if even a single dof has a near-zero DIAGONAL
       entry (not inherently meaningful -- can happen in an advection-
       dominated or weakly-coupled row), inflating that dof's entire
       row/column and producing a spuriously wide spectrum unrelated to
       genuine bifurcation structure; row-norm scaling only breaks down
       if a dof's entire row vanishes, a much stronger condition.

    2. TRANSVERSALITY: phi^T F_eta, where phi is the LEFT null vector
       (the left singular vector paired with the smallest singular
       value, mapped back through the row-norm scaling) and F_eta =
       dF/deta (via finite differencing, as in run_case_arclength). At a
       genuine simple fold this is transverse (bounded away from zero);
       collapsing toward zero indicates something more degenerate even
       if the conditioning gap from (1) alone looked fold-like.

    Returns a dict with the raw diagnostics (singular values, the
    transversality quantity) and a plain-language verdict string.
    """
    from petsc4py import PETSc
    import scipy.sparse as sp

    P = dict(DEFAULT_PARAMS)
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])

    ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)

    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2

    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]
    bc_dofs_all = np.concatenate([dofs_p_in, dofs_p_out, dofs_cw_in, dofs_cgo2_in])

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0
    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix

    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)

    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)
    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    Weff_w = lam_l + lam_g * CgH2O_local
    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
               - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
    F_total = F_pres + F_water + F_oxy

    fc_opts = {"quadrature_degree": 4}
    F_form = fem.form(F_total, form_compiler_options=fc_opts)
    J_ufl = ufl.derivative(F_total, w)
    J_form = fem.form(J_ufl, form_compiler_options=fc_opts)

    def assemble_F_vec(x_arr, eta_val):
        w.x.array[:] = x_arr
        w.x.scatter_forward()
        eta_const.value = eta_val
        b = fem.petsc.assemble_vector(F_form)
        fem.petsc.apply_lifting(b, [J_form], bcs=[bcs], x0=[w.x.petsc_vec], alpha=-1.0)
        b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
        fem.petsc.set_bc(b, bcs, w.x.petsc_vec, -1.0)
        return b

    def assemble_J_mat():
        A = fem.petsc.assemble_matrix(J_form, bcs=bcs)
        A.assemble()
        return A

    def assemble_Feta_vec(x_arr, eta_val, fd_eps=1e-6):
        b_plus = assemble_F_vec(x_arr, eta_val + fd_eps)
        b_minus = assemble_F_vec(x_arr, eta_val - fd_eps)
        b_plus.axpy(-1.0, b_minus)
        b_plus.scale(1.0 / (2.0 * fd_eps))
        b_minus.destroy()
        arr = b_plus.array_w
        arr[bc_dofs_all] = 0.0
        return b_plus

    def analyze_point(x_arr, eta_val, label):
        """Assemble J and dF/deta at (x_arr, eta_val) and return its
        row-norm-equilibrated singular value spectrum plus the
        transversality quantity. Row-norm scaling (rescale each row/col
        by 1/sqrt(its L2 norm)) is used instead of pure Jacobi/diagonal
        scaling -- diagonal scaling blows up if even a single dof has a
        near-zero DIAGONAL entry (common in an advection-dominated or
        near-decoupled row), which artificially inflates that dof's
        entire row/column and can produce a spurious wide spectrum with
        no relation to genuine bifurcation structure. Row-norm scaling
        only breaks down if a dof's entire row is near-zero, which is a
        much stronger (and more meaningful) degeneracy condition."""
        F_vec = assemble_F_vec(x_arr, eta_val)
        J_mat = assemble_J_mat()
        Feta_vec = assemble_Feta_vec(x_arr, eta_val)

        n = J_mat.getSize()[0]
        indptr, indices, data = J_mat.getValuesCSR()
        J_dense = sp.csr_matrix((data, indices, indptr), shape=(n, n)).toarray()

        row_norms = np.sqrt(np.sum(J_dense**2, axis=1))
        scale = 1.0 / np.sqrt(np.maximum(row_norms, 1e-30))
        J_scaled = (J_dense * scale[None, :]) * scale[:, None]

        U, svals, Vt = np.linalg.svd(J_scaled)
        order_idx = np.argsort(svals)
        svals_sorted = svals[order_idx]

        idx_min = order_idx[0]
        phi_scaled = U[:, idx_min]
        phi = scale * phi_scaled
        phi = phi / np.linalg.norm(phi)
        # right null vector (solution-space direction the branch is
        # degenerate along) -- for row-norm scaling, J_scaled = diag(scale)
        # @ J @ diag(scale), so the RIGHT singular vector maps back the
        # same way as the left one (both sides scaled identically here,
        # unlike a genuinely asymmetric row/column scaling)
        psi_scaled = Vt[idx_min, :]
        psi = scale * psi_scaled
        psi = psi / np.linalg.norm(psi)

        Feta_arr = Feta_vec.array.copy()
        transversality_raw = float(np.dot(phi, Feta_arr))
        transversality_normalized = transversality_raw / (
            np.linalg.norm(phi) * np.linalg.norm(Feta_arr) + 1e-300)

        J_mat.destroy(); F_vec.destroy(); Feta_vec.destroy()

        if verbose:
            print(f"[{label}] eta={eta_val:.5f}  smallest {n_smallest} scaled svals: "
                  f"{np.array2string(svals_sorted[:n_smallest], precision=3)}", flush=True)
            print(f"[{label}]   largest={svals_sorted[-1]:.4g}  "
                  f"cond={svals_sorted[-1]/max(svals_sorted[0], 1e-300):.4e}  "
                  f"transversality(normalized)={transversality_normalized:.4e}", flush=True)

        return dict(eta=eta_val, svals_scaled=svals_sorted,
                    transversality_normalized=transversality_normalized,
                    cond=svals_sorted[-1] / max(svals_sorted[0], 1e-300),
                    phi=phi, psi=psi)

    # --- stuck point, from the checkpoint ---
    ckpt = np.load(checkpoint_path)
    w_stuck = ckpt["w_last"]
    eta_stuck = float(ckpt["eta_last"])
    _progress_log(f"diagnose_fold_point: loaded checkpoint, eta={eta_stuck:.5f}")
    if verbose:
        print(f"Diagnosing stuck point at eta={eta_stuck:.5f} (from {checkpoint_path})\n", flush=True)
    stuck = analyze_point(w_stuck, eta_stuck, "STUCK")

    if save_nullvec_path:
        # Split phi (left null vector -- which equation/residual is
        # poorly constrained) and psi (right null vector -- which
        # solution-space direction the branch is degenerate along, i.e.
        # roughly "what a fold-crossing perturbation would look like")
        # into their p_hat/Cw/CgO2 components and save with mesh
        # coordinates, so their SPATIAL structure can be plotted: if the
        # near-singular direction is concentrated in a small region of
        # the domain (e.g. near a developing saturation front), that
        # points at mesh resolution rather than a genuine bifurcation;
        # if it's spread across the whole domain, that's more consistent
        # with a true higher-codimension point.
        w_phi = fem.Function(W)
        w_phi.x.array[:] = stuck["phi"]
        w_psi = fem.Function(W)
        w_psi.x.array[:] = stuck["psi"]
        phi_p = w_phi.sub(0).collapse()
        phi_cw = w_phi.sub(1).collapse()
        phi_cgo2 = w_phi.sub(2).collapse()
        psi_p = w_psi.sub(0).collapse()
        psi_cw = w_psi.sub(1).collapse()
        psi_cgo2 = w_psi.sub(2).collapse()
        coords_p = phi_p.function_space.tabulate_dof_coordinates()[:, :2]
        coords_cw = phi_cw.function_space.tabulate_dof_coordinates()[:, :2]
        coords_cgo2 = phi_cgo2.function_space.tabulate_dof_coordinates()[:, :2]
        np.savez(save_nullvec_path, eta=eta_stuck,
                 coords_p=coords_p, coords_Cw=coords_cw, coords_CgO2=coords_cgo2,
                 phi_p=phi_p.x.array.copy(), phi_Cw=phi_cw.x.array.copy(),
                 phi_CgO2=phi_cgo2.x.array.copy(),
                 psi_p=psi_p.x.array.copy(), psi_Cw=psi_cw.x.array.copy(),
                 psi_CgO2=psi_cgo2.x.array.copy())
        if verbose:
            print(f"Saved null-vector spatial data to {save_nullvec_path}", flush=True)

    # --- reference point: a fresh ordinary solve at a KNOWN-regular eta,
    # analyzed with the exact same method, so the comparison is apples-
    # to-apples rather than judged against an uncalibrated absolute
    # threshold (an earlier version of this function did that and it was
    # unreliable -- FEM discretizations naturally have singular value
    # spectra spanning many orders of magnitude even at perfectly regular
    # points, so an absolute cutoff flags most of the spectrum as "small"
    # regardless of whether anything unusual is actually happening) ---
    if verbose:
        print(f"\nSolving reference point at eta={reference_eta} for comparison...", flush=True)
    snes_petsc_options = {
        "snes_type": "newtonls", "snes_rtol": P["newton_rtol"], "snes_atol": P["newton_atol"],
        "snes_max_it": P["newton_max_it"], "snes_error_if_not_converged": False,
        "ksp_type": "preonly", "pc_type": "lu",
    }
    ref_problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                    petsc_options_prefix="pemfc_diag_ref_",
                                    form_compiler_options=fc_opts)
    w.sub(0).interpolate(lambda x: np.full(x.shape[1], (p_in - p_out) / P_scale))
    w.sub(1).interpolate(lambda x: np.full(x.shape[1], Y_H2O_in))
    w.sub(2).interpolate(lambda x: np.full(x.shape[1], Y_O2_in))
    w.x.scatter_forward()

    # Warm-up ramp: solving directly at reference_eta (default 0.30) from
    # this naive/inlet-like guess is NOT reliable on its own (same issue
    # found and fixed in fold_svd_sweep() / run_case_arclength()'s
    # bootstrap -- SNES can fail to converge, leaving w in a divergent
    # state with Cw far outside its physical range, which would silently
    # corrupt every downstream condition-ratio comparison in this
    # function without this check).
    if reference_eta > 0.05:
        for eta_now in np.linspace(0.02, reference_eta, 8)[:-1]:
            eta_const.value = eta_now
            ref_problem.solve()
            if ref_problem.solver.getConvergedReason() <= 0:
                raise RuntimeError(
                    f"diagnose_fold_point: reference warm-up solve failed to converge "
                    f"at eta={eta_now:.4f}.")
    eta_const.value = reference_eta
    ref_problem.solve()
    if ref_problem.solver.getConvergedReason() <= 0:
        raise RuntimeError(
            f"diagnose_fold_point: reference point solve failed to converge at "
            f"eta={reference_eta} even after warm-up -- cannot produce a trustworthy "
            f"comparison. Try a different reference_eta.")
    w_ref = w.x.array.copy()
    reference = analyze_point(w_ref, reference_eta, "REFERENCE")

    smallest_ratio = reference["svals_scaled"][0] / max(stuck["svals_scaled"][0], 1e-300)
    cond_ratio = stuck["cond"] / max(reference["cond"], 1e-300)

    verdict_lines = [
        "",
        f"Stuck point (eta={stuck['eta']:.5f}): smallest scaled sval = "
        f"{stuck['svals_scaled'][0]:.4e}, cond = {stuck['cond']:.4e}",
        f"Reference point (eta={reference['eta']:.5f}): smallest scaled sval = "
        f"{reference['svals_scaled'][0]:.4e}, cond = {reference['cond']:.4e}",
        f"Smallest-singular-value ratio (reference/stuck): {smallest_ratio:.4e}",
        f"Condition-number ratio (stuck/reference): {cond_ratio:.4e}",
        f"Stuck-point transversality (normalized): {stuck['transversality_normalized']:.4e}",
    ]

    if cond_ratio < 20:
        verdict = ("NOT clearly singular relative to a normal point -- the stuck point's "
                   "conditioning is within roughly the same order of magnitude as an "
                   "ordinary, easily-converged reference point. This does NOT look like a "
                   "genuine fold, cusp, or bifurcation; the stall is more likely an "
                   "algorithmic/step-control issue (tangent update, tolerance, or step size) "
                   "rather than a true mathematical singularity.")
    elif abs(stuck["transversality_normalized"]) < 1e-3:
        verdict = ("LIKELY HIGHER-CODIMENSION POINT: substantially worse-conditioned than the "
                   "reference AND the transversality condition is collapsing toward zero -- "
                   "consistent with a cusp or bifurcation rather than a simple fold. "
                   "Branch-switching or a specialized unfolding would be needed to proceed "
                   "past this point with continuation.")
    else:
        verdict = ("Substantially worse-conditioned than the reference, but transversality is "
                   "still clearly nonzero -- consistent with being NEAR a simple fold (possibly "
                   "very close to, but not exactly at, a genuine limit point). Worth trying "
                   "smaller ds_min / relaxed newton_tol / more max_newton_it before concluding "
                   "this needs branch-switching.")

    verdict_lines.append("")
    verdict_lines.append("VERDICT: " + verdict)
    report = "\n".join(verdict_lines)
    if verbose:
        print(report, flush=True)
    _progress_log(f"diagnose_fold_point verdict: {verdict}")

    return dict(stuck=stuck, reference=reference, smallest_ratio=smallest_ratio,
                cond_ratio=cond_ratio, verdict=verdict, report=report)


def run_case_arclength_auto(overrides=None, eta_start=0.30, ds=0.01, ds_min=1e-4, ds_max=0.03,
                             checkpoint_path="/content/pemfc_m2_arclength_checkpoint.npz",
                             snapshot_dir=None, total_steps_budget=5000, steps_per_attempt=100,
                             svd_diagnostic_every=10, svd_out_path=None, diagnostics_dir=None,
                             refresh_step_ladder=(1e-4,),  # single entry: refresh_tangent is now a
                                                            # deterministic direct Jacobian computation,
                                                            # not parametrized by step size, so repeating
                                                            # it would just redo the same computation
                             newton_tol_ladder=(1e-5, 1e-4),
                             max_newton_it_ladder=(60, 100),
                             max_recovery_rounds=1, verbose=True, stats_out_path=None,
                             init_w_array=None):
    # Update 7: init_w_array, if given, is threaded straight through to run_case_arclength's
    # own bootstrap-warm-start (used only on the very first, not-yet-resumed call below -- once
    # a checkpoint exists every later call resumes from it and ignores this). Lets a caller that
    # already has a converged-but-not-at-eta_target field state (e.g. solve_to_eta's fixed-eta
    # Newton stalling right at a fold) hand it over directly instead of arclength re-tracing the
    # whole branch from eta=0.02.
    """Fully automated, unattended pseudo-arclength continuation. Repeatedly
    calls run_case_arclength() in batches of `steps_per_attempt`, and
    whenever a batch makes literally zero progress (the classic "stuck"
    symptom seen manually several times: ds shrinks to ds_min repeatedly
    with eta frozen), automatically works through the SAME recovery
    ladder that was previously done by hand:

      1. refresh_tangent=True, trying refresh_step_ladder in order (each
         value guarded against branch-jumping by run_case_arclength's own
         built-in check, which raises rather than corrupting the
         checkpoint -- caught here and treated as "try the next, smaller
         refresh_step").
      2. If refresh doesn't help, relaxed (newton_tol, max_newton_it)
         pairs from the ladders, on the theory that a moderately (not
         extremely) ill-conditioned point just needs more/looser Newton
         iterations rather than a different tangent.

    Deliberately does NOT attempt rollback_checkpoint automatically --
    unlike the steps above (which either succeed or safely no-op),
    deciding how far to roll back a bad excursion requires judgment
    (see the manual sessions this was built from) and is too risky to
    automate blindly during an unattended multi-hour run. If the full
    ladder is exhausted `max_recovery_rounds` times without meaningful
    progress, this stops gracefully -- the checkpoint is left exactly as
    the last successful run_case_arclength() call left it (never mid-
    write) -- and AUTOMATICALLY runs diagnose_fold_point() (read-only,
    doesn't touch the checkpoint) before returning, logging its verdict
    so there's no separate manual step needed just to see whether the
    stall is a genuine higher-codimension point or something a bad
    excursion / wider recovery ladder could still fix. Actually acting on
    a genuine higher-codimension verdict (branch-switching) still needs a
    human -- that technique isn't implemented here.

    snapshot_dir, if given, is passed straight through to
    run_case_arclength() -- the full Fig. 7 field set (s, p, Cw, CgO2,
    CgH2O, C_O2_mix) is saved for EVERY accepted step (not just the
    checkpoint's latest one), so any point along the branch can be
    plotted after the fact via a fresh save_field_snapshot()-style load
    of the relevant step_*.npz file, without needing to re-trace anything.

    Returns the total number of accepted steps in the checkpoint when
    this returns (whether by reaching total_steps_budget or by stopping
    stuck).
    """
    def _count():
        if os.path.exists(checkpoint_path):
            try:
                return len(np.load(checkpoint_path)["eta_hist"])
            except Exception:
                return 0
        return 0

    # Recovery-ladder usage statistics -- how often each strategy was
    # actually needed, for a "practical robustness" discussion in the
    # paper. Persists across resumed sessions by loading whatever's
    # already at stats_out_path.
    stats = dict(n_plain_attempts=0, n_plain_success=0, n_stuck_events=0,
                 n_refresh_success=0, n_tolerance_success=0, n_give_up=0)
    if stats_out_path and os.path.exists(stats_out_path):
        try:
            import json
            with open(stats_out_path) as f:
                stats.update(json.load(f))
            _progress_log(f"  loaded prior recovery-ladder stats from {stats_out_path}")
        except Exception as e:
            _progress_log(f"  could not load prior stats ({e}), starting fresh")

    def _save_stats():
        if stats_out_path:
            try:
                import json
                with open(stats_out_path, "w") as f:
                    json.dump(stats, f, indent=2)
            except Exception as e:
                _progress_log(f"  stats save failed (continuing anyway): {e}")

    total_accepted = _count()
    _progress_log(f"run_case_arclength_auto: starting, {total_accepted} steps already in checkpoint")

    def _eta_now():
        if os.path.exists(checkpoint_path):
            try:
                return float(np.load(checkpoint_path)["eta_last"])
            except Exception:
                return None
        return None

    # Minimum cumulative |eta| advance over a whole attempt (up to
    # steps_per_attempt steps) to count as REAL progress -- not just
    # "some new steps got accepted". Without this, a region where the
    # tangent's eta-component (deta_tan) is near-zero (confirmed to
    # happen at the second stuck point) can keep accepting steps
    # indefinitely -- each a genuine, converged, non-branch-jumping step,
    # advancing true arc-length -- while eta itself stays essentially
    # frozen to 4-5 decimal places, so total_accepted keeps climbing and
    # the old check (total_accepted > prev_total) never detects the stall.
    min_eta_progress = 20 * ds_min

    recovery_rounds = 0
    while total_accepted < total_steps_budget:
        resume = os.path.exists(checkpoint_path)
        prev_total = total_accepted
        prev_eta = _eta_now()
        stats["n_plain_attempts"] += 1
        try:
            result = run_case_arclength(
                overrides=overrides, eta_start=eta_start, ds=ds, ds_min=ds_min, ds_max=ds_max,
                n_steps=steps_per_attempt, resume=resume, checkpoint_path=checkpoint_path,
                snapshot_dir=snapshot_dir, verbose=verbose, init_w_array=init_w_array,
                svd_diagnostic_every=svd_diagnostic_every, svd_out_path=svd_out_path)
            total_accepted = len(result["eta"])
        except Exception as e:
            _progress_log(f"run_case_arclength_auto: plain attempt raised {type(e).__name__}: {e}")
            total_accepted = _count()

        eta_progress = None
        if prev_eta is not None:
            eta_now = _eta_now()
            if eta_now is not None:
                eta_progress = abs(eta_now - prev_eta)

        if total_accepted > prev_total and (eta_progress is None or eta_progress >= min_eta_progress):
            stats["n_plain_success"] += 1
            _save_stats()
            recovery_rounds = 0  # making real progress -- reset the stuck counter
            continue

        if total_accepted > prev_total and eta_progress is not None and eta_progress < min_eta_progress:
            _progress_log(f"run_case_arclength_auto: {total_accepted - prev_total} step(s) accepted but "
                          f"eta only advanced {eta_progress:.2e} (< {min_eta_progress:.2e} threshold) -- "
                          f"treating as STUCK despite nominal progress (near-zero deta_tan region).")

        # --- stuck: zero (or negligible) progress in a plain attempt. ---
        # --- Work through the recovery ladder before giving up.       ---
        stats["n_stuck_events"] += 1
        _save_stats()
        _progress_log(f"run_case_arclength_auto: STUCK at {total_accepted} accepted steps "
                      f"-- starting recovery ladder (round {recovery_rounds + 1}/{max_recovery_rounds})")
        recovered_this_round = False
        eta_before_ladder = _eta_now()

        for refresh_step_try in refresh_step_ladder:
            _progress_log(f"  [ladder: refresh] now attempting refresh_step={refresh_step_try:.1e} "
                          f"(n_steps={steps_per_attempt})")
            try:
                result = run_case_arclength(
                    overrides=overrides, eta_start=eta_start, ds=ds, ds_min=ds_min, ds_max=ds_max,
                    n_steps=steps_per_attempt, resume=True, checkpoint_path=checkpoint_path,
                    refresh_tangent=True, refresh_step=refresh_step_try,
                    snapshot_dir=snapshot_dir, verbose=verbose, init_w_array=init_w_array,
                svd_diagnostic_every=svd_diagnostic_every, svd_out_path=svd_out_path)
                new_total = len(result["eta"])
            except Exception as e:
                _progress_log(f"  refresh_step={refresh_step_try:.1e} raised "
                              f"{type(e).__name__}: {e} -- trying next")
                new_total = _count()
                continue
            eta_progress_ladder = None
            if eta_before_ladder is not None:
                eta_now_ladder = _eta_now()
                if eta_now_ladder is not None:
                    eta_progress_ladder = abs(eta_now_ladder - eta_before_ladder)
            decisive = new_total > total_accepted + 1 and (
                eta_progress_ladder is None or eta_progress_ladder >= min_eta_progress)
            if decisive:
                # +1 tolerance: the refresh bootstrap itself always adds
                # exactly one accepted point even when the main loop
                # after it can't proceed -- only count this as real
                # recovery if it went further than just that AND eta
                # actually advanced meaningfully (not just more steps
                # accepted in a near-zero-deta_tan region).
                _progress_log(f"  recovered via refresh_step={refresh_step_try:.1e}: "
                              f"{total_accepted} -> {new_total}")
                total_accepted = new_total
                recovered_this_round = True
                stats["n_refresh_success"] += 1
                _save_stats()
                break
            if new_total > total_accepted + 1 and eta_progress_ladder is not None:
                _progress_log(f"  refresh_step={refresh_step_try:.1e}: {total_accepted} -> {new_total} "
                              f"steps accepted, but eta only advanced {eta_progress_ladder:.2e} "
                              f"(< {min_eta_progress:.2e}) -- NOT counting as real recovery")
            _progress_log(f"  refresh_step={refresh_step_try:.1e} did not decisively recover "
                          f"({total_accepted} -> {new_total}, only the refresh's own bootstrap "
                          f"point) -- trying next")
            total_accepted = new_total

        if recovered_this_round:
            recovery_rounds = 0
            continue

        _progress_log("  [ladder: refresh] exhausted with no decisive recovery -- "
                      "moving to relaxed-tolerance ladder")
        for tol_try, max_it_try in zip(newton_tol_ladder, max_newton_it_ladder):
            _progress_log(f"  [ladder: tolerance] now attempting newton_tol={tol_try:.1e}, "
                          f"max_newton_it={max_it_try} (n_steps={steps_per_attempt})")
            try:
                result = run_case_arclength(
                    overrides=overrides, eta_start=eta_start, ds=ds, ds_min=ds_min, ds_max=ds_max,
                    n_steps=steps_per_attempt, resume=True, checkpoint_path=checkpoint_path,
                    newton_tol=tol_try, max_newton_it=max_it_try,
                    snapshot_dir=snapshot_dir, verbose=verbose, init_w_array=init_w_array,
                svd_diagnostic_every=svd_diagnostic_every, svd_out_path=svd_out_path)
                new_total = len(result["eta"])
            except Exception as e:
                _progress_log(f"  newton_tol={tol_try:.1e}/max_it={max_it_try} raised "
                              f"{type(e).__name__}: {e} -- trying next")
                new_total = _count()
                continue
            eta_progress_ladder = None
            if eta_before_ladder is not None:
                eta_now_ladder = _eta_now()
                if eta_now_ladder is not None:
                    eta_progress_ladder = abs(eta_now_ladder - eta_before_ladder)
            decisive = new_total > total_accepted and (
                eta_progress_ladder is None or eta_progress_ladder >= min_eta_progress)
            if decisive:
                _progress_log(f"  progress via newton_tol={tol_try:.1e}, max_it={max_it_try}: "
                              f"{total_accepted} -> {new_total}")
                total_accepted = new_total
                recovered_this_round = True
                stats["n_tolerance_success"] += 1
                _save_stats()
                break
            if new_total > total_accepted and eta_progress_ladder is not None:
                _progress_log(f"  newton_tol={tol_try:.1e}, max_it={max_it_try}: "
                              f"{total_accepted} -> {new_total} steps accepted, but eta only "
                              f"advanced {eta_progress_ladder:.2e} (< {min_eta_progress:.2e}) "
                              f"-- NOT counting as real recovery")
            else:
                _progress_log(f"  newton_tol={tol_try:.1e}/max_it={max_it_try} made no progress "
                              f"({total_accepted} -> {new_total}) -- trying next")
            total_accepted = new_total
            total_accepted = new_total

        if recovered_this_round:
            recovery_rounds = 0
            continue

        recovery_rounds += 1
        if recovery_rounds >= max_recovery_rounds:
            stats["n_give_up"] += 1
            _save_stats()
            # Bug fix: if EVERY attempt (the plain attempt plus the whole recovery ladder)
            # failed before even a single step was ever accepted, checkpoint_path was never
            # created at all -- confirmed the hard way on a real run: this crashed here with an
            # uncaught FileNotFoundError when a caller warm-started the bootstrap (init_w_array)
            # right at a fold, since bootstrap solve 2 is itself just an ordinary fixed-eta
            # Newton step (eta_start + ds) -- exactly the kind of step that can already be known
            # to fail at that location. Report using eta_start (the point the bootstrap was
            # trying to start from) instead of assuming a checkpoint always exists by this point.
            if os.path.exists(checkpoint_path):
                eta_stuck = float(np.load(checkpoint_path)["eta_last"])
                stuck_note = ""
            else:
                eta_stuck = eta_start
                stuck_note = (" (never produced even one accepted step -- most likely the "
                               "bootstrap itself failing right at eta_start, not a mid-branch "
                               "stall)")
            _progress_log(
                f"run_case_arclength_auto: recovery ladder exhausted {recovery_rounds} time(s) "
                f"in a row with no progress -- STOPPING at {total_accepted} accepted steps "
                f"(eta={eta_stuck:.5f}){stuck_note}. Checkpoint is safe and unmodified beyond "
                f"this point. Running diagnose_fold_point() automatically to characterize the "
                f"stuck point...")
            if verbose:
                print(f"\n[run_case_arclength_auto] STOPPED, stuck after exhausting recovery "
                      f"{recovery_rounds} time(s). {total_accepted} accepted steps in checkpoint. "
                      f"Running diagnose_fold_point() automatically...", flush=True)

            nullvec_path = None
            outlet_path = None
            if diagnostics_dir:
                os.makedirs(diagnostics_dir, exist_ok=True)
                nullvec_path = os.path.join(diagnostics_dir, f"nullvec_eta_{eta_stuck:.5f}.npz")
                outlet_path = os.path.join(diagnostics_dir, f"outletflow_eta_{eta_stuck:.5f}.npz")

            try:
                diag = diagnose_fold_point(checkpoint_path, overrides=overrides, verbose=verbose,
                                            save_nullvec_path=nullvec_path)
                _progress_log(f"run_case_arclength_auto: automatic diagnosis verdict -- "
                              f"{diag['verdict']}")
                if verbose:
                    print(f"\n[run_case_arclength_auto] Automatic diagnosis: {diag['verdict']}",
                          flush=True)
                    if diag["cond_ratio"] < 20:
                        print("[run_case_arclength_auto] -> NOT a genuine singularity by this "
                              "check. A bad excursion may have slipped through undetected -- "
                              "consider rollback_checkpoint() before retrying, or widen the "
                              "recovery ladders (refresh_step_ladder / newton_tol_ladder) and "
                              "call run_case_arclength_auto() again.", flush=True)
                    else:
                        print("[run_case_arclength_auto] -> Looks like a genuine higher-"
                              "codimension point (fold/cusp/bifurcation). This automated "
                              "continuation cannot proceed past it -- branch-switching or a "
                              "specialized unfolding would be needed, which isn't implemented "
                              "here.", flush=True)
            except Exception as e:
                _progress_log(f"run_case_arclength_auto: automatic diagnosis itself failed "
                              f"({type(e).__name__}: {e}) -- run diagnose_fold_point() manually.")
                if verbose:
                    print(f"[run_case_arclength_auto] Automatic diagnosis failed ({e}) -- "
                          f"run diagnose_fold_point() manually.", flush=True)

            try:
                check_outlet_flow(checkpoint_path, overrides=overrides, verbose=verbose,
                                   save_path=outlet_path)
            except Exception as e:
                _progress_log(f"run_case_arclength_auto: automatic check_outlet_flow failed "
                              f"({type(e).__name__}: {e}) -- run check_outlet_flow() manually.")
                if verbose:
                    print(f"[run_case_arclength_auto] Automatic check_outlet_flow failed ({e}) "
                          f"-- run check_outlet_flow() manually.", flush=True)
            break

    _progress_log(f"run_case_arclength_auto: returning with {total_accepted} total accepted steps")
    _save_stats()
    if verbose:
        print(f"\n[run_case_arclength_auto] Recovery-ladder usage summary: {stats}", flush=True)
    return total_accepted



def rollback_checkpoint(checkpoint_path, overrides=None, n_pop=1, verbose=True):
    """Undo the last `n_pop` accepted step(s) recorded in checkpoint_path's
    history (e.g. a bad excursion caused by refresh_tangent bootstrapping
    in the wrong eta direction -- confirmed to happen when the branch is
    locally decreasing in eta but the refresh always tried "+ds" for its
    second point, landing on a discontinuous, off-branch solution rather
    than a valid nearby point).

    Since only the MOST RECENT solution vector is kept in the checkpoint
    (not one per history entry), rolling back can't just restore an old
    w array -- it re-SOLVES (ordinary Newton, warm-started from whatever
    is currently in the checkpoint) at the eta value that becomes the new
    "last" entry after popping, and overwrites w_last/eta_last with that
    fresh solution. The old (possibly-bad) tangent is left in the
    checkpoint as-is; follow this call with resume=True,
    refresh_tangent=True before continuing, since a rolled-back point's
    saved tangent is not trustworthy.
    """
    P = dict(DEFAULT_PARAMS)
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])

    ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)

    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2

    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0
    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix

    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)

    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)
    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    Weff_w = lam_l + lam_g * CgH2O_local
    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
               - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
    F_total = F_pres + F_water + F_oxy

    fc_opts = {"quadrature_degree": 4}
    snes_petsc_options = {
        "snes_type": "newtonls", "snes_rtol": P["newton_rtol"], "snes_atol": P["newton_atol"],
        "snes_max_it": P["newton_max_it"], "snes_error_if_not_converged": False,
        "ksp_type": "preonly", "pc_type": "lu",
    }
    solve_problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                      petsc_options_prefix="pemfc_rollback_",
                                      form_compiler_options=fc_opts)

    d = np.load(checkpoint_path)
    eta_hist = list(d["eta_hist"])
    I_hist = list(d["I_hist"])
    smax_hist = list(d["smax_hist"])
    ds_hist = list(d["ds_hist"])
    w_current = d["w_last"]

    dropped = []
    for _ in range(n_pop):
        dropped.append((eta_hist.pop(), I_hist.pop(), smax_hist.pop(), ds_hist.pop()))
    if verbose:
        for de, dI, dsm, _ in dropped:
            print(f"Dropped: eta={de:.5f}  I={dI/1e4:.4f} A/cm^2  s_max={dsm:.3f}", flush=True)

    good_eta = eta_hist[-1]
    if verbose:
        print(f"\nRolling back to eta={good_eta:.5f} (re-solving fresh)...", flush=True)
    _progress_log(f"rollback_checkpoint: popped {n_pop}, re-solving at eta={good_eta:.5f}")

    eta_const.value = good_eta
    w.x.array[:] = w_current
    w.x.scatter_forward()
    solve_problem.solve()
    converged = solve_problem.solver.getConvergedReason() > 0
    if not converged:
        if verbose:
            print("  warm-started solve did not converge, retrying from a naive initial "
                  "guess...", flush=True)
        w.sub(0).interpolate(lambda x: np.full(x.shape[1], (p_in - p_out) / P_scale))
        w.sub(1).interpolate(lambda x: np.full(x.shape[1], Y_H2O_in))
        w.sub(2).interpolate(lambda x: np.full(x.shape[1], Y_O2_in))
        w.x.scatter_forward()
        solve_problem.solve()
        converged = solve_problem.solver.getConvergedReason() > 0

    if not converged:
        raise RuntimeError(f"rollback_checkpoint: could not re-solve at eta={good_eta} "
                            f"either warm-started or from a naive guess -- checkpoint left "
                            f"UNCHANGED on disk, nothing was overwritten.")

    w_good = w.x.array.copy()
    if verbose:
        print(f"  re-solve converged. Saving repaired checkpoint at eta={good_eta:.5f}.",
              flush=True)

    if np.all(np.isfinite(w_good)):
        np.savez(checkpoint_path, w_last=w_good, eta_last=good_eta,
                 dw_tan=d["dw_tan"], deta_tan=d["deta_tan"],
                 eta_hist=np.array(eta_hist), I_hist=np.array(I_hist),
                 smax_hist=np.array(smax_hist), ds_hist=np.array(ds_hist),
                 cur_ds=float(d["cur_ds"]))
        _progress_log(f"rollback_checkpoint: repaired checkpoint saved at eta={good_eta:.5f}")
        if verbose:
            print("\nDone. The saved tangent is now STALE (it was for the point you just "
                  "dropped) -- call run_case_arclength(resume=True, refresh_tangent=True, "
                  "...) next, not a plain resume.", flush=True)
    else:
        raise RuntimeError("rollback_checkpoint: re-solved state contains NaN/Inf -- "
                            "refusing to save, checkpoint left UNCHANGED on disk.")

    return dict(eta=good_eta, w=w_good)
def save_field_snapshot(checkpoint_path, out_path, overrides=None, verbose=True):
    """Extract and save the full 2D field snapshot (all four Fig. 7
    panels -- water mixture C^{H2O}, water vapor Cg^{H2O}, oxygen mixture
    C^{O2}, gas oxygen mass fraction Cg^{O2} -- plus s and p for
    completeness) at whatever point is currently saved in checkpoint_path
    (from run_case_arclength). Read-only: does not modify the checkpoint.

    Field definitions match the paper (Table 1 / Sec. 2.1): water vapor
    Cg^{H2O} = min(C^{H2O}, Cg_sat) (paper's own two-phase equilibrium
    condition -- gas-phase water saturates once liquid forms, same
    min_smooth() expression used internally for the model's Weff_w term);
    oxygen mixture C^{O2} = (rho_g/rho_mix) * Cg^{O2} (from rho*C^{O2} =
    rho_g*Cg^{O2}, Eq. 11 with oxygen existing only in the gas phase).
    """
    P = dict(DEFAULT_PARAMS)
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])
    dx = ufl.Measure("dx", domain=domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)

    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    CgH2O_local = min_smooth(Cw_sol, Cg_sat)          # Fig. 7b: water vapor
    C_O2_mix = rho_g_const * CgO2_sol / rho_mix         # Fig. 7c: oxygen mixture

    ckpt = np.load(checkpoint_path)
    w.x.array[:] = ckpt["w_last"]
    w.x.scatter_forward()
    eta_val = float(ckpt["eta_last"])
    if verbose:
        print(f"Extracting field snapshot at eta={eta_val:.5f} from {checkpoint_path}", flush=True)

    coords = V.tabulate_dof_coordinates()[:, :2]
    p_arr = _project_scalar_standalone(p_sol, V, dx)
    Cw_arr = _project_scalar_standalone(Cw_sol, V, dx)
    CgO2_arr = _project_scalar_standalone(CgO2_sol, V, dx)
    CgH2O_arr = _project_scalar_standalone(CgH2O_local, V, dx)
    CO2mix_arr = _project_scalar_standalone(C_O2_mix, V, dx)
    s_arr = _project_scalar_standalone(s, V, dx)
    deta_sign = float(ckpt["deta_tan"]) if "deta_tan" in ckpt.files else 1.0
    branch_state = _classify_branch_state(eta_val, float(s_arr.max()), deta_sign)

    np.savez(out_path, coords=coords, eta=eta_val, p=p_arr, Cw=Cw_arr, CgO2=CgO2_arr,
             CgH2O=CgH2O_arr, C_O2_mix=CO2mix_arr, s=s_arr, branch_state=branch_state)
    if verbose:
        print(f"Saved to {out_path}", flush=True)
        print(f"  branch_state: {branch_state}")
        print(f"  s range: {s_arr.min():.3f} - {s_arr.max():.3f}")
        print(f"  Cw range: {Cw_arr.min():.3f} - {Cw_arr.max():.3f}")
        print(f"  CgO2 range: {CgO2_arr.min():.4f} - {CgO2_arr.max():.4f}")
        print(f"  CgH2O (vapor) range: {CgH2O_arr.min():.4f} - {CgH2O_arr.max():.4f}")
        print(f"  C_O2_mix range: {CO2mix_arr.min():.4f} - {CO2mix_arr.max():.4f}")

    return dict(eta=eta_val, coords=coords, p=p_arr, Cw=Cw_arr, CgO2=CgO2_arr,
                CgH2O=CgH2O_arr, C_O2_mix=CO2mix_arr, s=s_arr)
def check_outlet_flow(checkpoint_path, overrides=None, y_band=None, verbose=True, save_path=None):
    """Diagnostic for the "degenerate outflow BC" hypothesis: the outlet
    boundary condition in F_oxy (and F_water) is a natural (Neumann-type)
    ADVECTIVE outflow condition -- CgO2*inner(W_O2, n)*v on ds(OUTLET) --
    which implicitly assumes a genuine outflow there. If the local normal
    velocity approaches zero or changes sign as flooding intensifies, that
    boundary condition's own well-posedness degrades, which would show up
    exactly as a spatially-localized Jacobian null direction concentrated
    at the outlet -- matching what psi_CgO2's spatial plot showed (top-10%
    energy concentrated at y=1.77-2.00mm, squarely inside the outlet
    segment y in [1.6mm, 2.0mm] for the default geometry, spanning the
    FULL GDL thickness rather than being membrane-localized).

    Projects the mixture Darcy velocity u_darcy and the gas-phase oxygen
    advective velocity W_O2 onto the scalar space, and reports statistics
    (min/max/sign) of their y-component (the component that matters for
    the outlet's normal flux, since the outlet is the y=Ly edge) within
    `y_band` (defaults to the outlet segment itself, [y_rib_end, Ly]).
    """
    P = dict(DEFAULT_PARAMS)
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb
    if y_band is None:
        y_band = (y_rib_end, Ly)

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])
    dx = ufl.Measure("dx", domain=domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)

    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    theta_c = theta_c_deg * np.pi / 180.0
    hydrophobic = theta_c_deg >= 90.0
    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix
    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
    W_O2 = rho_mix * lam_g * u_darcy

    ckpt = np.load(checkpoint_path)
    w.x.array[:] = ckpt["w_last"]
    w.x.scatter_forward()
    eta_val = float(ckpt["eta_last"])

    coords = V.tabulate_dof_coordinates()[:, :2]
    uy = _project_scalar_standalone(u_darcy[1], V, dx)
    Wo2y = _project_scalar_standalone(W_O2[1], V, dx)
    s_arr = _project_scalar_standalone(s, V, dx)
    CgO2_arr = _project_scalar_standalone(CgO2_sol, V, dx)

    band_mask = (coords[:, 1] >= y_band[0]) & (coords[:, 1] <= y_band[1])
    n_band = int(band_mask.sum())

    if verbose:
        print(f"eta = {eta_val:.5f}, checking y in [{y_band[0]*1e3:.3f}, {y_band[1]*1e3:.3f}] mm "
              f"({n_band} dofs)", flush=True)
        print(f"  u_darcy_y in band: min={uy[band_mask].min():.4e}  max={uy[band_mask].max():.4e}  "
              f"(sign change: {uy[band_mask].min() < 0 < uy[band_mask].max()})")
        print(f"  W_O2_y (gas oxygen advective velocity) in band: "
              f"min={Wo2y[band_mask].min():.4e}  max={Wo2y[band_mask].max():.4e}  "
              f"(sign change: {Wo2y[band_mask].min() < 0 < Wo2y[band_mask].max()})")
        print(f"  s in band: min={s_arr[band_mask].min():.4f}  max={s_arr[band_mask].max():.4f}")
        print(f"  CgO2 in band: min={CgO2_arr[band_mask].min():.4f}  max={CgO2_arr[band_mask].max():.4f}")
        print(f"  (for context, over the WHOLE domain) u_darcy_y: "
              f"min={uy.min():.4e} max={uy.max():.4e}   W_O2_y: min={Wo2y.min():.4e} max={Wo2y.max():.4e}")
        if Wo2y[band_mask].min() < 0 < Wo2y[band_mask].max():
            print("\n  -> W_O2's normal (y) component CHANGES SIGN inside the outlet band -- "
                  "directly supports the degenerate-outflow-BC hypothesis.")
        elif abs(Wo2y[band_mask]).max() < 0.05 * abs(Wo2y).max():
            print("\n  -> W_O2's normal (y) component is much smaller in this band than "
                  "elsewhere in the domain (near-stagnant) -- also consistent with the "
                  "outflow BC becoming numerically marginal there.")
        else:
            print("\n  -> No sign change or clear near-stagnation found in this band -- "
                  "the degenerate-outflow-BC hypothesis is not directly supported by this check.")

    if save_path:
        np.savez(save_path, eta=eta_val, coords=coords, u_darcy_y=uy, W_O2_y=Wo2y,
                 s=s_arr, CgO2=CgO2_arr, y_band=np.array(y_band))
        if verbose:
            print(f"Saved to {save_path}", flush=True)

    return dict(eta=eta_val, coords=coords, u_darcy_y=uy, W_O2_y=Wo2y, s=s_arr, CgO2=CgO2_arr,
                band_mask=band_mask)
def fold_svd_sweep(eta_list, overrides=None, out_path=None, verbose=True,
                    max_newton_it=50, resume=True):
    """Sweep eta (ordinary Newton continuation, warm-started from the
    previous point) and record the row-norm-equilibrated Jacobian's
    smallest singular value and condition number at each converged
    point. Dedicated, standalone re-implementation of the ad hoc sweep
    originally used to first discover the genuine fold near eta~0.36 (at
    the time, only logged via _progress_log, never saved as reusable
    data) -- built for reproducibly regenerating that figure.

    eta_list should be sorted and start comfortably below the fold (e.g.
    0.30) with fine spacing near where the fold is expected (e.g. 0.005
    steps dropping to 0.001 above ~0.34) -- ordinary Newton continuation
    cannot converge AT or PAST a genuine fold, so the sweep should stop
    naturally (remaining points just won't converge) once it's close
    enough; this is fine, expected behavior, not a bug.

    max_newton_it (default 50): worst-case iteration cap per point.
    Lower this (e.g. to 15-20) on expensive meshes (refined) where a
    single difficult point near the fold -- or near the second,
    higher-codimension obstruction further down the branch, which this
    sweep can also run into if eta_list extends that far -- can otherwise
    take many tens of minutes for one point alone.

    resume (default True): if out_path already has saved data, resume
    from there instead of starting over from eta_list[0] -- skips any
    eta_list values already covered (within 1e-6) and warm-starts from
    the last existing point. Saves incrementally (after every point, not
    just at the end) specifically so an interrupted run doesn't lose
    already-computed points -- this sweep can take hours near a
    difficult region, and re-running it from scratch after an
    interruption previously wasted all of that time.

    Returns dict with eta, sv_smallest, sv_largest, cond, I_avg, s_max
    (arrays, one entry per CONVERGED eta -- non-converged points are
    silently skipped) and saves the same to out_path if given. I_avg/
    s_max are recorded alongside the SVD diagnostics specifically so
    this ALSO doubles as a standard-continuation (eta, I) trace directly
    comparable against the arclength-traced branch on the same plot --
    showing exactly where ordinary continuation stops relative to how
    far pseudo-arclength continuation is able to go.
    """
    P = dict(DEFAULT_PARAMS)
    if overrides:
        P.update(overrides)

    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx)
        facet_markers.append(np.full_like(idx, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order], facet_markers[order])

    ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)
    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0
    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix
    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)
    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)
    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    Weff_w = lam_l + lam_g * CgH2O_local
    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
               - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
    F_total = F_pres + F_water + F_oxy

    snes_petsc_options = {
        "snes_type": "newtonls", "snes_rtol": 1e-7, "snes_atol": 1e-9,
        "snes_max_it": max_newton_it, "snes_error_if_not_converged": False,
        "ksp_type": "preonly", "pc_type": "lu",
    }
    problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                petsc_options_prefix="pemfc_svdsweep_",
                                form_compiler_options={"quadrature_degree": 4})

    w.sub(0).interpolate(lambda x: np.full(x.shape[1], (p_in - p_out) / P_scale))
    w.sub(1).interpolate(lambda x: np.full(x.shape[1], Y_H2O_in))
    w.sub(2).interpolate(lambda x: np.full(x.shape[1], Y_O2_in))
    w.x.scatter_forward()

    # Warm-up ramp: jumping straight to eta_list[0] (e.g. 0.30) from this
    # naive/inlet-like guess is NOT reliable (confirmed: SNES_DIVERGED_
    # LINE_SEARCH) -- every other place in this codebase that starts a
    # naive-guess solve (run_case's default eta_list) ramps up gradually
    # from a much smaller eta first. Reuse that same warm-up here,
    # silently (not recorded in the output), before the requested sweep.
    eta_list = list(eta_list)
    eta_out, sv_small_out, sv_large_out, cond_out, I_out, smax_out = [], [], [], [], [], []

    if resume and out_path and os.path.exists(out_path):
        try:
            _prior = np.load(out_path)
            eta_out = list(_prior["eta"])
            sv_small_out = list(_prior["sv_smallest"])
            sv_large_out = list(_prior["sv_largest"])
            cond_out = list(_prior["cond"])
            I_out = list(_prior["I_avg"]) if "I_avg" in _prior.files else [np.nan] * len(eta_out)
            smax_out = list(_prior["s_max"]) if "s_max" in _prior.files else [np.nan] * len(eta_out)
            covered = set(np.round(eta_out, 6))
            n_before = len(eta_list)
            eta_list = [e for e in eta_list if round(e, 6) not in covered]
            if verbose:
                for i in range(len(eta_out)):
                    print(f"eta={eta_out[i]:.5f}: smallest_sval={sv_small_out[i]:.4e}  "
                          f"cond={cond_out[i]:.4e}  I_avg={I_out[i]/1e4:.4f} A/cm^2  "
                          f"s_max={smax_out[i]:.4f}  (loaded from prior run)", flush=True)
                print(f"Resuming: {len(eta_out)} point(s) already in {out_path}, "
                      f"{n_before - len(eta_list)} requested point(s) already covered "
                      f"(skipping), {len(eta_list)} remaining.", flush=True)
        except Exception as e:
            if verbose:
                print(f"Could not load prior sweep data ({e}) -- starting fresh.", flush=True)
            eta_out, sv_small_out, sv_large_out, cond_out, I_out, smax_out = [], [], [], [], [], []

    if not eta_list:
        if verbose:
            print("Nothing left to do -- all requested points already covered.", flush=True)
        return dict(eta=np.array(eta_out), sv_smallest=np.array(sv_small_out),
                    sv_largest=np.array(sv_large_out), cond=np.array(cond_out),
                    I_avg=np.array(I_out), s_max=np.array(smax_out))

    def _save_progress():
        if out_path:
            np.savez(out_path, eta=np.array(eta_out), sv_smallest=np.array(sv_small_out),
                     sv_largest=np.array(sv_large_out), cond=np.array(cond_out),
                     I_avg=np.array(I_out), s_max=np.array(smax_out))

    # Warm-up ramp: jumping straight to the first remaining target from a
    # naive/inlet-like guess is NOT reliable (confirmed: SNES_DIVERGED_
    # LINE_SEARCH) -- every other place in this codebase that starts a
    # naive-guess solve (run_case's default eta_list) ramps up gradually
    # from a much smaller eta first. Reuse that same warm-up here,
    # silently (not recorded in the output), before the requested sweep.
    # If resuming, ramp all the way up to the LAST already-covered point
    # instead of from scratch -- we don't save full field state in this
    # sweep's output, only scalar diagnostics, so a valid warm-start
    # state has to be reconstructed either way.
    first_target = eta_out[-1] if eta_out else eta_list[0]
    eta_prev = None  # tracks wherever `problem`'s current solved state actually is,
                      # right before the main loop -- used as the step-halving base for
                      # the very first target point (previously left as None here, which
                      # silently made eta_start_for_step == eta_target for that point,
                      # collapsing the halving formula into a no-op: eta_now never
                      # actually got smaller, so a failed first attempt looked like "even
                      # the smallest sub-step failed" after only ONE real attempt)
    if first_target > 0.05:
        warmup = list(np.linspace(0.02, first_target, 8))[:-1]  # exclude the
                                                                   # last point,
                                                                   # first_target
                                                                   # covers it
        if verbose:
            print(f"Warming up from eta=0.02 to eta={first_target:.4f} "
                  f"before the requested sweep...", flush=True)
        _progress_log(f"fold_svd_sweep: warming up from eta=0.02 to eta={first_target:.4f} "
                      f"({len(warmup)} steps) before the requested sweep")
        for eta_now in warmup:
            eta_const.value = eta_now
            problem.solve()
            if problem.solver.getConvergedReason() <= 0:
                _progress_log(f"fold_svd_sweep: warm-up FAILED at eta={eta_now:.4f}")
                raise RuntimeError(
                    f"fold_svd_sweep: warm-up solve failed at eta={eta_now:.4f} -- "
                    f"cannot proceed to the requested sweep.")
            _progress_log(f"fold_svd_sweep: warm-up step converged at eta={eta_now:.4f}")
        if warmup:
            eta_prev = warmup[-1]  # the actual eta problem is currently solved at
    if eta_out:
        # Solve AT the resume point itself, to get a genuinely valid
        # warm-start state for the next (new) target below -- needed
        # regardless of the warmup ramp above (this sweep doesn't save
        # full field state, only scalar diagnostics, so this
        # reconstruction step is unconditional when resuming).
        _progress_log(f"fold_svd_sweep: re-solving at the resume point eta={first_target:.4f}")
        eta_const.value = first_target
        problem.solve()
        if problem.solver.getConvergedReason() <= 0:
            _progress_log(f"fold_svd_sweep: FAILED to re-solve at resume point eta={first_target:.4f}")
            raise RuntimeError(
                f"fold_svd_sweep: could not re-solve at the resume point "
                f"eta={first_target:.4f} -- cannot safely continue.")
        eta_prev = first_target  # overrides warmup's eta_prev -- this solve is exactly
                                  # at first_target, an even better base than warmup's

    for eta_target in eta_list:
        # Adaptive step-halving toward eta_target: if the full step fails
        # to converge, retry with progressively smaller steps from the
        # last successfully converged point, so the sweep can get as
        # close as possible to the fold before truly giving up. Added
        # because the first version of this sweep stopped well short of
        # the fold -- eta=0.355 vs the fold at ~0.36 -- missing almost
        # all of the dramatic singular-value collapse.
        eta_start_for_step = eta_prev if eta_prev is not None else eta_target
        eta_now = eta_target
        min_substep = 1e-5
        converged_this_target = False
        while True:
            eta_const.value = eta_now
            problem.solve()
            reason = problem.solver.getConvergedReason()
            if reason > 0:
                converged_this_target = (eta_now == eta_target)
                break
            step_size = abs(eta_now - eta_start_for_step)
            if step_size < min_substep:
                if verbose:
                    print(f"eta={eta_now:.5f}: did NOT converge (reason={reason}) even at "
                          f"the smallest sub-step tried -- stopping sweep here (expected "
                          f"once at/past a fold)", flush=True)
                _progress_log(f"fold_svd_sweep: stuck at eta={eta_now:.5f}, stopping")
                break
            eta_now = eta_start_for_step + 0.5 * (eta_now - eta_start_for_step)
            if verbose:
                print(f"  (target eta={eta_target:.5f} did not converge, retrying at "
                      f"eta={eta_now:.5f})", flush=True)
            _progress_log(f"fold_svd_sweep: target eta={eta_target:.5f} did not converge "
                          f"(reason={reason}), retrying at eta={eta_now:.5f}")

        if reason <= 0:
            break  # smallest sub-step also failed -- genuinely stuck, stop the whole sweep

        sv_small_raw, sv_large_raw, sv_small_sc, sv_large_sc = _jacobian_singular_values(problem)
        cond_sc = sv_large_sc[-1] / max(sv_small_sc[0], 1e-300)
        Jloc_arr = _project_scalar_standalone(I_local, V, dx)
        I_avg_now = float(np.mean(Jloc_arr[membrane_dofs_V]))
        s_arr = _project_scalar_standalone(s, V, dx)
        smax_now = float(s_arr.max())
        eta_out.append(eta_now)
        sv_small_out.append(sv_small_sc[0])
        sv_large_out.append(sv_large_sc[-1])
        cond_out.append(cond_sc)
        I_out.append(I_avg_now)
        smax_out.append(smax_now)
        eta_prev = eta_now
        _save_progress()
        if verbose:
            print(f"eta={eta_now:.5f}: smallest_sval={sv_small_sc[0]:.4e}  cond={cond_sc:.4e}  "
                  f"I_avg={I_avg_now/1e4:.4f} A/cm^2  s_max={smax_now:.4f}",
                  flush=True)
        _progress_log(f"fold_svd_sweep: eta={eta_now:.5f} smallest_sval={sv_small_sc[0]:.4e} "
                      f"cond={cond_sc:.4e} I_avg={I_avg_now/1e4:.4f} s_max={smax_now:.4f}")
        if not converged_this_target and verbose:
            print(f"Target eta={eta_target:.5f} not reached directly -- recorded the "
                  f"closest convergeable point instead (eta={eta_now:.5f}), continuing "
                  f"to the next target.", flush=True)

    result = dict(eta=np.array(eta_out), sv_smallest=np.array(sv_small_out),
                  sv_largest=np.array(sv_large_out), cond=np.array(cond_out),
                  I_avg=np.array(I_out), s_max=np.array(smax_out))
    if out_path:
        np.savez(out_path, **result)
        if verbose:
            print(f"\nSaved to {out_path}", flush=True)
    return result


def collect_diagnostic_summary(entries, out_path=None, verbose=True):
    """Run diagnose_fold_point() over a labeled list of checkpoints and
    compile the key comparison numbers into one clean summary table --
    e.g. the confirmed genuine fold vs. the second (bistable) stuck
    point, at coarse and refined mesh resolution -- ready for direct use
    building a paper table without hunting through progress logs.

    entries: list of dicts, each with:
        "label"      -- short name for this row (e.g. "second point, coarse mesh")
        "checkpoint" -- path to the checkpoint .npz to diagnose
        "overrides"  -- (optional) dict, e.g. {"nx": 36, "ny": 90}; must
                        match whatever mesh that checkpoint was generated with

    Saves (if out_path given) a JSON file (human-readable, easy to turn
    into a LaTeX/Word table directly) with one row per entry:
        label, eta, cond_ratio, smallest_sv_ratio, transversality, verdict

    Returns the same list of row-dicts.
    """
    rows = []
    for entry in entries:
        label = entry["label"]
        ckpt = entry["checkpoint"]
        overrides = entry.get("overrides")
        if verbose:
            print(f"\n{'='*70}\nDiagnosing: {label}  ({ckpt})\n{'='*70}", flush=True)
        try:
            diag = diagnose_fold_point(ckpt, overrides=overrides, verbose=verbose)
            rows.append(dict(
                label=label,
                checkpoint=ckpt,
                eta=diag["stuck"]["eta"],
                cond_ratio=diag["cond_ratio"],
                smallest_sv_ratio=diag["smallest_ratio"],
                transversality=diag["stuck"]["transversality_normalized"],
                verdict=diag["verdict"],
            ))
        except Exception as e:
            if verbose:
                print(f"  FAILED to diagnose {label}: {type(e).__name__}: {e}", flush=True)
            rows.append(dict(label=label, checkpoint=ckpt, error=str(e)))

    if verbose:
        print(f"\n{'='*70}\nSummary\n{'='*70}")
        for r in rows:
            if "error" in r:
                print(f"{r['label']}: FAILED ({r['error']})")
            else:
                print(f"{r['label']}: eta={r['eta']:.5f}  cond_ratio={r['cond_ratio']:.2f}x  "
                      f"transversality={r['transversality']:.4f}")

    if out_path:
        import json
        with open(out_path, "w") as f:
            json.dump(rows, f, indent=2)
        if verbose:
            print(f"\nSaved to {out_path}", flush=True)

    return rows


def run_case_to_target_eta(overrides=None, target_eta=0.55, eta_ramp_start=0.30,
                             ds=0.02, ds_min=1e-4, ds_max=0.1, steps_per_batch=5,
                             max_arclength_steps=300, eta_tol=1e-3, verbose=True):
    """Lightweight utility for parameter studies: reach target_eta (default
    0.55, matching this paper's own parametric-study comparison point --
    see e.g. "at eta=0.55" in every Fig. 11/12/14/16/17/19/20/22/24/25/27
    caption) as cheaply as possible, WITHOUT the heavy paper-backing-data
    infrastructure (no SVD tracking, no per-step field snapshots).

    Two-path strategy, since whether target_eta needs arclength at all
    depends on the parameter value -- the fold's location is NOT fixed,
    it can shift well above or below target_eta depending on the swept
    parameter (e.g. porosity, wettability):

    1. Try ordinary (eta-fixed) Newton continuation straight to
       target_eta first. Fast (a normal run_case() sweep), and works
       whenever this parameter combination's fold (if any) sits above
       target_eta.
    2. If that stalls before reaching target_eta, fall back to
       pseudo-arclength continuation (run_case_arclength), resuming from
       wherever ordinary Newton got to, stepping in small batches and
       checking after each whether eta has come within eta_tol of
       target_eta. Stops as soon as it does, or after
       max_arclength_steps, whichever first -- does NOT guarantee
       reaching target_eta (the traced branch may turn around and never
       come back to it, as confirmed for the base case's own branch,
       which peaks near eta~0.36 then decreases). Returns the CLOSEST
       approach either way, clearly flagged via target_reached.

    Returns dict with:
        eta_hist, I_hist, s_max_hist -- full accepted-step trace (ordinary
                                         Newton portion + arclength portion
                                         if used)
        target_reached  -- bool, True only if the closest approach was
                            within eta_tol of target_eta
        best_eta         -- actual eta of the returned field snapshot
        gap              -- |best_eta - target_eta|
        fields           -- dict of field arrays at best_eta (same layout
                             as save_field_snapshot's output)
        method           -- "newton" or "arclength", which path supplied
                             the returned fields
    """
    import tempfile

    newton_eta_list = np.concatenate([np.linspace(0.02, eta_ramp_start, 8),
                                       np.linspace(eta_ramp_start, target_eta, 6)])
    if verbose:
        print(f"[run_case_to_target_eta] Trying ordinary Newton up to eta={target_eta}...",
              flush=True)
    result = run_case(overrides=overrides, eta_list=newton_eta_list, verbose=False,
                       save_fields_at=(target_eta,))
    tag = f"eta{target_eta:.3f}"

    if tag in result["fields"] and abs(result["eta"][-1] - target_eta) < eta_tol:
        if verbose:
            print(f"[run_case_to_target_eta] Reached target directly via ordinary Newton "
                  f"(no fold before eta={target_eta} for this parameter set).", flush=True)
        return dict(eta_hist=result["eta"], I_hist=result["I_avg"], s_max_hist=result["s_max"],
                    target_reached=True, best_eta=float(result["eta"][-1]), gap=0.0,
                    fields=result["fields"][tag], method="newton")

    eta_reached_by_newton = float(result["eta"][-1]) if len(result["eta"]) else eta_ramp_start
    if verbose:
        print(f"[run_case_to_target_eta] Ordinary Newton stopped at eta={eta_reached_by_newton:.4f} "
              f"(target {target_eta}) -- falling back to arclength continuation...", flush=True)

    scratch_ckpt = tempfile.mktemp(suffix="_scratch_arclength.npz")
    best_eta, best_gap = eta_reached_by_newton, abs(eta_reached_by_newton - target_eta)
    total_steps, resumed = 0, False
    try:
        while total_steps < max_arclength_steps:
            r = run_case_arclength(overrides=overrides, eta_start=eta_reached_by_newton, ds=ds,
                                    ds_min=ds_min, ds_max=ds_max, n_steps=steps_per_batch,
                                    resume=resumed, checkpoint_path=scratch_ckpt, verbose=False)
            resumed = True
            total_steps += steps_per_batch
            eta_now = float(r["eta"][-1])
            gap = abs(eta_now - target_eta)
            if gap < best_gap:
                best_eta, best_gap = eta_now, gap
            if gap < eta_tol:
                if verbose:
                    print(f"[run_case_to_target_eta] Reached target via arclength "
                          f"(eta={eta_now:.4f}) after ~{total_steps} step(s).", flush=True)
                break
        else:
            if verbose:
                print(f"[run_case_to_target_eta] Exhausted ~{max_arclength_steps} arclength "
                      f"steps without reaching eta={target_eta} (closest: {best_eta:.4f}, "
                      f"gap={best_gap:.4f}). Returning closest approach.", flush=True)

        tmp_field_path = tempfile.mktemp(suffix="_scratch_fields.npz")
        raw_fields = save_field_snapshot(scratch_ckpt, tmp_field_path, overrides=overrides,
                                          verbose=False)
        d = np.load(scratch_ckpt)
        eta_hist_full = np.concatenate([result["eta"], d["eta_hist"][1:]])
        I_hist_full = np.concatenate([result["I_avg"], d["I_hist"][1:]])
        smax_hist_full = np.concatenate([result["s_max"], d["smax_hist"][1:]])

        # Post-process into the same membrane_y/s/I/CgO2 + interface_y/x
        # format run_case's save_fields_at produces (save_field_snapshot
        # only gives raw 2D fields, no membrane-line extraction or local
        # current density) -- makes this drop-in compatible with
        # run_parametric_study()'s expected per-value result format.
        P_local = dict(DEFAULT_PARAMS)
        if overrides:
            P_local.update(overrides)
        I0_arc = P_local["I0"] * np.exp(
            -(P_local["Ea_orr"] / P_local["R"]) * (1.0 / P_local["T_cell"] - 1.0 / P_local["T_ref_I0"]))
        rho_g_const_arc = P_local["p_out"] * (
            P_local["X_O2_in"] * P_local["M_O2"] + P_local["X_N2_in"] * P_local["M_N2"]
        ) / (P_local["R"] * P_local["T_cell"])
        C_O2_molar_arr = rho_g_const_arc * raw_fields["CgO2"] / P_local["M_O2"]
        I_local_arr = ((1.0 - raw_fields["s"]) * I0_arc * (C_O2_molar_arr / P_local["C_O2_ref"])
                        * np.exp(P_local["alpha_c"] * P_local["F"] * best_eta
                                 / (P_local["R"] * P_local["T_cell"])))
        mem = extract_line(raw_fields["coords"], {"s": raw_fields["s"], "CgO2": raw_fields["CgO2"],
                                                   "I": I_local_arr}, x_target=0.0)
        interface_y, interface_x = extract_interface(raw_fields["coords"], raw_fields["s"])
        fields = dict(membrane_y=mem["y"], membrane_s=mem["s"], membrane_I=mem["I"],
                      membrane_CgO2=mem["CgO2"], interface_y=interface_y, interface_x=interface_x)
    finally:
        for p in [scratch_ckpt, locals().get("tmp_field_path")]:
            if p:
                try:
                    os.remove(p)
                except Exception:
                    pass

    return dict(eta_hist=eta_hist_full, I_hist=I_hist_full, s_max_hist=smax_hist_full,
                target_reached=best_gap < eta_tol, best_eta=best_eta, gap=best_gap,
                fields=fields, method="arclength")


def run_parametric_study_to_target_eta(param_key, values, target_eta=0.55,
                                         eta_ramp_start=0.30, verbose=True, **base_overrides):
    """Like run_parametric_study(), but aims for target_eta (default 0.55,
    matching this paper's own parametric-study comparison point) using
    run_case_to_target_eta() per value -- ordinary Newton first, falling
    back to lightweight pseudo-arclength continuation only for whichever
    parameter values actually need it to get near target_eta. Much
    cheaper than the full paper-backing-data arclength infrastructure
    (no SVD tracking, no per-step snapshots) since this only needs ONE
    good field snapshot per parameter value, not the whole traced branch.

    Returns the same {value: {eta, I_avg, s_max, iters, membrane_y,
    membrane_s, membrane_I, membrane_CgO2, interface_y, interface_x}}
    format as run_parametric_study(), so it's a drop-in replacement for
    save_study_npz() and the existing plotting cells -- plus each
    per-value dict also carries `target_reached` (bool) and `gap` (float)
    so the plots/tables can flag which parameter values didn't quite
    make it to target_eta.
    """
    results = {}
    for v in values:
        if verbose:
            print(f"\n{'='*70}\n{param_key} = {v}\n{'='*70}", flush=True)
        overrides = dict(base_overrides)
        overrides[param_key] = v
        r = run_case_to_target_eta(overrides=overrides, target_eta=target_eta,
                                    eta_ramp_start=eta_ramp_start, verbose=verbose)
        fld = r["fields"] or {}
        results[v] = dict(
            eta=r["eta_hist"], I_avg=r["I_hist"], s_max=r["s_max_hist"],
            iters=np.ones_like(r["eta_hist"]),  # placeholder: not tracked uniformly across both
                                                 # paths (see method/gap instead), but MUST match
                                                 # eta_hist's length for existing plot cells'
                                                 # `iters < MAX_ITERS_CAP` filters to work correctly
            membrane_y=fld.get("membrane_y"), membrane_s=fld.get("membrane_s"),
            membrane_I=fld.get("membrane_I"), membrane_CgO2=fld.get("membrane_CgO2"),
            interface_y=fld.get("interface_y"), interface_x=fld.get("interface_x"),
            target_reached=np.array([r["target_reached"]]), gap=np.array([r["gap"]]),
            method=np.array([r["method"]]))
        if verbose:
            status = "reached" if r["target_reached"] else f"NOT reached (gap={r['gap']:.4f})"
            print(f"[{param_key}={v}] target_eta={target_eta} {status} via {r['method']}", flush=True)
    return results


## Mount Google Drive for result caching

Done here, in a plain cell, on purpose: `%%fenicsx` cells run in a separate subprocess without
the `google.colab` package available, so mounting has to happen here first.

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted')

DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f"Drive backup folder: {DRIVE_BACKUP}")


## Step 1: Extract full-field checkpoints at $\eta=1.24$, $\eta=1.27$, and $\eta=1.28$

Re-solves from $\eta=0.02$ up to each target using the SAME adaptive stepper, VI bound-
constrained solver, and physical-validity check (I >= -50 A/m^2, Cw/CgO2 in [0,1]) already
established for this parameter setting -- all three targets are within the range already
confirmed reachable (coarse mesh: up to $\eta\approx1.293$; refined mesh: up to
$\eta\approx1.284$), so this should be fast (targets sorted ascending and chained, so most of
the shared low-$\eta$ range is only solved once) and should not hit any obstruction on its own.
Real-time progress logged to `checkpoint_extraction_progress.log`.

In [ ]:
%%fenicsx -np 1 --time

import sys, os, time, shutil
sys.path.insert(0, "/content")
import numpy as np
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
from petsc4py import PETSc
import ufl
from pemfc_lib import (DEFAULT_PARAMS, Psat_atm, _smooth_outflow, _project_scalar_standalone,
                       run_case_arclength_auto)

OUT = "/content/pemfc_sensitivity_rh1p0_stability_results"
os.makedirs(OUT, exist_ok=True)
LOG_PATH = f"{OUT}/checkpoint_extraction_progress.log"


def alog(line):
    ts = time.strftime("%H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {line}\n")
        f.flush()
        os.fsync(f.fileno())
    print(line)


assert os.path.ismount('/content/drive'), (
    "Drive not mounted -- run the 'Mount Google Drive' cell above first.")
DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
n_restored = 0
for fname in os.listdir(DRIVE_BACKUP):
    local_fp = os.path.join(OUT, fname)
    if not os.path.exists(local_fp):
        shutil.copy2(os.path.join(DRIVE_BACKUP, fname), local_fp)
        n_restored += 1
if n_restored:
    alog(f"Restored {n_restored} file(s) from Drive backup into {OUT}")


def build_problem(overrides):
    P = dict(DEFAULT_PARAMS)
    P.update(overrides)
    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx_ = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx_)
        facet_markers.append(np.full_like(idx_, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order_f = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order_f], facet_markers[order_f])

    ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)
    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0
    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix
    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)
    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)
    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    Weff_w = lam_l + lam_g * CgH2O_local
    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
               - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
    F_total = F_pres + F_water + F_oxy

    snes_petsc_options = {
        "snes_type": "newtonls", "snes_rtol": 1e-7, "snes_atol": 1e-9,
        "snes_max_it": 50, "snes_error_if_not_converged": False,
        "ksp_type": "preonly", "pc_type": "lu",
    }
    problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                petsc_options_prefix=f"pemfc_hesc_{id(overrides)}_",
                                form_compiler_options={"quadrature_degree": 4})

    problem.solver.setType("vinewtonrsls")
    Cw_dofs_global = W.sub(1).collapse()[1]
    CgO2_dofs_global = W.sub(2).collapse()[1]
    n_total = len(w.x.array)
    lb_arr = np.full(n_total, -1e20)
    ub_arr = np.full(n_total, 1e20)
    lb_arr[Cw_dofs_global] = 0.0
    ub_arr[Cw_dofs_global] = 1.0
    lb_arr[CgO2_dofs_global] = 0.0
    ub_arr[CgO2_dofs_global] = 1.0
    lb = w.x.petsc_vec.copy()
    ub = w.x.petsc_vec.copy()
    lb.array[:] = lb_arr
    ub.array[:] = ub_arr
    lb.assemble()
    ub.assemble()
    problem.solver.setVariableBounds(lb, ub)

    def measure():
        Jloc_arr = _project_scalar_standalone(I_local, V, dx)
        I_now = float(np.mean(Jloc_arr[membrane_dofs_V]))
        s_arr = _project_scalar_standalone(s, V, dx)
        s_now = float(s_arr.max())
        return I_now, s_now

    def check_physical(I_now, tol=0.01, I_tol=-50.0):
        Cw_arr = w.sub(1).collapse().x.array
        CgO2_arr = w.sub(2).collapse().x.array
        Cw_min, Cw_max = float(Cw_arr.min()), float(Cw_arr.max())
        CgO2_min, CgO2_max = float(CgO2_arr.min()), float(CgO2_arr.max())
        conc_ok = (Cw_min > -tol and Cw_max < 1 + tol and
                   CgO2_min > -tol and CgO2_max < 1 + tol)
        I_ok = I_now > I_tol
        return conc_ok and I_ok, dict(Cw_min=Cw_min, Cw_max=Cw_max, CgO2_min=CgO2_min,
                                        CgO2_max=CgO2_max, I=I_now)

    return w, eta_const, problem, measure, check_physical, V, s, dx


def solve_to_eta(overrides, eta_target, ds_init=0.02, ds_min=5e-4, log_fn=alog,
                  s_max_flooding_thresh=0.05, ds_flooding_cap=0.005, resume_from=None,
                  s_max_jump_thresh=0.1, I_jump_thresh=3000.0,
                  big_jump_multipliers=(1.0, 2.0, 3.0)):
    """Same adaptive stepper as the high-eta exploration notebook
    (VI-constrained + physical check + escalating shrink-on-repeated-
    failure), just returning the final full-field state at eta_target
    instead of a whole curve.

    Also PROACTIVELY caps the step size whenever s_max rises above
    s_max_flooding_thresh, rather than only reacting after a step
    actually fails -- flooding transitions are both where this stepper
    has repeatedly run into numerical difficulty in practice AND the
    operationally important regime for a real cell (mass-transport-
    limited operation), so resolving that regime more finely by default
    is a genuine improvement, not just a numerical safety margin.

    resume_from=(w_array, eta_start), if given, starts from that
    already-converged state instead of re-solving from eta=0.02 --
    confirmed the hard way that extracting two nearby targets
    independently (each restarting from scratch) redoes the ENTIRE
    shared range twice, wasting most of the wall-clock time when the
    targets are close together (e.g. eta=1.24 and eta=1.27 share the
    whole eta=0.02-1.24 range)."""
    w, eta_const, problem, measure, check_physical, V, s_form, dx = build_problem(overrides)
    if resume_from is not None:
        w_init_arr, eta = resume_from
        assert len(w_init_arr) == len(w.x.array), "resume_from state doesn't match this mesh."
        w.x.array[:] = w_init_arr
        w.x.scatter_forward()
        eta_const.value = eta
        problem.solve()
        if problem.solver.getConvergedReason() <= 0:
            # Update 10: a bare AssertionError here used to crash the whole Stage-1 cell
            # uncaught -- confirmed the hard way on a real tau1p5 run: eta1.27_postfold's own
            # checkpoint came from arclength continuation and landed at eta=0.3174 mid-fold
            # (its own log showed "moderate flooding, eta decreasing" right before stopping
            # there) -- exactly the kind of delicate, near-singular point ordinary fixed-eta
            # Newton cannot be trusted to re-converge, which is the whole reason arclength was
            # needed to reach it in the first place. Raising a RuntimeError with the same
            # .w_last/.eta_last attachment used elsewhere (Update 7) instead of a bare assert
            # routes this into extract_with_arclength_fallback's EXISTING machinery -- it warm-
            # starts arclength directly from this state/eta, and if that bootstrap also can't
            # take a single step, Update 9's ramped-bootstrap fallback (which re-derives the
            # state via a fresh warm-up ramp from eta=0.02, rather than reusing this exact array)
            # already handles it -- no new fallback logic needed here.
            resume_err = RuntimeError(
                f"resume_from state (eta={eta:.4f}) did not re-converge cleanly under ordinary "
                f"fixed-eta Newton -- likely a delicate, near-fold point inherited from a "
                f"previous label's arclength-derived checkpoint.")
            resume_err.w_last = w_init_arr
            resume_err.eta_last = eta
            raise resume_err
        log_fn(f"  resumed from eta={eta:.4f} (skipping the already-solved range below it)")
    else:
        eta = 0.02
        eta_const.value = eta
        problem.solve()
        assert problem.solver.getConvergedReason() > 0, f"initial solve failed at eta={eta}"
    w_last_good = w.x.array.copy()
    I_last_accepted, s_last_accepted = measure()
    ds = ds_init
    consecutive_failures = 0
    consecutive_I_jump_rejections = 0

    while eta < eta_target - 1e-9:
        eta_try = min(eta + ds, eta_target)
        eta_const.value = eta_try
        problem.solve()
        newton_ok = problem.solver.getConvergedReason() > 0
        if newton_ok:
            I_now, s_now = measure()
            phys_ok, phys_info = check_physical(I_now)
        else:
            I_now, s_now, phys_ok, phys_info = None, None, False, None

        s_jump = (s_now - s_last_accepted) if newton_ok and phys_ok else None
        I_jump = (abs(I_now - I_last_accepted)) if newton_ok and phys_ok else None
        s_jump_rejected = newton_ok and phys_ok and s_jump is not None and s_jump > s_max_jump_thresh
        I_jump_rejected = newton_ok and phys_ok and I_jump is not None and I_jump > I_jump_thresh
        jump_rejected = s_jump_rejected or I_jump_rejected
        # If Newton converged and stayed physically valid, and the ONLY
        # problem is the jump-size policy, and ds is already about to
        # bottom out -- just accept it rather than shrinking further or
        # giving up; Newton was never the real obstacle in that case.
        at_floor = (ds / 2.0) < ds_min
        I_jump_twice_in_a_row = I_jump_rejected and consecutive_I_jump_rejections >= 2
        force_accept = (jump_rejected and at_floor) or I_jump_twice_in_a_row

        if newton_ok and phys_ok and (not jump_rejected or force_accept):
            w_last_good = w.x.array.copy()
            eta = eta_try
            s_last_accepted = s_now
            I_last_accepted = I_now
            consecutive_failures = 0
            consecutive_I_jump_rejections = 0
            growth_cap = ds_flooding_cap if s_now > s_max_flooding_thresh else ds_init
            ds = min(ds * 1.5, growth_cap)
            flood_note = " (flooding-capped)" if s_now > s_max_flooding_thresh else ""
            if force_accept:
                floor_bits = []
                if s_jump_rejected:
                    floor_bits.append(f"s_max jump of {s_jump:.4f}")
                if I_jump_rejected:
                    floor_bits.append(f"I jump of {I_jump/1e4:.4f} A/cm^2")
                if I_jump_twice_in_a_row and not at_floor:
                    accept_note = (f" [accepted despite {' and '.join(floor_bits)} -- "
                                    f"I_jump_thresh rejected 2 times in a row at this point]")
                else:
                    accept_note = (f" [accepted despite {' and '.join(floor_bits)} -- ds was "
                                    f"already at its floor]")
            else:
                accept_note = ""
            log_fn(f"  eta={eta:.4f}: I={I_now/1e4:.4f} A/cm^2, s_max={s_now:.4f}, "
                    f"ds now {ds:.4f}{flood_note}{accept_note}")
        else:
            w.x.array[:] = w_last_good
            w.x.scatter_forward()
            consecutive_failures += 1
            consecutive_I_jump_rejections = (consecutive_I_jump_rejections + 1
                                              if I_jump_rejected else 0)
            shrink_factor = 2.0 if consecutive_failures == 1 else 8.0
            ds /= shrink_factor
            if not newton_ok:
                reason = "Newton FAILED"
            elif not phys_ok:
                reason = f"REJECTED on physical-bounds check ({phys_info})"
            else:
                bits = []
                if s_jump_rejected:
                    bits.append(f"s_max jumped by {s_jump:.4f} (from {s_last_accepted:.4f} "
                                f"to {s_now:.4f}), exceeding s_max_jump_thresh={s_max_jump_thresh}")
                if I_jump_rejected:
                    bits.append(f"I jumped by {I_jump/1e4:.4f} A/cm^2 (from "
                                f"{I_last_accepted/1e4:.4f} to {I_now/1e4:.4f}), exceeding "
                                f"I_jump_thresh={I_jump_thresh/1e4:.4f} A/cm^2")
                reason = "REJECTED: " + "; ".join(bits)
            log_fn(f"  eta={eta_try:.4f}: {reason} -- shrinking ds by {shrink_factor:.0f}x to "
                    f"{ds:.5f}, retrying from eta={eta:.4f}")
            if ds < ds_min:
                # Genuine Newton/physical failure persisting even at the
                # smallest step size -- before giving up, try one or more
                # BIG JUMPS (bypassing the jump-size policy entirely, same
                # as the high-eta exploration notebook) in case a
                # continuously-connected path just doesn't exist through
                # small steps here, even though a valid point exists
                # further out.
                log_fn(f"  ds below {ds_min} -- trying BIG JUMP fallback(s) before giving up")
                jumped = False
                for jump_mult in big_jump_multipliers:
                    eta_try_big = min(eta + ds_init * jump_mult, eta_target)
                    if eta_try_big <= eta + 1e-9:
                        continue
                    eta_const.value = eta_try_big
                    problem.solve()
                    newton_ok_big = problem.solver.getConvergedReason() > 0
                    if newton_ok_big:
                        I_big, s_big = measure()
                        phys_ok_big, phys_info_big = check_physical(I_big)
                    else:
                        I_big, s_big, phys_ok_big, phys_info_big = None, None, False, None
                    if newton_ok_big and phys_ok_big:
                        log_fn(f"  BIG JUMP to eta={eta_try_big:.4f} (x{jump_mult:.1f} ds_init) "
                                f"SUCCEEDED: I={I_big/1e4:.4f} A/cm^2, s_max={s_big:.4f} -- "
                                f"resuming normal stepping from here")
                        w_last_good = w.x.array.copy()
                        eta = eta_try_big
                        s_last_accepted = s_big
                        ds = ds_init
                        consecutive_failures = 0
                        jumped = True
                        break
                    else:
                        w.x.array[:] = w_last_good
                        w.x.scatter_forward()
                        reason_big = ("Newton FAILED" if not newton_ok_big
                                      else f"REJECTED ({phys_info_big})")
                        log_fn(f"  BIG JUMP to eta={eta_try_big:.4f} (x{jump_mult:.1f} ds_init) "
                                f"also failed ({reason_big})")
                if jumped:
                    continue
                stall_err = RuntimeError(
                    f"Could not reach eta_target={eta_target} -- stuck near "
                    f"eta={eta:.4f} even after BIG JUMP fallbacks. This point "
                    f"may itself be right at a genuine obstruction; try a "
                    f"slightly lower eta_target.")
                # Update 7: attach the actual field state reached (w_last_good, eta) to the
                # exception -- Update 5's arclength fallback had no way to recover this on a
                # plain RuntimeError and fell back to carry_state (or eta=0.02), silently
                # discarding all the fixed-eta progress already made and re-tracing it from
                # scratch via arclength. Confirmed the hard way from a real tau1p3 log: ~2h40m
                # of fixed-eta progress to eta=0.3578 thrown away, arclength then taking another
                # ~54min to only get back to eta=0.2754 -- still short of where it already was.
                stall_err.w_last = w_last_good
                stall_err.eta_last = eta
                raise stall_err
    return w_last_good, eta


# ============ EDIT THIS ============
OVERRIDES = dict(tau_brug=1.1, Lx=0.2e-3, RH_in=1.0)
TARGETS = {
    "eta1.24_prefold": 1.24,
    "eta1.27_postfold": 1.27,
    "eta1.28_postfold": 1.28,
}
FORCE_RECOMPUTE = False
# =====================================

# Sorted ascending so each target can reuse the PREVIOUS (smaller)
# target's already-converged state as its own starting point, instead of
# every target independently re-solving the same shared low-eta range
# from scratch.
sorted_targets = sorted(TARGETS.items(), key=lambda kv: kv[1])
carry_state = None  # (w_array, eta) from the most recently computed point, for chaining

# Fold fallback: pseudo-arclength continuation, not target-backoff.
#
# Update 5 finding: reviewing REAL tau1p3/tau1p5 logs (both already running the ds_min=1e-6 fix
# from Update 3) showed that fix was necessary but not sufficient -- it pushed the reachable
# range a little further (tau1p5: eta~0.328 -> eta~0.332) but then hit a WALL that no amount of
# further step-halving gets past: s_max keeps climbing while eta itself stays essentially frozen
# for dozens of steps, Newton failing outright even at ds well below 1e-6. That is the textbook
# signature of a genuine fold (a vertical tangent, deta/ds -> 0), not a resolution problem: at a
# true fold eta is not a valid local coordinate for the solution branch, so solve_to_eta's
# fixed-eta Newton CANNOT converge there no matter how finely eta is bisected -- confirmed
# independently by this project's own Jacobian-SVD diagnostic (PEMFC_Tau1.1_Lx0.2mm_
# ArclengthContinuation.ipynb / pemfc_lib.run_case_arclength's docstring): the smallest singular
# value collapses from ~1.6e-5 to ~2.9e-10 crossing exactly this kind of point. Backing off the
# *target* eta by a few mV (the old MAX_BACKOFF_STEPS/BACKOFF_STEP mechanism) cannot help either
# -- confirmed the hard way: tau1p3/tau1p5 hit the identical eta wall on every one of several
# backoff attempts, each a wasted ~1-2.5 hour full restart from eta=0.02.
#
# The project already has a validated tool for exactly this situation: pseudo-arclength
# continuation (Keller's bordering algorithm, `run_case_arclength_auto` in pemfc_lib.py), which
# treats eta as an unknown alongside the field state (instead of a fixed target) so it can trace
# the branch smoothly THROUGH a fold, with eta itself moving non-monotonically if that's what the
# branch does. It's the same machinery already used to cross this model's own well-known primary
# high-eta fold near eta=1.27. On an ordinary-Newton RuntimeError, this cell now falls back to it
# once, in growing chunks (so an easy case that only needs a few dozen steps doesn't pay for a
# 400-step budget), stopping as soon as it's within ARC_ETA_TOL of the target or the chunk budget
# is exhausted -- whichever point it reaches is used as this label's checkpoint, exactly like an
# accepted backoff point was before (Update 1's own philosophy: the point actually reached is
# itself useful sensitivity information, not just a fallback value).
ARC_ETA_TOL = 0.01     # V -- "close enough" to the nominal target to stop early
ARC_CHUNK = 100        # steps_per call to run_case_arclength_auto, growing by this much
ARC_MAX_BUDGET = 400   # same total budget PEMFC_Tau1.1_Lx0.2mm_ArclengthContinuation.ipynb uses


def extract_with_arclength_fallback(label, eta_target, carry_state):
    arc_ckpt = f"{OUT}/{label}_arclength_checkpoint.npz"
    arc_drive_backup = f"{DRIVE_BACKUP}/{label}_arclength_checkpoint.npz"
    if os.path.exists(arc_drive_backup) and not os.path.exists(arc_ckpt):
        shutil.copy2(arc_drive_backup, arc_ckpt)
        alog(f"{label}: restored an in-progress arclength checkpoint from Drive backup")

    def run_arclength_budget_loop(eta_start, w_stalled):
        budget = ARC_CHUNK
        while True:
            n_accepted = run_case_arclength_auto(
                overrides=OVERRIDES, eta_start=eta_start, init_w_array=w_stalled,
                ds=0.01, ds_min=1e-6, ds_max=0.03,
                checkpoint_path=arc_ckpt, snapshot_dir=None,
                total_steps_budget=budget, steps_per_attempt=25,
                svd_diagnostic_every=None, svd_out_path=None,
                diagnostics_dir=None, stats_out_path=None,
                max_recovery_rounds=2, verbose=True,
            )
            if not os.path.exists(arc_ckpt):
                # Update 9: warm-starting the arclength bootstrap directly from the exact point
                # fixed-eta Newton just got stuck at (Update 7) can itself fail to take even a
                # single step -- bootstrap solve 2 is just an ordinary fixed-eta Newton step
                # (eta_start + ds), the same kind of step that was ALREADY failing at this exact
                # location. Confirmed the hard way on a real tau1p5 run: every attempt across the
                # whole recovery ladder failed before ever writing a checkpoint, and this used to
                # propagate as an uncaught crash that killed the rest of Stage 1 (and, in a
                # cascade, every downstream Part A-D cell that depends on its output). Fall back
                # to the slower but far more battle-tested ramped bootstrap (starting from
                # carry_state/eta=0.02, letting run_case_arclength's own warm-up ramp handle the
                # climb, exactly as before Update 7 existed) instead of giving up on this label.
                if w_stalled is not None:
                    alog(f"{label}: warm-started arclength bootstrap produced ZERO accepted "
                         f"steps (failed to take even the first step from eta={eta_start:.4f}) "
                         f"-- retrying with the slower ramped bootstrap instead of the warm "
                         f"start")
                    eta_start = carry_state[1] if carry_state is not None else 0.02
                    w_stalled = None
                    continue
                else:
                    alog(f"{label}: arclength produced ZERO accepted steps even with the ramped "
                         f"bootstrap starting from eta={eta_start:.4f} -- this label genuinely "
                         f"cannot be extracted right now; raising so this is visible rather than "
                         f"silently skipped.")
                    raise RuntimeError(
                        f"{label}: arclength could not take even one accepted step "
                        f"(tried both a warm start and the ramped bootstrap, both from "
                        f"eta={eta_start:.4f}).")
            shutil.copy2(arc_ckpt, arc_drive_backup)
            d_arc = np.load(arc_ckpt)
            eta_hist = d_arc["eta_hist"]
            eta_now = float(eta_hist[-1])
            gap = abs(eta_now - eta_target)
            alog(f"{label}: arclength continuation at {n_accepted} accepted step(s) -- eta now "
                 f"{eta_now:.4f} (gap to target {gap:.4f}), full range so far "
                 f"[{eta_hist.min():.4f}, {eta_hist.max():.4f}]")
            if gap < ARC_ETA_TOL:
                alog(f"{label}: within {ARC_ETA_TOL} V of target eta={eta_target} -- stopping "
                     f"here and using this as the checkpoint.")
                break
            if n_accepted >= budget and budget < ARC_MAX_BUDGET:
                budget = min(budget + ARC_CHUNK, ARC_MAX_BUDGET)
                alog(f"{label}: not close enough yet and still making progress (budget not "
                     f"exhausted by a stall) -- raising budget to {budget} and continuing "
                     f"(resumes automatically from its own checkpoint).")
                continue
            alog(f"{label}: arclength continuation stopped at eta={eta_now:.4f} (target "
                 f"{eta_target}, gap {gap:.4f}) -- either genuinely stuck (recovery ladder "
                 f"exhausted) or at ARC_MAX_BUDGET={ARC_MAX_BUDGET}. Using this closest-reached "
                 f"point as the checkpoint; the gap itself is useful sensitivity information "
                 f"(how far this parameter value's fold/branch shape differs from baseline).")
            break
        return d_arc["w_last"], eta_now

    # Update 8: if this label already has arclength progress checkpointed from a PRIOR run or
    # session, trying solve_to_eta first would just re-climb the ENTIRE fixed-eta range from
    # eta=0.02 only to hit the exact same wall again -- confirmed the hard way on a real tau1p3
    # restart: 2h20m+ re-spent getting back to eta=0.3578, purely to re-discover a fallback this
    # label was already known to need, without benefiting the arclength phase at all (it resumes
    # from its own Drive checkpoint regardless of anything solve_to_eta does). Skip straight to
    # arclength whenever this label already has checkpointed arclength progress.
    if os.path.exists(arc_ckpt):
        alog(f"{label}: an arclength checkpoint already exists (from a prior run/session) -- "
             f"skipping the fixed-eta attempt (this label is already known to need arclength) "
             f"and resuming arclength continuation directly")
        eta_start = carry_state[1] if carry_state is not None else 0.02
        return run_arclength_budget_loop(eta_start, None)

    try:
        w_final, eta_reached = solve_to_eta(OVERRIDES, eta_target, resume_from=carry_state,
                                             ds_min=1e-6)
        return w_final, eta_reached
    except RuntimeError as e:
        alog(f"{label}: ordinary (fixed-eta) Newton continuation stalled ({e})")
        alog(f"{label}: falling back to pseudo-arclength continuation -- see this cell's own "
             f"comment above for why (this is very likely a genuine fold, not a step-size "
             f"problem, so target-backoff would not help here either).")
        # Update 7: warm-start arclength from the exact field state fixed-eta Newton had
        # already reached (attached to the exception above) instead of discarding it --
        # pemfc_lib.run_case_arclength already has an init_w_array parameter built for exactly
        # this case ("warm-start from an existing solution ... skips the warmup ramp entirely"),
        # it just wasn't threaded through run_case_arclength_auto or used here yet. Falls back
        # to the old carry_state/0.02 behavior only if the exception doesn't carry a state (e.g.
        # a RuntimeError raised somewhere else in solve_to_eta).
        w_stalled = getattr(e, "w_last", None)
        eta_stalled = getattr(e, "eta_last", None)
        if w_stalled is not None and eta_stalled is not None:
            eta_start = eta_stalled
            alog(f"{label}: warm-starting arclength from the fixed-eta state already reached "
                 f"(eta={eta_start:.4f}) instead of retracing it from scratch")
        else:
            eta_start = carry_state[1] if carry_state is not None else 0.02
            w_stalled = None
        return run_arclength_budget_loop(eta_start, w_stalled)


for label, eta_target in sorted_targets:
    out_path = f"{OUT}/{label}_checkpoint.npz"
    if os.path.exists(out_path) and not FORCE_RECOMPUTE:
        d_existing = np.load(out_path)
        eta_here = float(d_existing['eta_last'])
        alog(f"{label}: SKIPPED (already extracted, eta={eta_here:.4f}) -- "
             f"set FORCE_RECOMPUTE=True to redo it")
        carry_state = (d_existing["w_last"], eta_here)  # still usable to chain the NEXT target
        continue
    alog(f"=== Extracting {label} (target eta={eta_target}) ===")
    # Update 12: extract_with_arclength_fallback can genuinely exhaust every option (fixed-eta
    # AND both the warm-start and ramped-bootstrap arclength attempts) and raise -- confirmed on
    # a real tau1p3 run where eta1.28_postfold stalled at eta=0.3578 under fixed-eta, then
    # arclength failed to take even one accepted step from either the warm start (eta=0.3578) or
    # the ramped bootstrap (from carry_state's own eta). Previously this uncaught RuntimeError
    # killed the WHOLE Stage-1 cell, even though earlier labels in this same loop
    # (eta1.24_prefold, eta1.27_postfold) had already been extracted and saved to disk --
    # wiping out Part A/B/C/D's ability to analyze those perfectly good checkpoints too, since
    # they used to assert every CHECKPOINTS entry exists before doing anything (see this
    # notebook's Update 12 fixes there). This is a genuinely new, expected outcome for a
    # parameter value where this specific label's target sits right on top of an obstruction
    # current step/tolerance settings can't get past from either direction -- not a bug -- so
    # it's now caught, logged, and skipped like every other "this one point isn't available"
    # case in this project, instead of crashing everything downstream of it too.
    try:
        w_final, eta_reached = extract_with_arclength_fallback(label, eta_target, carry_state)
    except RuntimeError as e:
        alog(f"{label}: COULD NOT be extracted at all (neither fixed-eta nor arclength could "
             f"make any progress) -- {e}. Skipping this label; carry_state is left unchanged so "
             f"any LATER label in this list can still chain from the last point that DID "
             f"succeed. No {label}_checkpoint.npz will exist -- downstream cells that need this "
             f"specific label already handle a missing checkpoint by skipping or falling back "
             f"(Update 12), not by crashing.")
        continue
    np.savez(out_path, w_last=w_final, eta_last=eta_reached, eta_target_original=eta_target)
    shutil.copy2(out_path, os.path.join(DRIVE_BACKUP, f"{label}_checkpoint.npz"))
    alog(f"{label}: saved checkpoint at eta={eta_reached:.4f} to {out_path}"
         + (f" (requested eta_target={eta_target})" if eta_reached < eta_target - 1e-9 else ""))
    carry_state = (w_final, eta_reached)


## Part A: Linear stability (generalized eigenvalue problem)

Same approximate-mass-matrix generalized eigenvalue approach validated for the original
base-case fold. Tests all three checkpoints: a genuine saddle-node fold predicts
`eta1.24_prefold` and `eta1.27_postfold` stable (0 unstable eigenvalues each -- confirmed for
both) and `eta1.28_postfold` unstable (confirmed: currently **3** unstable eigenvalues).

> **Note on this count changing (was 7, now 3).** The `eta1.28_postfold` checkpoint used to be
> `eta1.29_deepflood`, extracted with an EARLIER, not-yet-synced version of the solver (missing
> the $I$-jump rejection check that `HighEta_Exploration.ipynb` already had). After syncing
> `solve_to_eta` to match exactly and re-extracting at the (now solver-reachable) $\eta=1.28$,
> the eigenvalue count dropped to 3. This is expected, not a regression: the earlier checkpoint
> was reached via a less-constrained path that likely let small unphysical excursions through,
> which the eigenvalue solver would have picked up as spurious extra unstable directions. 3 is
> the number to trust going forward; if this notebook is ever re-run against a *different*
> checkpoint (e.g. after further solver changes), expect this number to be re-confirmed rather
> than assumed.

In [ ]:
%%fenicsx -np 1 --time

import sys, os
sys.path.insert(0, "/content")
import numpy as np
import scipy.linalg
import scipy.sparse as sp
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
from petsc4py import PETSc
import ufl
from pemfc_lib import DEFAULT_PARAMS, Psat_atm, _smooth_outflow, _project_scalar_standalone

OUT = "/content/pemfc_sensitivity_rh1p0_stability_results"
os.makedirs(OUT, exist_ok=True)
EIG_LOG = f"{OUT}/eigenvalue_progress.log"

import shutil
assert os.path.ismount('/content/drive'), (
    "Drive not mounted -- run the 'Mount Google Drive' cell above first.")
DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
n_restored = 0
for fname in os.listdir(DRIVE_BACKUP):
    local_fp = os.path.join(OUT, fname)
    if not os.path.exists(local_fp):
        shutil.copy2(os.path.join(DRIVE_BACKUP, fname), local_fp)
        n_restored += 1
if n_restored:
    print(f"Restored {n_restored} file(s) from Drive backup into {OUT}")


def eig_log(line):
    import time as _time
    ts = _time.strftime("%H:%M:%S")
    with open(EIG_LOG, "a") as f:
        f.write(f"[{ts}] {line}\n")
        f.flush()
        os.fsync(f.fileno())

# ============ EDIT THIS ============
CHECKPOINTS = {
    "eta1.24_prefold": f"{OUT}/eta1.24_prefold_checkpoint.npz",
    "eta1.27_postfold": f"{OUT}/eta1.27_postfold_checkpoint.npz",
    "eta1.28_postfold": f"{OUT}/eta1.28_postfold_checkpoint.npz",
}
OVERRIDES = dict(tau_brug=1.1, Lx=0.2e-3, RH_in=1.0)  # MUST match the checkpoints' own parameters
# ====================================

import numpy as _np_check
# Update 12: this used to be a blanket assert requiring EVERY CHECKPOINTS entry to exist before
# this cell did anything -- so if Stage 1 (the extraction cell) could not extract even one label
# at all (a genuine "stuck at an obstruction from both directions" outcome, not a bug -- see
# that cell's own Update 12 note), this cell refused to analyze the OTHER labels that had
# already been extracted successfully. Now it just logs which are missing and continues; the
# per-label loop below already skips any that aren't there.
_missing_checkpoints = [_lbl for _lbl, _fp in CHECKPOINTS.items() if not os.path.exists(_fp)]
if _missing_checkpoints:
    print(f"NOTE: {len(_missing_checkpoints)} checkpoint(s) not found -- will be SKIPPED below, "
          f"not treated as an error: {_missing_checkpoints}. This is expected when Stage 1's "
          f"extraction loop could not reach that particular label at all for this parameter "
          f"value. If you expect all of these to exist, re-run the extraction cell first.")

P = dict(DEFAULT_PARAMS)
if OVERRIDES:
    P.update(OVERRIDES)
F, R = P["F"], P["R"]
K_perm, eps_p = P["K_perm"], P["eps_p"]
p_in, p_out = P["p_in"], P["p_out"]
mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
rho_l = P["rho_l"]
M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
mu_l = P["mu_l"]
T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
RH_in = P["RH_in"]
theta_c_deg = P["theta_c_deg"]
theta_c = theta_c_deg * np.pi / 180.0
sigma_st = P["sigma_st"]
X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
s_smooth_eps = P["s_smooth_eps"]
single_phase = P["single_phase"]
nx, ny = P["nx"], P["ny"]

P_sat = Psat_atm(T_cell - 273.15) * 101325.0
Ly = w_ch + w_rb
y_inlet_end = 0.5 * w_ch
y_rib_end = 0.5 * w_ch + w_rb

comm = MPI.COMM_WORLD
domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
tdim = domain.topology.dim
fdim = tdim - 1

def on_membrane(x): return np.isclose(x[0], 0.0)
def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
facet_indices, facet_markers = [], []
for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                      (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
    idx_ = mesh.locate_entities_boundary(domain, fdim, locator)
    facet_indices.append(idx_)
    facet_markers.append(np.full_like(idx_, tag))
facet_indices = np.concatenate(facet_indices)
facet_markers = np.concatenate(facet_markers)
order_f = np.argsort(facet_indices)
facet_tags = mesh.meshtags(domain, fdim, facet_indices[order_f], facet_markers[order_f])

ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
dx = ufl.Measure("dx", domain=domain)
nvec = ufl.FacetNormal(domain)

V = fem.functionspace(domain, ("Lagrange", 1))
membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))
P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
ME = basix.ufl.mixed_element([P1e, P1e, P1e])
W = fem.functionspace(domain, ME)

w = fem.Function(W, name="w")
v_te = ufl.TestFunction(W)
p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
v_p, v_cw, v_cgo2 = ufl.split(v_te)
P_scale = 1000.0
p_sol = p_out + P_scale * p_hat_sol

X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
dry_scale = 1.0 - X_H2O_in
X_O2_eff = X_O2_in * dry_scale
X_N2_eff = X_N2_in * dry_scale
M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
Y_O2_in = X_O2_eff * M_O2 / M_mix_in
Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
rho_g_const = p_out * M_mix_in / (R * T_cell)
Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
Cl_sat = 1.0

inlet_facets = facet_tags.find(TAG_INLET)
outlet_facets = facet_tags.find(TAG_OUTLET)
dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
bcs = [
    fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
    fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
    fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
    fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
]

def s_expr(Cw):
    if single_phase:
        return 0.0 * Cw
    raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
    s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
    s_smooth = ufl.max_value(s_smooth, 0.0)
    return ufl.min_value(s_smooth, s_max_cap)

def min_smooth(a, b, eps=s_smooth_eps):
    return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

hydrophobic = theta_c_deg >= 90.0
s = s_expr(Cw_sol)
krl = s ** n_corey
krg = (1.0 - s) ** n_corey
lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
lam_g = 1.0 - lam_l
rho_mix = rho_g_const * (1.0 - s) + rho_l * s
nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
mu_mix = nu_mix * rho_mix
kappa = rho_mix * K_perm / mu_mix
if hydrophobic:
    dJds = 1.417 - 4.240 * s + 3.789 * s**2
else:
    u_hp = 1.0 - s
    dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)
Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
      * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
grads = ufl.grad(s)
u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
CgH2O_local = min_smooth(Cw_sol, Cg_sat)
Weff_w = lam_l + lam_g * CgH2O_local
Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
Jl_vec = Dc * grads
F_conv_w = Weff_w * rho_mix * u_darcy
W_O2 = rho_mix * lam_g * u_darcy

eta_const = fem.Constant(domain, default_scalar_type(0.0))
C_O2_molar = rho_g_const * CgO2_sol / M_O2
I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
N_O2_expr = (M_O2 / (4.0 * F)) * I_local
N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
          - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
           - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
           + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
           - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
         + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
         + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
         + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
         + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
F_total = F_pres + F_water + F_oxy

fc_opts = {"quadrature_degree": 4}
J_ufl = ufl.derivative(F_total, w)
J_form = fem.form(J_ufl, form_compiler_options=fc_opts)

snes_petsc_options = {
    # rtol/atol here MUST NOT be tighter than the rest of this project
    # (1e-7/1e-9) -- see Part A's own note above for why: this exact
    # combination (1e-8/1e-10) was confirmed to make the checkpoint
    # re-convergence itself fail right at this near-fold point.
    "snes_type": "newtonls", "snes_rtol": 1e-7, "snes_atol": 1e-9,
    "snes_max_it": 50, "snes_error_if_not_converged": False,
    "ksp_type": "preonly", "pc_type": "lu",
}
problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                            petsc_options_prefix="pemfc_eig_",
                            form_compiler_options=fc_opts)

# VI bound-constrained solver -- confirmed the hard way that RE-solving
# at eta=1.27 without this (plain newtonls) fails to re-converge even
# though the checkpoint was originally extracted successfully WITH it.
# The extreme-eta points this notebook targets need this to reliably
# stay solvable, not just to avoid unphysical excursions.
problem.solver.setType("vinewtonrsls")
Cw_dofs_global = W.sub(1).collapse()[1]
CgO2_dofs_global = W.sub(2).collapse()[1]
n_total = len(w.x.array)
lb_arr = np.full(n_total, -1e20)
ub_arr = np.full(n_total, 1e20)
lb_arr[Cw_dofs_global] = 0.0
ub_arr[Cw_dofs_global] = 1.0
lb_arr[CgO2_dofs_global] = 0.0
ub_arr[CgO2_dofs_global] = 1.0
lb = w.x.petsc_vec.copy()
ub = w.x.petsc_vec.copy()
lb.array[:] = lb_arr
ub.array[:] = ub_arr
lb.assemble()
ub.assemble()
problem.solver.setVariableBounds(lb, ub)

w.sub(0).interpolate(lambda x: np.full(x.shape[1], (p_in - p_out) / P_scale))
w.sub(1).interpolate(lambda x: np.full(x.shape[1], Y_H2O_in))
w.sub(2).interpolate(lambda x: np.full(x.shape[1], Y_O2_in))
w.x.scatter_forward()

# Mass matrix (diagonal, approximate): eps_p on Cw/CgO2 dofs, ZERO on
# pressure dofs (quasi-steady -- no accumulation there).
p_dofs = W.sub(0).collapse()[1]
cw_dofs = W.sub(1).collapse()[1]
cgo2_dofs = W.sub(2).collapse()[1]
n_dofs = len(w.x.array)
M_diag = np.zeros(n_dofs)
M_diag[cw_dofs] = eps_p
M_diag[cgo2_dofs] = eps_p
# p_dofs left at 0

# ============ EDIT THIS ============
FORCE_RECOMPUTE_EIG = False  # set True to ignore existing eigenvalue results and redo everything
# =====================================

eig_out_path = f"{OUT}/eigenvalue_check.npz"
results = {}
if os.path.exists(eig_out_path) and not FORCE_RECOMPUTE_EIG:
    # Reconstruct results{} from any already-saved checkpoints' data --
    # dense eigenvalue decomposition is by far the slowest part of this
    # whole notebook (tens of minutes per point), so re-running this
    # cell just to test a Part B tweak should not have to redo it.
    d_existing_eig = np.load(eig_out_path, allow_pickle=True)
    for full_key in d_existing_eig.files:
        ckpt_label_existing, field = full_key.split("__", 1)
        results.setdefault(ckpt_label_existing, {})[field] = d_existing_eig[full_key]
    if results:
        eig_log(f"Loaded {len(results)} already-computed checkpoint(s) from {eig_out_path}: "
                f"{list(results.keys())}")

for ckpt_label, ckpt_path in CHECKPOINTS.items():
    if not os.path.exists(ckpt_path):
        eig_log(f"{ckpt_label}: SKIPPED -- checkpoint file not found (Stage 1 could not extract "
                f"this label at all for this parameter value; see Update 12).")
        continue
    if ckpt_label in results and not FORCE_RECOMPUTE_EIG:
        n_unst_existing = int(results[ckpt_label]["n_unstable"])
        eig_log(f"{ckpt_label}: SKIPPED (already computed, n_unstable={n_unst_existing}) -- "
                f"set FORCE_RECOMPUTE_EIG=True to redo it")
        continue
    eig_log(f"Loading checkpoint '{ckpt_label}'")
    ckpt = _np_check.load(ckpt_path)
    w_loaded = ckpt["w_last"]
    assert len(w_loaded) == len(w.x.array), (
        f"Checkpoint dof count ({len(w_loaded)}) doesn't match this mesh's "
        f"({len(w.x.array)}) -- OVERRIDES above must match the checkpoint's own parameters.")
    w.x.array[:] = w_loaded
    w.x.scatter_forward()
    eta_target = float(ckpt["eta_last"])
    eta_const.value = eta_target
    problem.solve()
    if problem.solver.getConvergedReason() <= 0:
        # Update 14: before giving up (Update 11's graceful skip), retry the SAME loaded state
        # with progressively relaxed SNES tolerances/iteration budget. Rationale: the initial
        # guess here is the checkpoint's OWN already-arclength-converged state, AT its own eta --
        # not a cold jump from somewhere else -- so it is already essentially at the solution
        # (to within arclength's own tolerance). An ordinary Newton failure from a
        # near-perfect initial guess usually means the ORDINARY (unbordered) Jacobian itself is
        # poorly conditioned at this exact point, not that no solution is nearby -- exactly the
        # situation run_case_arclength_auto's own recovery ladder already relaxes tolerance for
        # (newton_tol_ladder=(1e-5, 1e-4)). A relaxed-tolerance re-solve tests whether THIS
        # SPECIFIC checkpoint's state is stable/unstable -- deliberately NOT restarting from a
        # different nearby point, which would only tell us whether *some* solution exists at
        # this eta, not whether the arclength-found point itself does.
        _saved_tol = problem.solver.getTolerances()
        _recovered = False
        for _rtol, _max_it in [(1e-6, 100), (1e-5, 200), (1e-4, 300)]:
            w.x.array[:] = w_loaded
            w.x.scatter_forward()
            eta_const.value = eta_target
            problem.solver.setTolerances(rtol=_rtol, max_it=_max_it)
            problem.solve()
            if problem.solver.getConvergedReason() > 0:
                eig_log(f"{ckpt_label}: re-converged with relaxed tolerances (rtol={_rtol}, "
                        f"max_it={_max_it}) -- the default tolerance was apparently too tight "
                        f"for this point, not that no nearby solution exists.")
                _recovered = True
                break
        problem.solver.setTolerances(*_saved_tol)
        if not _recovered:
            # Update 11: this used to crash the WHOLE cell (raise, uncaught) on a single label's
            # re-convergence failure -- so one problem label blocked every OTHER label from being
            # analyzed at all. Part B (this notebook's next cell) already handles the identical
            # situation gracefully -- "FAILED to re-converge cleanly ... skipping" plus `continue`.
            # This is expected, not a real error: a checkpoint whose eta_last came from arclength
            # continuation (Update 5) can legitimately sit in a delicate, near-fold region that
            # ordinary fixed-eta Newton cannot be trusted to re-converge to at ANY reasonable
            # tolerance, even though it is a perfectly valid point on the branch.
            eig_log(f"{ckpt_label}: FAILED to re-converge cleanly at eta={eta_target:.5f} even "
                    f"with relaxed tolerances up to rtol=1e-4/max_it=300 -- skipping (a checkpoint "
                    f"from arclength continuation can legitimately sit in a delicate, near-fold "
                    f"region ordinary Newton can't re-converge; this is itself informative, not "
                    f"an error).")
            continue

    eig_log(f"{ckpt_label} (eta={eta_target}): assembling dense Jacobian")
    J_mat = fem.petsc.assemble_matrix(J_form, bcs=bcs)
    J_mat.assemble()
    n = J_mat.getSize()[0]
    indptr, indices, data = J_mat.getValuesCSR()
    J_dense = sp.csr_matrix((data, indices, indptr), shape=(n, n)).toarray()
    J_mat.destroy()
    eig_log(f"eta={eta_target}: Jacobian assembled ({n}x{n}) -- starting dense eigenvalue "
            f"solve (this is the slow part, especially on a refined mesh)")

    # Row-norm equilibration (same as the fold diagnostics elsewhere in
    # this project) before the eigenvalue solve, for numerical sanity --
    # NOTE this rescales M too (M_scaled = scale * diag(M_diag) * scale),
    # which does NOT change the SIGN of the eigenvalues (a similarity-like
    # transform for a diagonal M), only their magnitude -- fine since sign
    # is what we actually care about here.
    row_norms = np.sqrt(np.sum(J_dense**2, axis=1))
    scale = 1.0 / np.sqrt(np.maximum(row_norms, 1e-30))
    J_scaled = (J_dense * scale[None, :]) * scale[:, None]
    M_scaled = np.diag(M_diag * scale**2)

    # Now requests eigenvectors too (right=True), not just eigenvalues --
    # needed to construct a perturbation direction in Part A that is
    # GUARANTEED to have nonzero projection onto an unstable mode (an
    # arbitrary perturbation, like a uniform Cw shift, can have nearly
    # zero overlap with a specific unstable eigenvector, especially when
    # there's only one unstable direction among thousands -- confirmed
    # this is a real risk, not just a theoretical one, when a uniform-
    # shift perturbation at a point with 1 confirmed unstable eigenvalue
    # relaxed flat with no visible growth).
    eigvals_all, eigvecs_all = scipy.linalg.eig(J_scaled, M_scaled, right=True)
    finite_mask = np.isfinite(eigvals_all) & (np.abs(eigvals_all) < 1e10)
    finite = eigvals_all[finite_mask]
    finite_vecs = eigvecs_all[:, finite_mask]
    order = np.argsort(finite.real)
    finite_sorted = finite[order]
    finite_vecs_sorted = finite_vecs[:, order]

    Jloc_arr = _project_scalar_standalone(I_local, V, dx)
    I_now = float(np.mean(Jloc_arr[membrane_dofs_V]))
    s_arr = _project_scalar_standalone(s, V, dx)
    s_now = float(s_arr.max())

    n_unstable = int(np.sum(finite_sorted.real < 0))
    results[ckpt_label] = dict(eigvals=finite_sorted, I=I_now, s_max=s_now, n_unstable=n_unstable)

    if n_unstable > 0:
        # Save the unstable eigenvector(s), unscaled back to physical dof
        # space. The row-norm equilibration used above was J_scaled = D J
        # D, M_scaled = D M D with D=diag(scale) -- so the physical-space
        # eigenvector is D @ v_scaled = scale * v_scaled (elementwise).
        # Real part only (a genuinely real instability should have a
        # real eigenvector, or come as a complex-conjugate pair whose
        # real part already spans the physically meaningful direction).
        unstable_idx = np.where(finite_sorted.real < 0)[0]
        # Confirmed the hard way: a genuine complex-conjugate PAIR of eigenvalues
        # (lambda, lambda*) has CONJUGATE eigenvectors (v, v*) -- and Re(v) == Re(v*)
        # exactly, since real part is invariant under conjugation. Saving each
        # member's real part independently (the previous approach) silently
        # duplicates the SAME direction under two different indices whenever a
        # conjugate pair is present, rather than giving two genuinely different
        # directions. The 2D real invariant subspace a conjugate pair actually
        # spans is Re(v) and Im(v) of ONE representative member -- those two ARE
        # generally independent, unlike Re(v) and Re(v*).
        unstable_vals = finite_sorted[unstable_idx]
        unstable_vecs_raw = finite_vecs_sorted[:, unstable_idx]
        directions = []
        direction_vals = []
        used = np.zeros(len(unstable_idx), dtype=bool)
        for i in range(len(unstable_idx)):
            if used[i]:
                continue
            lam_i = unstable_vals[i]
            if abs(lam_i.imag) < 1e-10:
                # genuinely real eigenvalue -- one real direction
                directions.append(unstable_vecs_raw[:, i].real)
                direction_vals.append(lam_i)
                used[i] = True
            else:
                # look for this eigenvalue's conjugate partner among the rest
                partner = None
                for j in range(i + 1, len(unstable_idx)):
                    if used[j]:
                        continue
                    lam_j = unstable_vals[j]
                    if abs(lam_j - np.conj(lam_i)) < 1e-8 * (abs(lam_i) + 1e-12):
                        partner = j
                        break
                if partner is not None:
                    # genuine pair -- Re(v) and Im(v) of ONE member, not Re(v) and Re(v*)
                    directions.append(unstable_vecs_raw[:, i].real)
                    directions.append(unstable_vecs_raw[:, i].imag)
                    direction_vals.append(lam_i)
                    direction_vals.append(lam_i)  # both real/imag parts tagged with the same pair
                    used[i] = True
                    used[partner] = True
                else:
                    # no partner found (shouldn't normally happen for a real-valued
                    # system's eigendecomposition) -- fall back to real part alone
                    directions.append(unstable_vecs_raw[:, i].real)
                    direction_vals.append(lam_i)
                    used[i] = True
        unstable_vecs_physical = np.array(directions).T * scale[:, None]
        results[ckpt_label]["unstable_eigvecs"] = unstable_vecs_physical
        results[ckpt_label]["unstable_eigvals"] = np.array(direction_vals)
        eig_log(f"{ckpt_label} (eta={eta_target}): saved {unstable_vecs_physical.shape[1]} "
                f"unstable direction(s) from {len(unstable_idx)} unstable eigenvalue(s) "
                f"(conjugate pairs expanded into Re/Im directions, not duplicated)")

    eig_log(f"{ckpt_label} (eta={eta_target}): DONE -- I={I_now/1e4:.4f} A/cm^2, s_max={s_now:.4f}, "
            f"{len(finite_sorted)} finite eigenvalue(s), {n_unstable} unstable")
    print(f"{ckpt_label} (eta={eta_target}): I={I_now/1e4:.4f} A/cm^2, s_max={s_now:.4f}, "
          f"{len(finite_sorted)} finite eigenvalue(s), "
          f"{n_unstable} with negative real part "
          f"(convention check: eta=0.30 should show 0)")
    print(f"  5 eigenvalues closest to zero: "
          f"{np.array2string(finite_sorted[np.argsort(np.abs(finite_sorted.real))][:5], precision=4)}")

np.savez(eig_out_path,
         **{f"{k}__{field}": v for k, dd in results.items() for field, v in dd.items()})
shutil.copy2(eig_out_path, os.path.join(DRIVE_BACKUP, "eigenvalue_check.npz"))
print(f"\nSaved to {eig_out_path}")
print("\nComparing n_unstable between eta1.24_prefold and eta1.27_postfold directly tests the")
print("candidate-fold hypothesis: a genuine saddle-node fold should show eta1.24_prefold with")
print("n_unstable=0 (stable) and eta1.27_postfold with n_unstable>=1 (unstable), mirroring the")
print("exact pattern independently confirmed for the ORIGINAL base-case fold in")
print("PEMFC_Stability_Investigation.ipynb.")


## Part B: Nonlinear time-marching relaxation

Perturbs each steady state and watches whether it relaxes back (stable) or grows/moves to a
different state (unstable). Default mode is `PERTURBATION_MODE="eigenvector"` -- uses Part A's
own unstable eigenvector as the perturbation direction (falling back to a uniform $C_w$ shift
automatically for a checkpoint with 0 unstable eigenvalues, since there's no eigenvector to use
there). Confirmed the hard way, both here and in the original fold's investigation, that a
uniform perturbation can have almost no overlap with a genuinely unstable direction and looks
falsely "stable" over a short time march even at a confirmed-unstable point -- the eigenvector
direction is the reliable choice whenever one is available.

In [ ]:
%%fenicsx -np 1 --time

import sys, os, time, shutil
sys.path.insert(0, "/content")
import numpy as np
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
from petsc4py import PETSc
import ufl
from pemfc_lib import DEFAULT_PARAMS, Psat_atm, _smooth_outflow, _project_scalar_standalone

OUT = "/content/pemfc_sensitivity_rh1p0_stability_results"
os.makedirs(OUT, exist_ok=True)
STAB_LOG = f"{OUT}/relaxation_progress.log"


def stab_log(line):
    ts = time.strftime("%H:%M:%S")
    with open(STAB_LOG, "a") as f:
        f.write(f"[{ts}] {line}\n")
        f.flush()
        os.fsync(f.fileno())
    print(line)


assert os.path.ismount('/content/drive'), (
    "Drive not mounted -- run the 'Mount Google Drive' cell above first.")
DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
n_restored = 0
for fname in os.listdir(DRIVE_BACKUP):
    local_fp = os.path.join(OUT, fname)
    if not os.path.exists(local_fp):
        shutil.copy2(os.path.join(DRIVE_BACKUP, fname), local_fp)
        n_restored += 1
if n_restored:
    print(f"Restored {n_restored} file(s) from Drive backup into {OUT}")

# ============ EDIT THESE ============
CHECKPOINTS = {
    "eta1.24_prefold": f"{OUT}/eta1.24_prefold_checkpoint.npz",
    "eta1.27_postfold": f"{OUT}/eta1.27_postfold_checkpoint.npz",
    "eta1.28_postfold": f"{OUT}/eta1.28_postfold_checkpoint.npz",
}
OVERRIDES = dict(tau_brug=1.1, Lx=0.2e-3, RH_in=1.0)  # MUST match the checkpoints' own parameters
PERTURB_FRACTION_INIT = 0.005  # starting point for the auto-search below (see this
                                # project's own hard-won lesson: right near a fold, s(Cw)
                                # can be extremely sensitive -- start modest)
DT = 0.5
N_STEPS = 200
PERTURBATION_MODE = "eigenvector"  # "uniform" or "eigenvector" -- confirmed the hard way
                                     # (both in the original fold investigation AND here)
                                     # that a uniform perturbation can have almost zero
                                     # overlap with a genuinely unstable direction when there
                                     # are many stable directions (or several unstable ones)
                                     # to get lost among -- looks falsely "stable" over a
                                     # short time march even at a confirmed-unstable point.
EIG_CHECK_PATH = f"{OUT}/eigenvalue_check.npz"  # from Part A above
UNSTABLE_EIGVEC_INDEX = 0  # which unstable eigenvector to use, if more than one was found
PERTURB_SIGN = +1           # +1 or -1 -- flip to test the opposite direction along the
                             # same unstable eigenvector (a genuine saddle-node structure
                             # can behave asymmetrically between the two signs)
FORCE_RECOMPUTE_RELAX = False  # set True to ignore an existing result and redo it.
                                 # False (the normal setting) is what makes switching
                                 # PERTURB_SIGN between runs efficient: eta1.24/eta1.27 use
                                 # sign-independent filenames (uniform-mode fallback, so
                                 # nothing to distinguish) and get correctly SKIPPED when
                                 # unchanged, while eta1.28's sign-tagged filename means
                                 # only the NEW sign actually gets computed. NOTE: even a
                                 # deleted/renamed LOCAL file gets silently restored from
                                 # the Drive backup at the top of this cell -- use
                                 # FORCE_RECOMPUTE_RELAX=True (not manual file deletion) if
                                 # you specifically want to force a fresh run of everything.
# =====================================

# Update 12: this used to be a blanket assert requiring EVERY CHECKPOINTS entry to exist
# before this cell did anything -- see Part A's identical Update 12 note above for why that's
# wrong when Stage 1 genuinely couldn't extract one label at all. Now it just logs and
# continues; the per-label loop below already skips any that aren't there.
_missing_checkpoints = [_lbl for _lbl, _fp in CHECKPOINTS.items() if not os.path.exists(_fp)]
if _missing_checkpoints:
    print(f"NOTE: {len(_missing_checkpoints)} checkpoint(s) not found -- will be SKIPPED below, "
          f"not treated as an error: {_missing_checkpoints}.")

P = dict(DEFAULT_PARAMS)
P.update(OVERRIDES)
F, R = P["F"], P["R"]
K_perm, eps_p = P["K_perm"], P["eps_p"]
p_in, p_out = P["p_in"], P["p_out"]
mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
rho_l = P["rho_l"]
M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
mu_l = P["mu_l"]
T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
RH_in = P["RH_in"]
theta_c_deg = P["theta_c_deg"]
theta_c = theta_c_deg * np.pi / 180.0
sigma_st = P["sigma_st"]
X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
s_smooth_eps = P["s_smooth_eps"]
single_phase = P["single_phase"]
nx, ny = P["nx"], P["ny"]

P_sat = Psat_atm(T_cell - 273.15) * 101325.0
Ly = w_ch + w_rb
y_inlet_end = 0.5 * w_ch
y_rib_end = 0.5 * w_ch + w_rb

comm = MPI.COMM_WORLD
domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
tdim = domain.topology.dim
fdim = tdim - 1

def on_membrane(x): return np.isclose(x[0], 0.0)
def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
facet_indices, facet_markers = [], []
for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                      (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
    idx_ = mesh.locate_entities_boundary(domain, fdim, locator)
    facet_indices.append(idx_)
    facet_markers.append(np.full_like(idx_, tag))
facet_indices = np.concatenate(facet_indices)
facet_markers = np.concatenate(facet_markers)
order_f = np.argsort(facet_indices)
facet_tags = mesh.meshtags(domain, fdim, facet_indices[order_f], facet_markers[order_f])

ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
dx = ufl.Measure("dx", domain=domain)
nvec = ufl.FacetNormal(domain)

V = fem.functionspace(domain, ("Lagrange", 1))
membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))
P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
ME = basix.ufl.mixed_element([P1e, P1e, P1e])
W = fem.functionspace(domain, ME)

w = fem.Function(W, name="w")
w_prev = fem.Function(W, name="w_prev")
v_te = ufl.TestFunction(W)
p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
p_hat_prev, Cw_prev, CgO2_prev = ufl.split(w_prev)
v_p, v_cw, v_cgo2 = ufl.split(v_te)
P_scale = 1000.0
p_sol = p_out + P_scale * p_hat_sol

X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
dry_scale = 1.0 - X_H2O_in
X_O2_eff = X_O2_in * dry_scale
X_N2_eff = X_N2_in * dry_scale
M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
Y_O2_in = X_O2_eff * M_O2 / M_mix_in
Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
rho_g_const = p_out * M_mix_in / (R * T_cell)
Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
Cl_sat = 1.0

inlet_facets = facet_tags.find(TAG_INLET)
outlet_facets = facet_tags.find(TAG_OUTLET)
dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
bcs = [
    fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
    fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
    fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
    fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
]

def s_expr(Cw):
    if single_phase:
        return 0.0 * Cw
    raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
    s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
    s_smooth = ufl.max_value(s_smooth, 0.0)
    return ufl.min_value(s_smooth, s_max_cap)

def min_smooth(a, b, eps=s_smooth_eps):
    return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

hydrophobic = theta_c_deg >= 90.0

def build_terms(Cw_field):
    s_ = s_expr(Cw_field)
    krl_ = s_ ** n_corey
    krg_ = (1.0 - s_) ** n_corey
    lam_l_ = (krl_ / mu_l) / (krl_ / mu_l + krg_ / mu_g + 1e-30)
    lam_g_ = 1.0 - lam_l_
    rho_mix_ = rho_g_const * (1.0 - s_) + rho_l * s_
    return dict(s=s_, lam_l=lam_l_, lam_g=lam_g_, rho_mix=rho_mix_)

terms_now = build_terms(Cw_sol)
terms_prev = build_terms(Cw_prev)
s, lam_l, lam_g, rho_mix = terms_now["s"], terms_now["lam_l"], terms_now["lam_g"], terms_now["rho_mix"]

nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
krl = s ** n_corey
krg = (1.0 - s) ** n_corey
nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
mu_mix = nu_mix * rho_mix
kappa = rho_mix * K_perm / mu_mix
if hydrophobic:
    dJds = 1.417 - 4.240 * s + 3.789 * s**2
else:
    u_hp = 1.0 - s
    dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)
Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
      * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
grads = ufl.grad(s)
u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
CgH2O_local = min_smooth(Cw_sol, Cg_sat)
Weff_w = lam_l + lam_g * CgH2O_local
Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
Jl_vec = Dc * grads
F_conv_w = Weff_w * rho_mix * u_darcy
W_O2 = rho_mix * lam_g * u_darcy

eta_const = fem.Constant(domain, default_scalar_type(0.0))
C_O2_molar = rho_g_const * CgO2_sol / M_O2
I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
N_O2_expr = (M_O2 / (4.0 * F)) * I_local
N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
          - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))

DT_const = fem.Constant(domain, default_scalar_type(DT))
water_mass_now = eps_p * rho_mix * Cw_sol
water_mass_prev = eps_p * terms_prev["rho_mix"] * Cw_prev
oxy_mass_now = eps_p * (1.0 - s) * rho_g_const * CgO2_sol
oxy_mass_prev = eps_p * (1.0 - terms_prev["s"]) * rho_g_const * CgO2_prev

F_water = (((water_mass_now - water_mass_prev) / DT_const) * v_cw * dx
           + Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
           - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
           + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
           - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
F_oxy = (((oxy_mass_now - oxy_mass_prev) / DT_const) * v_cgo2 * dx
         - CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
         + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
         + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
         + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
         + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
F_total = F_pres + F_water + F_oxy

snes_petsc_options = {
    "snes_type": "newtonls", "snes_rtol": 1e-7, "snes_atol": 1e-9,
    "snes_max_it": 50, "snes_error_if_not_converged": False,
    "ksp_type": "preonly", "pc_type": "lu",
}
problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                            petsc_options_prefix="pemfc_higheta_relax_",
                            form_compiler_options={"quadrature_degree": 4})

# VI bound-constrained solver -- same reason as Part A: re-solving at
# eta=1.27 (and the transient steps that follow) without this can fail
# even though the checkpoint itself was extracted successfully with it.
problem.solver.setType("vinewtonrsls")
Cw_dofs_global_vi = W.sub(1).collapse()[1]
CgO2_dofs_global_vi = W.sub(2).collapse()[1]
n_total_vi = len(w.x.array)
lb_arr_vi = np.full(n_total_vi, -1e20)
ub_arr_vi = np.full(n_total_vi, 1e20)
lb_arr_vi[Cw_dofs_global_vi] = 0.0
ub_arr_vi[Cw_dofs_global_vi] = 1.0
lb_arr_vi[CgO2_dofs_global_vi] = 0.0
ub_arr_vi[CgO2_dofs_global_vi] = 1.0
lb_vi = w.x.petsc_vec.copy()
ub_vi = w.x.petsc_vec.copy()
lb_vi.array[:] = lb_arr_vi
ub_vi.array[:] = ub_arr_vi
lb_vi.assemble()
ub_vi.assemble()
problem.solver.setVariableBounds(lb_vi, ub_vi)

free_cw_dofs = np.setdiff1d(W.sub(1).collapse()[1], dofs_cw_in)
all_bc_dofs = np.concatenate([dofs_p_in, dofs_p_out, dofs_cw_in, dofs_cgo2_in])


def measure():
    Jloc_arr = _project_scalar_standalone(I_local, V, dx)
    I_now = float(np.mean(Jloc_arr[membrane_dofs_V]))
    s_arr = _project_scalar_standalone(s, V, dx)
    s_now = float(s_arr.max())
    return I_now, s_now


# --- loop over both candidate points ---
for ckpt_label, ckpt_path in CHECKPOINTS.items():
    # Determine BEFORE picking a filename whether this checkpoint will
    # actually use eigenvector mode (vs. falling back to uniform for a
    # confirmed-stable, n_unstable=0 point) -- uniform mode ignores
    # PERTURB_SIGN entirely, so tagging +1/-1 filenames for a point that
    # can't tell them apart would just duplicate the same computation
    # under two different names every time the sign is flipped.
    will_use_eigvec = False
    if PERTURBATION_MODE == "eigenvector" and os.path.exists(EIG_CHECK_PATH):
        will_use_eigvec = f"{ckpt_label}__unstable_eigvecs" in np.load(EIG_CHECK_PATH).files
    if will_use_eigvec:
        sign_tag = "plus" if PERTURB_SIGN > 0 else "minus"
        relax_path = f"{OUT}/relaxation_{ckpt_label}_sign{sign_tag}.npz"
    else:
        relax_path = f"{OUT}/relaxation_{ckpt_label}.npz"
    if os.path.exists(relax_path) and not FORCE_RECOMPUTE_RELAX:
        d_existing_relax = np.load(relax_path)
        stab_log(f"{ckpt_label} (sign={PERTURB_SIGN:+d}): SKIPPED (already computed -- final "
                 f"I={float(d_existing_relax['I_avg'][-1])/1e4:.4f} A/cm^2, "
                 f"s_max={float(d_existing_relax['s_max'][-1]):.4f}) -- "
                 f"set FORCE_RECOMPUTE_RELAX=True to redo it")
        continue
    if not os.path.exists(ckpt_path):
        stab_log(f"{ckpt_label}: SKIPPED -- checkpoint file not found (Stage 1 could not "
                 f"extract this label at all for this parameter value; see Update 12).")
        continue
    stab_log(f"=== {ckpt_label} (sign={PERTURB_SIGN:+d}): loading checkpoint ===")
    ckpt = np.load(ckpt_path)
    w_loaded = ckpt["w_last"]
    assert len(w_loaded) == len(w.x.array), (
        f"Checkpoint dof count ({len(w_loaded)}) doesn't match this mesh's ({len(w.x.array)}).")
    ETA_TEST = float(ckpt["eta_last"])
    w.x.array[:] = w_loaded
    w.x.scatter_forward()
    w_prev.x.array[:] = w.x.array
    eta_const.value = ETA_TEST

    problem.solve()
    if problem.solver.getConvergedReason() <= 0:
        # Update 14: same relaxed-tolerance retry ladder as Part A above -- see that cell's
        # Update 14 note for why retrying the SAME loaded state (not a fresh initial guess from
        # elsewhere) is the methodologically correct way to test whether THIS checkpoint's own
        # state is stable/unstable.
        _saved_tol = problem.solver.getTolerances()
        _recovered = False
        for _rtol, _max_it in [(1e-6, 100), (1e-5, 200), (1e-4, 300)]:
            w.x.array[:] = w_loaded
            w.x.scatter_forward()
            eta_const.value = ETA_TEST
            problem.solver.setTolerances(rtol=_rtol, max_it=_max_it)
            problem.solve()
            if problem.solver.getConvergedReason() > 0:
                stab_log(f"{ckpt_label}: re-converged with relaxed tolerances (rtol={_rtol}, "
                         f"max_it={_max_it}).")
                _recovered = True
                break
        problem.solver.setTolerances(*_saved_tol)
        if not _recovered:
            stab_log(f"{ckpt_label}: FAILED to re-converge cleanly at eta={ETA_TEST:.5f} even "
                     f"with relaxed tolerances up to rtol=1e-4/max_it=300 -- skipping.")
            continue
    w_prev.x.array[:] = w.x.array
    w_steady = w.x.array.copy()
    I_steady, s_steady = measure()
    stab_log(f"{ckpt_label}: steady state at eta={ETA_TEST:.5f}: "
             f"I={I_steady/1e4:.4f} A/cm^2, s_max={s_steady:.4f}")

    def apply_perturbation_and_measure(fraction, direction):
        w.x.array[:] = w_steady + fraction * direction
        w.x.scatter_forward()
        s_arr_local = _project_scalar_standalone(s, V, dx)
        return float(s_arr_local.max())

    if PERTURBATION_MODE == "uniform":
        direction = np.zeros_like(w.x.array)
        direction[free_cw_dofs] = np.mean(np.abs(w_steady[free_cw_dofs]))
    elif PERTURBATION_MODE == "eigenvector":
        assert os.path.exists(EIG_CHECK_PATH), (
            f"{EIG_CHECK_PATH} not found -- run Part A above first (with n_unstable>0 at "
            f"this checkpoint) so there's an eigenvector to load.")
        d_eig = np.load(EIG_CHECK_PATH)
        eigvec_key = f"{ckpt_label}__unstable_eigvecs"
        eigval_key = f"{ckpt_label}__unstable_eigvals"
        if eigvec_key not in d_eig.files:
            # Graceful fallback, not a hard failure -- a checkpoint with
            # n_unstable=0 (e.g. a confirmed-stable point like the
            # pre-fold candidate) has no unstable eigenvector to load by
            # definition, but that's an expected, legitimate outcome
            # here, not an error condition worth stopping the whole loop
            # over when testing several checkpoints together.
            stab_log(f"{ckpt_label}: no unstable eigenvector on file (this checkpoint's own "
                     f"n_unstable was 0) -- falling back to uniform-mode perturbation for "
                     f"this point specifically")
            direction = np.zeros_like(w.x.array)
            direction[free_cw_dofs] = np.mean(np.abs(w_steady[free_cw_dofs]))
            eigvecs = None
        else:
            eigvecs = d_eig[eigvec_key]
        if eigvecs is not None:
            assert UNSTABLE_EIGVEC_INDEX < eigvecs.shape[1], (
                f"Only {eigvecs.shape[1]} unstable eigenvector(s) available for '{ckpt_label}', "
                f"UNSTABLE_EIGVEC_INDEX={UNSTABLE_EIGVEC_INDEX} is out of range.")
            direction = PERTURB_SIGN * eigvecs[:, UNSTABLE_EIGVEC_INDEX].copy()
            direction[all_bc_dofs] = 0.0  # Dirichlet dofs must stay exact
            # Rescale the WHOLE vector by a SINGLE scalar -- confirmed the hard way in the
            # original fold investigation that scaling only the Cw component (leaving other
            # fields at their raw, tiny unit-2-norm-normalized magnitude) distorts the
            # eigenvector's true field-to-field proportions, which is NOT the direction Part A
            # actually found unstable.
            cw_component = np.abs(direction[free_cw_dofs])
            if cw_component.max() > 0:
                scale_factor = np.mean(np.abs(w_steady[free_cw_dofs])) / cw_component.max()
                direction *= scale_factor
            stab_log(f"{ckpt_label}: using unstable eigenvector index {UNSTABLE_EIGVEC_INDEX} "
                     f"(eigenvalue={d_eig[eigval_key][UNSTABLE_EIGVEC_INDEX]}), sign={PERTURB_SIGN:+d}")
    else:
        raise ValueError(f"Unknown PERTURBATION_MODE={PERTURBATION_MODE!r}")

    S_MAX_SHIFT_TARGET = 0.02
    MIN_FRACTION = 1e-7
    fraction = PERTURB_FRACTION_INIT
    s_shift = None
    while fraction > MIN_FRACTION:
        s_perturbed = apply_perturbation_and_measure(fraction, direction)
        s_shift = abs(s_perturbed - s_steady)
        stab_log(f"{ckpt_label}: trying PERTURB_FRACTION={fraction:.2e}: s_max shift = {s_shift:.4f}")
        if s_shift <= S_MAX_SHIFT_TARGET:
            stab_log(f"{ckpt_label}: accepted PERTURB_FRACTION={fraction:.2e}")
            break
        fraction /= 2.0
    else:
        stab_log(f"{ckpt_label}: even the smallest tested perturbation moved s_max too much -- "
                 f"skipping the time march for this point.")
        continue

    w_prev.x.array[:] = w.x.array  # the perturbed state is the actual initial condition
    eta_const.value = ETA_TEST

    # The raw t=0 diagnostic (I/s_max computed directly from the just-
    # perturbed, not-yet-time-integrated field) is logged for visibility
    # but deliberately NOT recorded into t_hist/I_hist/s_hist -- confirmed
    # the hard way that this can be a wild L2-projection-overshoot
    # artifact near a sharp front (e.g. I=-4.10 A/cm^2 logged once, on a
    # perturbation whose OWN s_max shift looked small), not a real
    # physical transient. The saved/plotted trajectory instead starts
    # from the first genuinely time-integrated (Newton-solved) step.
    I0_h, s0_h = measure()
    stab_log(f"{ckpt_label}: perturbed by {fraction*100:.4f}% -- raw post-perturbation "
             f"diagnostic (NOT part of the saved trajectory, may include projection "
             f"overshoot): I={I0_h/1e4:.4f} A/cm^2, s_max={s0_h:.4f} "
             f"(steady was s_max={s_steady:.4f})")
    t_hist, I_hist, s_hist = [], [], []

    DT_MIN = DT / 64
    step = 0
    w_backup = w.x.array.copy()
    w_prev_backup = w_prev.x.array.copy()
    dt_cur = DT
    while step < N_STEPS and (not t_hist or t_hist[-1] < N_STEPS * DT - 1e-9):
        DT_const.value = dt_cur
        problem.solve()
        if problem.solver.getConvergedReason() > 0:
            w_prev.x.array[:] = w.x.array
            t_hist.append((t_hist[-1] if t_hist else 0.0) + dt_cur)
            I_now, s_now = measure()
            I_hist.append(I_now); s_hist.append(s_now)
            w_backup = w.x.array.copy()
            w_prev_backup = w_prev.x.array.copy()
            step += 1
            if step % 10 == 0 or dt_cur < DT:
                stab_log(f"{ckpt_label}: step {step}/{N_STEPS} (t={t_hist[-1]:.2f}s, "
                         f"dt={dt_cur:.4f}): I={I_now/1e4:.4f} A/cm^2, s_max={s_now:.4f}")
            dt_cur = min(dt_cur * 1.5, DT)
        else:
            w.x.array[:] = w_backup
            w_prev.x.array[:] = w_prev_backup
            dt_cur = dt_cur / 2.0
            stab_log(f"{ckpt_label}: step {step}: did not converge, halving to dt={dt_cur:.4f}")
            if dt_cur < DT_MIN:
                stab_log(f"{ckpt_label}: dt below {DT_MIN:.5f} -- stopping time march here")
                break

    np.savez(relax_path, t=np.array(t_hist), I_avg=np.array(I_hist), s_max=np.array(s_hist),
             I_steady=I_steady, s_steady=s_steady, eta=ETA_TEST, dt=DT, perturb_sign=PERTURB_SIGN)
    shutil.copy2(relax_path, os.path.join(DRIVE_BACKUP, os.path.basename(relax_path)))
    stab_log(f"{ckpt_label} (sign={PERTURB_SIGN:+d}): saved to {relax_path}. "
             f"Final: I={I_hist[-1]/1e4:.4f} A/cm^2 (steady was {I_steady/1e4:.4f}), "
             f"s_max={s_hist[-1]:.4f} (steady was {s_steady:.4f})")


## Plot: relaxation trajectories

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

OUT = "/content/pemfc_sensitivity_rh1p0_stability_results"
labels = ["eta1.24_prefold", "eta1.27_postfold", "eta1.28_postfold"]
base_colors = ["#2874a6", "#c0392b", "#27ae60"]

# For each label, plot BOTH sign variants if present (new naming:
# relaxation_{label}_sign{plus,minus}.npz), falling back to the old,
# sign-unaware filename (relaxation_{label}.npz) if that's all that
# exists -- keeps this compatible with results saved before sign-aware
# filenames were added, without needing to recompute them.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
plotted_any = False
for label, base_color in zip(labels, base_colors):
    candidates = sorted(glob.glob(f"{OUT}/relaxation_{label}_sign*.npz"))
    old_style = f"{OUT}/relaxation_{label}.npz"
    if not candidates and os.path.exists(old_style):
        candidates = [old_style]
    if not candidates:
        print(f"{label}: no relaxation data found, skipping")
        continue
    for fp in candidates:
        d = np.load(fp)
        sign = int(d["perturb_sign"]) if "perturb_sign" in d.files else None
        sign_label = f", sign={sign:+d}" if sign is not None else ""
        line_style = "-" if (sign is None or sign > 0) else "--"
        t, I_avg, s_max = d["t"], d["I_avg"] / 1e4, d["s_max"]
        I_steady, s_steady = float(d["I_steady"]) / 1e4, float(d["s_steady"])
        plot_label = f"{label} (eta={float(d['eta']):.3f}{sign_label})"
        axes[0].plot(t, I_avg, line_style, color=base_color, label=plot_label)
        axes[0].axhline(I_steady, color=base_color, ls=":", lw=0.8, alpha=0.5)
        axes[1].plot(t, s_max, line_style, color=base_color, label=plot_label)
        axes[1].axhline(s_steady, color=base_color, ls=":", lw=0.8, alpha=0.5)
        plotted_any = True

axes[0].set_xlabel("time (s)")
axes[0].set_ylabel(r"$I$ (A/cm$^2$)")
axes[0].set_title("Current density relaxation (dotted = steady-state value)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel("time (s)")
axes[1].set_ylabel(r"$s_{max}$")
axes[1].set_title("Saturation relaxation")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
if plotted_any:
    plt.savefig(f"{OUT}/relaxation_comparison.png", dpi=150)
plt.show()


## Compute: field snapshots across fractions x eigenvector indices

For each fraction in `FRACTIONS`, tries `EIGVEC_INDICES_TO_TRY` (default `[0, 1, 2]`) in order,
each run out to `TARGET_TIME=40s` (not a step count -- every (fraction, index) combination gets
the same real time budget regardless of how small `dt` got along the way), with full 2D field
snapshots captured at 11 sampled times up to that point.

**This notebook runs `EIGVEC_INDICES_TO_TRY = [0, 2]`** (matching the baseline's third-branch and second-branch seed indices exactly) and **`FRACTIONS = [0.032]`** only (matching the baseline seed fraction).

In [ ]:
%%fenicsx -np 1 --time

import sys, os, time, shutil
sys.path.insert(0, "/content")
import numpy as np
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
from petsc4py import PETSc
import ufl
from pemfc_lib import DEFAULT_PARAMS, Psat_atm, _smooth_outflow, _project_scalar_standalone

# ============ EDIT THIS FIRST, ESPECIALLY WHEN RUNNING FRACTIONS IN PARALLEL ============
# Give each parallel copy of this notebook (one fraction per Colab session) its own tag,
# so their log files don't collide when writing to the SAME Drive backup folder at the
# same time -- e.g. "frac00025" for the fraction=0.00025 copy, "frac0005" for 0.0005, etc.
# Doesn't need to match FRACTIONS below exactly -- it's just a label for this run's log.
PARALLEL_TAG = "sens_rh1p0"
# ==========================================================================================

OUT = "/content/pemfc_sensitivity_rh1p0_minusdir_results"
os.makedirs(OUT, exist_ok=True)
STAB_LOG = f"{OUT}/minusdir_fields_progress_{PARALLEL_TAG}.log"


def stab_log(line):
    ts = time.strftime("%H:%M:%S")
    with open(STAB_LOG, "a") as f:
        f.write(f"[{ts}] {line}\n")
        f.flush()
        os.fsync(f.fileno())
    print(line)


assert os.path.ismount('/content/drive'), (
    "Drive not mounted -- run the 'Mount Google Drive' cell above first.")
DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_minusdir_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
n_restored = 0
for fname in os.listdir(DRIVE_BACKUP):
    local_fp = os.path.join(OUT, fname)
    if not os.path.exists(local_fp):
        shutil.copy2(os.path.join(DRIVE_BACKUP, fname), local_fp)
        n_restored += 1
if n_restored:
    print(f"Restored {n_restored} file(s) from Drive backup into {OUT}")

OVERRIDES = dict(tau_brug=1.1, Lx=0.2e-3, RH_in=1.0)  # MUST match the checkpoint's own parameters
DT = 0.5  # needed early (DT_const, in the transient weak form below), not just in the
          # later EDIT block where the sweep-specific parameters live

P = dict(DEFAULT_PARAMS)
P.update(OVERRIDES)
F, R = P["F"], P["R"]
K_perm, eps_p = P["K_perm"], P["eps_p"]
p_in, p_out = P["p_in"], P["p_out"]
mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
rho_l = P["rho_l"]
M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
mu_l = P["mu_l"]
T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
RH_in = P["RH_in"]
theta_c_deg = P["theta_c_deg"]
theta_c = theta_c_deg * np.pi / 180.0
sigma_st = P["sigma_st"]
X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
s_smooth_eps = P["s_smooth_eps"]
single_phase = P["single_phase"]
nx, ny = P["nx"], P["ny"]

P_sat = Psat_atm(T_cell - 273.15) * 101325.0
Ly = w_ch + w_rb
y_inlet_end = 0.5 * w_ch
y_rib_end = 0.5 * w_ch + w_rb

comm = MPI.COMM_WORLD
domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
tdim = domain.topology.dim
fdim = tdim - 1

def on_membrane(x): return np.isclose(x[0], 0.0)
def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
facet_indices, facet_markers = [], []
for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                      (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
    idx_ = mesh.locate_entities_boundary(domain, fdim, locator)
    facet_indices.append(idx_)
    facet_markers.append(np.full_like(idx_, tag))
facet_indices = np.concatenate(facet_indices)
facet_markers = np.concatenate(facet_markers)
order_f = np.argsort(facet_indices)
facet_tags = mesh.meshtags(domain, fdim, facet_indices[order_f], facet_markers[order_f])

ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
dx = ufl.Measure("dx", domain=domain)
nvec = ufl.FacetNormal(domain)

V = fem.functionspace(domain, ("Lagrange", 1))
membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))
P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
ME = basix.ufl.mixed_element([P1e, P1e, P1e])
W = fem.functionspace(domain, ME)

w = fem.Function(W, name="w")
w_prev = fem.Function(W, name="w_prev")
v_te = ufl.TestFunction(W)
p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
p_hat_prev, Cw_prev, CgO2_prev = ufl.split(w_prev)
v_p, v_cw, v_cgo2 = ufl.split(v_te)
P_scale = 1000.0
p_sol = p_out + P_scale * p_hat_sol

X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
dry_scale = 1.0 - X_H2O_in
X_O2_eff = X_O2_in * dry_scale
X_N2_eff = X_N2_in * dry_scale
M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
Y_O2_in = X_O2_eff * M_O2 / M_mix_in
Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
rho_g_const = p_out * M_mix_in / (R * T_cell)
Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
Cl_sat = 1.0

inlet_facets = facet_tags.find(TAG_INLET)
outlet_facets = facet_tags.find(TAG_OUTLET)
dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
bcs = [
    fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
    fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
    fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
    fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
]

def s_expr(Cw):
    if single_phase:
        return 0.0 * Cw
    raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
    s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
    s_smooth = ufl.max_value(s_smooth, 0.0)
    return ufl.min_value(s_smooth, s_max_cap)

def min_smooth(a, b, eps=s_smooth_eps):
    return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

hydrophobic = theta_c_deg >= 90.0

def build_terms(Cw_field):
    s_ = s_expr(Cw_field)
    krl_ = s_ ** n_corey
    krg_ = (1.0 - s_) ** n_corey
    lam_l_ = (krl_ / mu_l) / (krl_ / mu_l + krg_ / mu_g + 1e-30)
    lam_g_ = 1.0 - lam_l_
    rho_mix_ = rho_g_const * (1.0 - s_) + rho_l * s_
    return dict(s=s_, lam_l=lam_l_, lam_g=lam_g_, rho_mix=rho_mix_)

terms_now = build_terms(Cw_sol)
terms_prev = build_terms(Cw_prev)
s, lam_l, lam_g, rho_mix = terms_now["s"], terms_now["lam_l"], terms_now["lam_g"], terms_now["rho_mix"]

nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
krl = s ** n_corey
krg = (1.0 - s) ** n_corey
nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
mu_mix = nu_mix * rho_mix
kappa = rho_mix * K_perm / mu_mix
if hydrophobic:
    dJds = 1.417 - 4.240 * s + 3.789 * s**2
else:
    u_hp = 1.0 - s
    dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)
Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
      * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
grads = ufl.grad(s)
u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
CgH2O_local = min_smooth(Cw_sol, Cg_sat)
Weff_w = lam_l + lam_g * CgH2O_local
Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
Jl_vec = Dc * grads
F_conv_w = Weff_w * rho_mix * u_darcy
W_O2 = rho_mix * lam_g * u_darcy

eta_const = fem.Constant(domain, default_scalar_type(0.0))
C_O2_molar = rho_g_const * CgO2_sol / M_O2
I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
N_O2_expr = (M_O2 / (4.0 * F)) * I_local
N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
          - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))

DT_const = fem.Constant(domain, default_scalar_type(DT))
water_mass_now = eps_p * rho_mix * Cw_sol
water_mass_prev = eps_p * terms_prev["rho_mix"] * Cw_prev
oxy_mass_now = eps_p * (1.0 - s) * rho_g_const * CgO2_sol
oxy_mass_prev = eps_p * (1.0 - terms_prev["s"]) * rho_g_const * CgO2_prev

F_water = (((water_mass_now - water_mass_prev) / DT_const) * v_cw * dx
           + Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
           - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
           + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
           - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
F_oxy = (((oxy_mass_now - oxy_mass_prev) / DT_const) * v_cgo2 * dx
         - CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
         + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
         + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
         + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
         + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
F_total = F_pres + F_water + F_oxy

snes_petsc_options = {
    "snes_type": "newtonls", "snes_rtol": 1e-7, "snes_atol": 1e-9,
    "snes_max_it": 50, "snes_error_if_not_converged": False,
    "ksp_type": "preonly", "pc_type": "lu",
}
problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                            petsc_options_prefix="pemfc_higheta_relax_",
                            form_compiler_options={"quadrature_degree": 4})

# VI bound-constrained solver -- same reason as Part A: re-solving at
# eta=1.27 (and the transient steps that follow) without this can fail
# even though the checkpoint itself was extracted successfully with it.
problem.solver.setType("vinewtonrsls")
Cw_dofs_global_vi = W.sub(1).collapse()[1]
CgO2_dofs_global_vi = W.sub(2).collapse()[1]
n_total_vi = len(w.x.array)
lb_arr_vi = np.full(n_total_vi, -1e20)
ub_arr_vi = np.full(n_total_vi, 1e20)
lb_arr_vi[Cw_dofs_global_vi] = 0.0
ub_arr_vi[Cw_dofs_global_vi] = 1.0
lb_arr_vi[CgO2_dofs_global_vi] = 0.0
ub_arr_vi[CgO2_dofs_global_vi] = 1.0
lb_vi = w.x.petsc_vec.copy()
ub_vi = w.x.petsc_vec.copy()
lb_vi.array[:] = lb_arr_vi
ub_vi.array[:] = ub_arr_vi
lb_vi.assemble()
ub_vi.assemble()
problem.solver.setVariableBounds(lb_vi, ub_vi)

free_cw_dofs = np.setdiff1d(W.sub(1).collapse()[1], dofs_cw_in)
all_bc_dofs = np.concatenate([dofs_p_in, dofs_p_out, dofs_cw_in, dofs_cgo2_in])


def measure():
    Jloc_arr = _project_scalar_standalone(I_local, V, dx)
    I_now = float(np.mean(Jloc_arr[membrane_dofs_V]))
    s_arr = _project_scalar_standalone(s, V, dx)
    s_now = float(s_arr.max())
    return I_now, s_now



# ============ EDIT THESE ============
STABILITY_DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup"
CKPT_LABEL_CANDIDATES = ["eta1.28_postfold", "eta1.27_postfold", "eta1.24_prefold"]  # tried
    # IN THIS ORDER: closest to the fold first, since that is where an unstable mode is
    # most likely to actually exist -- see the Update 13 note below the EDIT block.
EIG_CHECK_PATH = f"{STABILITY_DRIVE_BACKUP}/eigenvalue_check.npz"
EIGVEC_INDICES_TO_TRY = [0, 2]  # matches baseline's third-branch (idx=0) and second-branch (idx=2) seeds  # tried IN ORDER for each fraction, before moving to the next fraction
PERTURB_SIGN = -1
FRACTIONS = [0.032] ###[0.00025, 0.0005, 0.001, 0.002, 0.004, 0.008, 0.016, 0.032, 0.064]
SNAPSHOT_TIMES = [1.0, 3.0, 5.0, 8.0, 12.0, 16.0, 20.0, 25.0, 30.0, 35.0, 40.0]  # seconds --
                                                           # matched to the nearest
                                                           # actual accepted step
TARGET_TIME = 60.0  ###40.0 # seconds -- ALL fractions run until this real time is reached
                      # (not a step count), so every fraction gets a consistent, comparable
                      # end time regardless of how small dt got along the way
MAX_STEPS_SAFETY = 500  # generous ceiling only to guard against a genuine infinite loop --
                          # not meant to bind in practice
DT_MIN_local = DT / 64
RESUME_ROLLBACK_POINTS = 0  # 0 = resume from the last saved point; 1 or 2 = back up that many points first
MAX_FIELD_HISTORY = 3  # how many recent field states to keep for rollback
FORCE_RECOMPUTE = False  # fresh sensitivity run
# =====================================

assert os.path.exists(EIG_CHECK_PATH), f"{EIG_CHECK_PATH} not found -- run Part A first."
d_eig = np.load(EIG_CHECK_PATH)

# Update 13: picking a checkpoint used to mean "does the file exist" alone (Update 12) -- but
# confirmed on a real tau1p5 run, a checkpoint can exist AND have been successfully analyzed by
# Part A, and STILL have nothing to perturb along: eta1.28_postfold's own eigenvalue count came
# back n_unstable=0 there (genuinely stable, not a bug -- Part A only saves an
# "..._unstable_eigvecs" entry when n_unstable>0), and eta1.27_postfold never got analyzed at
# all (failed to re-converge in Part A). `d_eig[eigvec_key]` on either one used to be a direct,
# unguarded dict lookup, so it crashed with a raw KeyError instead of trying the next candidate.
# This now searches all 3 labels, in fold-proximity order, for the first one that actually has
# a saved unstable eigenvector to use.
CKPT_LABEL, CKPT_PATH, eigvecs = None, None, None
_diagnosis = {}
for _cand in CKPT_LABEL_CANDIDATES:
    _cand_path = f"{STABILITY_DRIVE_BACKUP}/{_cand}_checkpoint.npz"
    if not os.path.exists(_cand_path):
        _diagnosis[_cand] = "checkpoint not extracted at all (Stage 1 could not reach it)"
        continue
    _cand_key = f"{_cand}__unstable_eigvecs"
    if _cand_key in d_eig.files:
        CKPT_LABEL, CKPT_PATH, eigvecs = _cand, _cand_path, d_eig[_cand_key]
        break
    _n_key = f"{_cand}__n_unstable"
    _n_unst = int(d_eig[_n_key]) if _n_key in d_eig.files else None
    _diagnosis[_cand] = (f"n_unstable={_n_unst} (genuinely stable -- nothing to perturb along)"
                          if _n_unst == 0 else
                          "not analyzed in Part A (checkpoint failed to re-converge there)")
if CKPT_LABEL is None:
    raise RuntimeError(
        f"None of {CKPT_LABEL_CANDIDATES} currently has a confirmed unstable eigenvector at "
        f"this parameter value, so this cell cannot pick a perturbation direction. Per-label "
        f"status: {_diagnosis}. This may be a genuine sensitivity finding (no unstable branch "
        f"reachable at these eta targets for this parameter value), not a bug -- consider "
        f"whether TARGETS in the extraction cell needs to probe further past the fold.")
stab_log(f"Using checkpoint '{CKPT_LABEL}' -- has {eigvecs.shape[1]} unstable eigenvector(s) "
         f"available.")

ckpt = np.load(CKPT_PATH)
w_loaded = ckpt["w_last"]
assert len(w_loaded) == len(w.x.array), "Checkpoint dof count doesn't match this mesh."
ETA_TEST = float(ckpt["eta_last"])
w.x.array[:] = w_loaded
w.x.scatter_forward()
w_prev.x.array[:] = w.x.array
eta_const.value = ETA_TEST
problem.solve()
if problem.solver.getConvergedReason() <= 0:
    # Update 14: same relaxed-tolerance retry ladder as Part A/B -- a checkpoint that needed
    # this ladder to be analyzed there (that is the only way it could have qualified as
    # CKPT_LABEL above, via Update 13's search) can need it again here, since this is an
    # independent fresh solve, not a reuse of Part A's already-converged state.
    _saved_tol = problem.solver.getTolerances()
    _recovered = False
    for _rtol, _max_it in [(1e-6, 100), (1e-5, 200), (1e-4, 300)]:
        w.x.array[:] = w_loaded
        w.x.scatter_forward()
        eta_const.value = ETA_TEST
        problem.solver.setTolerances(rtol=_rtol, max_it=_max_it)
        problem.solve()
        if problem.solver.getConvergedReason() > 0:
            stab_log(f"Re-converged {CKPT_LABEL} with relaxed tolerances (rtol={_rtol}, "
                     f"max_it={_max_it}).")
            _recovered = True
            break
    problem.solver.setTolerances(*_saved_tol)
    assert _recovered, (
        f"Loaded checkpoint '{CKPT_LABEL}' did not re-converge cleanly at eta={ETA_TEST:.5f}, "
        f"even with relaxed tolerances up to rtol=1e-4/max_it=300 -- unlike in Part A, there is "
        f"no other candidate label to fall back to here (this one was already the best "
        f"available among the 3). Try relaxing further, or investigate with "
        f"diagnose_fold_point().")
w_steady = w.x.array.copy()
I_steady, s_steady = measure()
stab_log(f"Steady state at eta={ETA_TEST:.5f}: I={I_steady/1e4:.4f} A/cm^2, s_max={s_steady:.4f}")

def build_direction(idx):
    d = PERTURB_SIGN * eigvecs[:, idx].copy()
    d[all_bc_dofs] = 0.0
    cw_component = np.abs(d[free_cw_dofs])
    if cw_component.max() > 0:
        scale_factor = np.mean(np.abs(w_steady[free_cw_dofs])) / cw_component.max()
        d *= scale_factor
    return d

# Precomputed once per index (cheap -- just array algebra), selected inside the nested loop
# below rather than recomputed every fraction.
#
# Bug fix: this checkpoint's own eigvecs array can have FEWER unstable eigenvalues than
# baseline's 3 -- e.g. only 1, if a Stage-1 backoff landed this parameter's "postfold"
# checkpoint at a lower eta_last than eta=1.28 (see this notebook's own log from Step 1/Part
# A). An idx beyond eigvecs.shape[1]-1 simply doesn't exist AT THIS PARAMETER POINT -- that is
# a valid sensitivity finding (fewer unstable modes here than at baseline), not an error -- so
# it is skipped with a clear log message instead of letting a raw IndexError abort every
# index's computation, including ones (like idx=0) that ARE available.
n_available = eigvecs.shape[1]
stab_log(f"eigvecs available at this checkpoint: {n_available} (indices 0..{n_available-1})")
directions_by_idx = {}
for _idx in EIGVEC_INDICES_TO_TRY:
    if _idx >= n_available:
        stab_log(f"idx={_idx}: SKIPPED -- only {n_available} unstable eigenvalue(s) at this "
                 f"parameter point's checkpoint (need index <= {n_available - 1}); a valid "
                 f"sensitivity finding, not an error, so it is simply not tested here.")
        continue
    directions_by_idx[_idx] = build_direction(_idx)

coords = V.tabulate_dof_coordinates()[:, :2]

def snapshots_to_arrays(snapshots):
    # .1f tag format, matching the ORIGINAL (already-saved, 7-fraction) files' own
    # convention exactly -- confirmed the hard way that a mismatched format here
    # would silently break compatibility with everything already computed.
    out = {}
    for target, snap in snapshots.items():
        tag = f"{target:.1f}".replace(".", "p")
        for field, arr in snap.items():
            out[f"snap_{tag}__{field}"] = arr
    return out


def arrays_to_snapshots(d):
    snaps = {}
    prefixes = sorted({k.split("__")[0] for k in d.files if k.startswith("snap_")})
    for pfx in prefixes:
        target = float(pfx.replace("snap_", "").replace("p", "."))
        snaps[target] = {k.split("__")[1]: d[k] for k in d.files if k.startswith(pfx + "__")}
    return snaps


for fraction in FRACTIONS:
    for eigvec_idx in EIGVEC_INDICES_TO_TRY:
        if eigvec_idx not in directions_by_idx:
            continue  # already logged as skipped above (idx not available at this checkpoint)
        direction = directions_by_idx[eigvec_idx]
        frac_tag = f"{fraction:.5f}".replace(".", "p")
        out_path = f"{OUT}/minusdir_fields_frac{frac_tag}_idx{eigvec_idx}.npz"
        inprogress_path = f"{OUT}/minusdir_fields_frac{frac_tag}_idx{eigvec_idx}_inprogress.npz"
        if os.path.exists(out_path) and not FORCE_RECOMPUTE:
            stab_log(f"fraction={fraction:.5f} idx={eigvec_idx}: SKIPPED (already computed) -- "
                     f"set FORCE_RECOMPUTE=True to redo it")
            continue

        w_field_history = []  # rolling buffer for rollback-on-resume, mirrors the
                               # high-eta exploration notebook's own resume logic
        resumed = False
        if os.path.exists(inprogress_path) and not FORCE_RECOMPUTE:
            d_ck = np.load(inprogress_path)
            w_hist_saved = d_ck["w_history"]
            w_prev_hist_saved = d_ck["w_prev_history"]
            n_avail = w_hist_saved.shape[0]
            back = min(RESUME_ROLLBACK_POINTS, n_avail - 1)
            if back != RESUME_ROLLBACK_POINTS:
                stab_log(f"fraction={fraction:.5f} idx={eigvec_idx}: only {n_avail} saved field state(s) -- "
                         f"rolling back {back} point(s) instead of the requested "
                         f"{RESUME_ROLLBACK_POINTS}")
            pick_idx = n_avail - 1 - back
            w.x.array[:] = w_hist_saved[pick_idx]
            w.x.scatter_forward()
            w_prev.x.array[:] = w_prev_hist_saved[pick_idx]
            eta_const.value = ETA_TEST
            problem.solve()
            if problem.solver.getConvergedReason() > 0:
                t_all = list(d_ck["t_hist"]); I_all = list(d_ck["I_hist"]); s_all = list(d_ck["s_hist"])
                keep_n = len(t_all) - back
                t_hist, I_hist, s_hist = t_all[:keep_n], I_all[:keep_n], s_all[:keep_n]
                step = int(d_ck["step"]) - back
                dt_cur = float(d_ck["dt_cur"]) if back == 0 else DT
                snapshots = arrays_to_snapshots(d_ck)
                # Drop any snapshot captured AFTER the point we rolled back to
                t_cutoff = t_hist[-1] if t_hist else 0.0
                snapshots = {tt: sn for tt, sn in snapshots.items() if tt <= t_cutoff + 1e-9}
                remaining_targets = sorted(t for t in SNAPSHOT_TIMES if t not in snapshots)
                w_field_history = [w_hist_saved[i] for i in range(pick_idx + 1)]
                w_prev_field_history = [w_prev_hist_saved[i] for i in range(pick_idx + 1)]
                stab_log(f"fraction={fraction:.5f} idx={eigvec_idx}: RESUMED from in-progress checkpoint "
                         f"(step {step}, t={t_cutoff:.2f}s, rolled back {back} point(s), "
                         f"{len(snapshots)} snapshot(s) preserved)")
                resumed = True
            else:
                stab_log(f"fraction={fraction:.5f} idx={eigvec_idx}: in-progress checkpoint did not re-converge "
                         f"cleanly -- starting this fraction fresh")

        if not resumed:
            stab_log(f"=== fraction={fraction:.5f} idx={eigvec_idx}: perturbing and capturing field snapshots ===")
            w.x.array[:] = w_steady + fraction * direction
            w.x.scatter_forward()
            w_prev.x.array[:] = w.x.array
            eta_const.value = ETA_TEST
            t_hist, I_hist, s_hist = [], [], []
            snapshots = {}
            remaining_targets = sorted(SNAPSHOT_TIMES)
            step = 0
            dt_cur = DT
            w_field_history = [w.x.array.copy()]
            w_prev_field_history = [w_prev.x.array.copy()]

        w_backup = w.x.array.copy()
        w_prev_backup = w_prev.x.array.copy()

        def write_inprogress():
            np.savez(inprogress_path,
                      w_history=np.array(w_field_history[-MAX_FIELD_HISTORY:]),
                      w_prev_history=np.array(w_prev_field_history[-MAX_FIELD_HISTORY:]),
                      t_hist=np.array(t_hist), I_hist=np.array(I_hist), s_hist=np.array(s_hist),
                      step=step, dt_cur=dt_cur, **snapshots_to_arrays(snapshots))
            shutil.copy2(inprogress_path, os.path.join(DRIVE_BACKUP, os.path.basename(inprogress_path)))

        while step < MAX_STEPS_SAFETY and (not t_hist or t_hist[-1] < TARGET_TIME - 1e-9):
            DT_const.value = dt_cur
            problem.solve()
            newton_ok = problem.solver.getConvergedReason() > 0
            if newton_ok:
                I_now, s_now = measure()
                phys_ok = I_now > -50.0
            else:
                I_now, s_now, phys_ok = None, None, False
            if newton_ok and phys_ok:
                w_prev.x.array[:] = w.x.array
                t_now = (t_hist[-1] if t_hist else 0.0) + dt_cur
                t_hist.append(t_now)
                I_hist.append(I_now); s_hist.append(s_now)
                w_backup = w.x.array.copy()
                w_prev_backup = w_prev.x.array.copy()
                w_field_history.append(w.x.array.copy())
                w_prev_field_history.append(w_prev.x.array.copy())
                if len(w_field_history) > MAX_FIELD_HISTORY:
                    w_field_history.pop(0)
                    w_prev_field_history.pop(0)
                step += 1
                dt_cur = min(dt_cur * 1.5, DT)
                if step % 5 == 0:
                    stab_log(f"  fraction={fraction:.5f} idx={eigvec_idx}: step {step} (t={t_now:.2f}s/{TARGET_TIME:.0f}s): "
                             f"I={I_now/1e4:.4f} A/cm^2, s_max={s_now:.4f}")
                    write_inprogress()
                # Capture a snapshot the first time we reach/pass each target time
                while remaining_targets and t_now >= remaining_targets[0] - 1e-9:
                    target = remaining_targets.pop(0)
                    s_arr = _project_scalar_standalone(s, V, dx)
                    p_hat_arr = w.sub(0).collapse().x.array
                    p_arr = p_out + P_scale * p_hat_arr
                    Cw_arr = w.sub(1).collapse().x.array
                    CgO2_arr = w.sub(2).collapse().x.array
                    snapshots[target] = dict(t_actual=t_now, s=s_arr, p=p_arr, Cw=Cw_arr,
                                              CgO2=CgO2_arr, I=I_now, s_max=s_now)
                    stab_log(f"  fraction={fraction:.5f} idx={eigvec_idx}: captured snapshot for target "
                             f"t={target:.1f}s (actual t={t_now:.2f}s)")
                    write_inprogress()
            else:
                w.x.array[:] = w_backup
                w_prev.x.array[:] = w_prev_backup
                dt_cur = dt_cur / 2.0
                if dt_cur < DT_MIN_local:
                    t_so_far = t_hist[-1] if t_hist else 0.0
                    stab_log(f"  fraction={fraction:.5f} idx={eigvec_idx}: dt below {DT_MIN_local:.5f} -- stopping "
                             f"(step {step}, t={t_so_far:.2f}s/{TARGET_TIME:.0f}s)")
                    break

        np.savez(out_path, coords=coords, t=np.array(t_hist), I_avg=np.array(I_hist),
                 s_max=np.array(s_hist), I_steady=I_steady, s_steady=s_steady, eta=ETA_TEST,
                 snapshot_times=np.array(sorted(snapshots.keys())), **snapshots_to_arrays(snapshots))
        shutil.copy2(out_path, os.path.join(DRIVE_BACKUP, os.path.basename(out_path)))
        if os.path.exists(inprogress_path):
            os.remove(inprogress_path)
        stab_log(f"fraction={fraction:.5f} idx={eigvec_idx}: saved {len(snapshots)} snapshot(s) to {out_path} "
                 f"(in-progress checkpoint cleared)")


## Plot: field contours, one figure per eigenvector index

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

OUT = "/content/pemfc_sensitivity_rh1p0_minusdir_results"

# ============ EDIT THIS ============
FRACTION_TO_PLOT = 0.032  # matches whichever fraction this session computed
EIGVEC_INDICES = [0, 2]
# =====================================

THICKNESS_EXAGGERATION = 2.0

# (data key, display label, colormap)
FIELDS = [
    ("s", "Liquid saturation $s$", "Blues"),
    ("CgO2", "Gas O$_2$ mass fraction $C_g^{O_2}$", "YlOrRd"),
]


def to_grid(x, y, values, decimals=9):
    xr, yr = np.round(x, decimals), np.round(y, decimals)
    xu, yu = np.unique(xr), np.unique(yr)
    x_idx = {v: i for i, v in enumerate(xu)}
    y_idx = {v: i for i, v in enumerate(yu)}
    grid = np.full((len(yu), len(xu)), np.nan)
    for xi, yi, val in zip(xr, yr, values):
        grid[y_idx[yi], x_idx[xi]] = val
    return xu, yu, grid


frac_tag = f"{FRACTION_TO_PLOT:.5f}".replace(".", "p")

for idx in EIGVEC_INDICES:
    fp = f"{OUT}/minusdir_fields_frac{frac_tag}_idx{idx}.npz"
    try:
        d = np.load(fp)
    except FileNotFoundError:
        print(f"idx={idx}: no data found ({fp}), skipping")
        continue

    snap_times = sorted(d["snapshot_times"])
    if not snap_times:
        print(f"idx={idx}: no snapshots captured, skipping")
        continue

    coords = d["coords"]
    x_gdl_mm = coords[:, 0] * 1e3 * THICKNESS_EXAGGERATION
    y_channel_mm = coords[:, 1] * 1e3

    # Shared color range per field, computed across ALL snapshot times for
    # this (fraction, index), so the same field is directly comparable frame to frame.
    field_ranges = {}
    for key, _, _ in FIELDS:
        all_vals = np.concatenate([d[f"snap_{f'{t:.1f}'.replace('.', 'p')}__{key}"]
                                    for t in snap_times])
        field_ranges[key] = (float(all_vals.min()), float(all_vals.max()))

    fig, axes = plt.subplots(len(FIELDS), len(snap_times),
                              figsize=(3.0 * len(snap_times), 4.2 * len(FIELDS)),
                              constrained_layout=True)
    if len(snap_times) == 1:
        axes = axes[:, np.newaxis]

    for row, (key, label, cmap) in enumerate(FIELDS):
        vmin, vmax = field_ranges[key]
        vmin, vmax = max(0.0, vmin), min(1.0, vmax)  # both s and CgO2 are physically in [0,1]
        for col, target in enumerate(snap_times):
            ax = axes[row, col]
            tag = f"{target:.1f}".replace(".", "p")
            field_arr = d[f"snap_{tag}__{key}"]
            t_actual = float(d[f"snap_{tag}__t_actual"])
            I_val = float(d[f"snap_{tag}__I"]) / 1e4
            xu, yu, grid = to_grid(x_gdl_mm, y_channel_mm, field_arr)
            tpc = ax.contourf(xu, yu, np.clip(grid, vmin, vmax),
                               levels=np.linspace(vmin, vmax, 21), cmap=cmap,
                               vmin=vmin, vmax=vmax)
            if row == 0:
                ax.set_title(f"t={t_actual:.1f}s\nI={I_val:.3f} A/cm$^2$", fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_aspect("equal")
        fig.colorbar(tpc, ax=axes[row, :], shrink=0.7, label=label)

    fig.suptitle(f"fraction={FRACTION_TO_PLOT:g}, eigvec idx={idx} -- field evolution "
                 f"(GDL thickness {THICKNESS_EXAGGERATION:.0f}$\\times$ exaggerated)",
                 fontsize=11)
    plt.savefig(f"{OUT}/minusdir_fields_frac{frac_tag}_idx{idx}.png", dpi=150, bbox_inches="tight")
    plt.show()


## Plot: response vs. eigenvector index, for one fraction

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

OUT = "/content/pemfc_sensitivity_rh1p0_minusdir_results"

# ============ EDIT THIS ============
FRACTION_TO_PLOT = 0.032
EIGVEC_INDICES = [0, 2]
# =====================================

frac_tag = f"{FRACTION_TO_PLOT:.5f}".replace(".", "p")
colors = {0: "#2874a6", 1: "#c0392b", 2: "#27ae60"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
found_any = False
for idx in EIGVEC_INDICES:
    fp = f"{OUT}/minusdir_fields_frac{frac_tag}_idx{idx}.npz"
    if not os.path.exists(fp):
        print(f"idx={idx}: no data found ({fp}), skipping")
        continue
    d = np.load(fp)
    t, I_avg, s_max = d["t"], d["I_avg"] / 1e4, d["s_max"]
    I_steady, s_steady = float(d["I_steady"]) / 1e4, float(d["s_steady"])
    axes[0].plot(t, I_avg, "-", color=colors[idx], lw=1.5, label=f"idx={idx}")
    axes[1].plot(t, s_max, "-", color=colors[idx], lw=1.5, label=f"idx={idx}")
    found_any = True

# Update 15: this used to be a bare `assert found_any, "..."` -- which crashed with a generic,
# unhelpful message whenever the Part C compute cell above (Update 13) could not produce ANY
# minusdir_fields_*.npz file for this parameter value (e.g. every CKPT_LABEL_CANDIDATES entry
# disqualified there -- see that cell's own RuntimeError). This is not a new failure mode, it's
# the SAME "nothing to plot yet" situation the field-contour plot cell above already handles
# gracefully (try/except FileNotFoundError, no downstream assert) -- this cell just hadn't been
# brought in line with that pattern yet. Close the empty figure and explain why, instead of
# crashing with a message that doesn't point at the actual root cause.
if not found_any:
    plt.close(fig)
    print(f"No data found for fraction={FRACTION_TO_PLOT} at any of {EIGVEC_INDICES} -- nothing "
          f"to plot. This is expected (not a bug) when the Part C compute cell above could not "
          f"produce ANY minusdir_fields_*.npz file for this parameter value -- e.g. because none "
          f"of CKPT_LABEL_CANDIDATES currently has a confirmed unstable eigenvector to perturb "
          f"along (see that cell's own RuntimeError message/log for the specific reason). "
          f"Re-run this plot cell once Part C has produced at least one result file.")
else:
    axes[0].axhline(I_steady, color="gray", ls="--", lw=0.8, label="original steady")
    axes[0].set_xlabel("time (s)")
    axes[0].set_ylabel(r"$I$ (A/cm$^2$)")
    axes[0].set_title(f"Current density vs. eigenvector index (fraction={FRACTION_TO_PLOT:g})")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)

    axes[1].axhline(s_steady, color="gray", ls="--", lw=0.8, label="original steady")
    axes[1].set_xlabel("time (s)")
    axes[1].set_ylabel(r"$s_{max}$")
    axes[1].set_title(f"Saturation vs. eigenvector index (fraction={FRACTION_TO_PLOT:g})")
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{OUT}/minusdir_fields_frac{frac_tag}_idx_comparison.png", dpi=150)
    plt.show()

## Download results

In [ ]:
import os, zipfile
from google.colab import files

OUT = "/content/pemfc_sensitivity_rh1p0_minusdir_results"
zip_path = "/content/pemfc_sensitivity_rh1p0_minusdir_results.zip"

# Update 17: this used to be a bare `assert os.path.isdir(OUT) and os.listdir(OUT), "..."` --
# crashed unhelpfully whenever Part C's compute cell (Update 13) could not produce ANY result
# file for this parameter value (e.g. no candidate checkpoint currently has a confirmed
# unstable eigenvector -- see that cell's own RuntimeError/log for the specific reason).
# Nothing to download in that case is expected, not a bug -- skip gracefully instead.
if not (os.path.isdir(OUT) and os.listdir(OUT)):
    print(f"No results found in {OUT} -- nothing to download yet. This is expected (not a "
          f"bug) when the compute cell above could not produce any result file for this "
          f"parameter value (see that cell's own diagnostic/log for the specific reason). Run "
          f"the cells above first, or re-run this cell once they've produced at least one "
          f"result file.")
else:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(OUT)):
            zf.write(os.path.join(OUT, fname), fname)

    print(f"Zipped {len(os.listdir(OUT))} files ({os.path.getsize(zip_path)/1e3:.1f} KB)")
    files.download(zip_path)


## Part D: branch continuation from each perturbation plateau

`MinusDirection_Summary.ipynb`'s merged classification table (all `idx`/direction combinations
at this SAME `eta=1.28` checkpoint -- `idx` is an eigenvector index, not a different $\eta$)
shows up to 4 visually distinct "settled -- new steady state" plateaus once plotted against
perturbation fraction:

- `idx=0` (`-` direction): $s_{max}\approx 0.28$--$0.33$
- `idx=0` (`+` direction, from Part C above): a lower plateau at large fractions,
  $s_{max}\approx 0.185$--$0.19$
- `idx=1` (`-` direction): drifts from $s_{max}\approx 0.50$--$0.60$ at small fractions down to
  $\approx 0.17$--$0.29$ at large fractions -- possibly passing near more than one of the other
  plateaus
- `idx=2` (`-` direction): $s_{max}\approx 0.09$--$0.19$ over most fractions (the original
  second-branch arc, seeded from `idx=2`/0.032 -- the one seed that's been continued the
  longest, though not the only one that's actually done)

Whether these are genuinely different branches, or the same branch reached from different
starting points, can't be settled by comparing $s_{max}$ values alone at a single $\eta$ -- two
points can coincidentally share an $s_{max}$ at $\eta=1.28$ while diverging immediately on
either side, or sit on the same curve despite looking different here. The only way to tell is to
actually continue each one in $\eta$ and see whether the resulting curves overlap.

This cell reuses `continue_branch()` verbatim from `SecondBranch_Continuation.ipynb` (same
adaptive stepper, same BIG JUMP jump-gate fix) and writes its results into that notebook's own
`pemfc_tau1.1_Lx0.2mm_secondbranch_backup` Drive folder -- so the two notebooks share one set of
checkpoints, and it doesn't matter which one you run first.

Of the 6 seeds below, only 2 are actually already done as of this update (`idx=2`/0.032,
`idx=1`/0.00025 -- their checkpoints already say `complete=True` and will just be skipped here).
The other 4 are NOT done yet and will genuinely compute here if this cell runs before
`SecondBranch_Continuation.ipynb` does:

- `idx=0`/0.016 and `idx=2`/0.064 (`-` direction) were added to `SecondBranch_Continuation.ipynb`
  this session but haven't actually been run anywhere yet -- this cell computes them exactly the
  same way that notebook would, into the same shared checkpoints, so running THIS notebook's
  Part D is enough on its own; `SecondBranch_Continuation.ipynb` doesn't need a separate run
  afterward (though its own plot cell is fine to open for the same result).
- **`idx=0`, fraction=0.032, `+` direction (new)** -- tests whether `idx=0`'s own lower
  `+`-direction plateau ($s_{max}\approx0.185$) lands on the same branch as `idx=2`'s arc, which
  sits at a similar $s_{max}$. Needs Part C above to have actually been (re-)run for
  fraction=0.032 since this update (see the note on Part C) -- otherwise its raw field state
  won't exist yet and this cell will say so plainly rather than silently skipping it.
- **`idx=1`, fraction=0.064, `-` direction (new)** -- tests whether `idx=1`'s own low-$s$
  endpoint ($s_{max}\approx0.167$, already confirmed via direct Newton in
  `MinusDirection_Summary.ipynb`) also lands there.

**Recommended order:** run Part C's raw-field-save update for fraction=0.032 first (needed by
the `+` direction seed above), then run this cell -- it covers everything in one pass.


In [ ]:
%%fenicsx -np 1 --time

import sys, os, time, shutil
sys.path.insert(0, "/content")
import numpy as np
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
from petsc4py import PETSc
import ufl
from pemfc_lib import DEFAULT_PARAMS, Psat_atm, _smooth_outflow, _project_scalar_standalone

OUT = "/content/pemfc_sensitivity_rh1p0_cont_results"
os.makedirs(OUT, exist_ok=True)
LOG_PATH = f"{OUT}/secondbranch_progress.log"


def alog(line):
    ts = time.strftime("%H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {line}\n")
        f.flush()
        os.fsync(f.fileno())
    print(line)


assert os.path.ismount('/content/drive'), (
    "Drive not mounted -- run the 'Mount Google Drive' cell above first (this %%fenicsx cell "
    "runs in a subprocess without google.colab available, so it can't mount Drive itself).")
# SAME Drive folder SecondBranch_Continuation.ipynb writes to -- whichever notebook already
# has a seed's checkpoint here, the other picks it up automatically (restored below) and
# skips it rather than recomputing.
DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_cont_backup"
os.makedirs(DRIVE_BACKUP, exist_ok=True)
MINUSDIR_DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_minusdir_backup"
PLUS_DRIVE_BACKUP = "/content/drive/MyDrive/pemfc_sensitivity_rh1p0_stability_backup"
n_restored = 0
for fname in os.listdir(DRIVE_BACKUP):
    local_fp = os.path.join(OUT, fname)
    if not os.path.exists(local_fp):
        shutil.copy2(os.path.join(DRIVE_BACKUP, fname), local_fp)
        n_restored += 1
if n_restored:
    alog(f"Restored {n_restored} file(s) from Drive backup into {OUT}")


def build_problem(overrides):
    """Steady-state (p, C_w, Cg_O2) mixed-space Newton problem -- identical to
    HighEta_Exploration.ipynb's own build_problem, unchanged. Returns (w, eta_const, problem,
    measure, check_physical); eta_const is a dolfinx Constant updated between solves."""
    P = dict(DEFAULT_PARAMS)
    P.update(overrides)
    F, R = P["F"], P["R"]
    K_perm, eps_p = P["K_perm"], P["eps_p"]
    p_in, p_out = P["p_in"], P["p_out"]
    mu_g, D_O2_g, D_H2O_g = P["mu_g"], P["D_O2_g"], P["D_H2O_g"]
    I0_ref, alpha_c, alpha_w = P["I0"], P["alpha_c"], P["alpha_w"]
    T_ref_I0, Ea_orr = P["T_ref_I0"], P["Ea_orr"]
    rho_l = P["rho_l"]
    M_H2O, M_O2, M_N2 = P["M_H2O"], P["M_O2"], P["M_N2"]
    mu_l = P["mu_l"]
    T_cell, C_O2_ref = P["T_cell"], P["C_O2_ref"]
    I0 = I0_ref * np.exp(-(Ea_orr / R) * (1.0 / T_cell - 1.0 / T_ref_I0))
    RH_in = P["RH_in"]
    theta_c_deg = P["theta_c_deg"]
    theta_c = theta_c_deg * np.pi / 180.0
    sigma_st = P["sigma_st"]
    X_O2_in, X_N2_in = P["X_O2_in"], P["X_N2_in"]
    w_ch, w_rb, Lx = P["w_ch"], P["w_rb"], P["Lx"]
    n_corey, tau_brug, s_max_cap = P["n_corey"], P["tau_brug"], P["s_max"]
    s_smooth_eps = P["s_smooth_eps"]
    single_phase = P["single_phase"]
    nx, ny = P["nx"], P["ny"]

    P_sat = Psat_atm(T_cell - 273.15) * 101325.0
    Ly = w_ch + w_rb
    y_inlet_end = 0.5 * w_ch
    y_rib_end = 0.5 * w_ch + w_rb

    comm = MPI.COMM_WORLD
    domain = mesh.create_rectangle(comm, [[0.0, 0.0], [Lx, Ly]], [nx, ny], mesh.CellType.triangle)
    tdim = domain.topology.dim
    fdim = tdim - 1

    def on_membrane(x): return np.isclose(x[0], 0.0)
    def on_inlet(x):    return np.isclose(x[0], Lx) & (x[1] <= y_inlet_end + 1e-12)
    def on_rib(x):      return np.isclose(x[0], Lx) & (x[1] > y_inlet_end + 1e-12) & (x[1] <= y_rib_end + 1e-12)
    def on_outlet(x):   return np.isclose(x[0], Lx) & (x[1] > y_rib_end + 1e-12)

    TAG_MEMBRANE, TAG_INLET, TAG_RIB, TAG_OUTLET = 1, 2, 3, 4
    facet_indices, facet_markers = [], []
    for tag, locator in [(TAG_MEMBRANE, on_membrane), (TAG_INLET, on_inlet),
                          (TAG_RIB, on_rib), (TAG_OUTLET, on_outlet)]:
        idx_ = mesh.locate_entities_boundary(domain, fdim, locator)
        facet_indices.append(idx_)
        facet_markers.append(np.full_like(idx_, tag))
    facet_indices = np.concatenate(facet_indices)
    facet_markers = np.concatenate(facet_markers)
    order_f = np.argsort(facet_indices)
    facet_tags = mesh.meshtags(domain, fdim, facet_indices[order_f], facet_markers[order_f])

    ds_meas = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags)
    dx = ufl.Measure("dx", domain=domain)
    nvec = ufl.FacetNormal(domain)

    V = fem.functionspace(domain, ("Lagrange", 1))
    membrane_dofs_V = fem.locate_dofs_topological(V, fdim, facet_tags.find(TAG_MEMBRANE))
    P1e = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
    ME = basix.ufl.mixed_element([P1e, P1e, P1e])
    W = fem.functionspace(domain, ME)

    w = fem.Function(W, name="w")
    v_te = ufl.TestFunction(W)
    p_hat_sol, Cw_sol, CgO2_sol = ufl.split(w)
    v_p, v_cw, v_cgo2 = ufl.split(v_te)
    P_scale = 1000.0
    p_sol = p_out + P_scale * p_hat_sol

    X_H2O_in = np.clip(RH_in * P_sat / p_in, 0.0, 0.98)
    dry_scale = 1.0 - X_H2O_in
    X_O2_eff = X_O2_in * dry_scale
    X_N2_eff = X_N2_in * dry_scale
    M_mix_in = X_O2_eff * M_O2 + X_N2_eff * M_N2 + X_H2O_in * M_H2O
    Y_O2_in = X_O2_eff * M_O2 / M_mix_in
    Y_H2O_in = X_H2O_in * M_H2O / M_mix_in
    M_dry_in = X_O2_in * M_O2 + X_N2_in * M_N2
    rho_g_const = p_out * M_mix_in / (R * T_cell)
    Cg_sat = (P_sat * M_H2O) / (P_sat * M_H2O + (p_in - P_sat) * M_dry_in)
    Cl_sat = 1.0

    inlet_facets = facet_tags.find(TAG_INLET)
    outlet_facets = facet_tags.find(TAG_OUTLET)
    dofs_p_in = fem.locate_dofs_topological(W.sub(0), fdim, inlet_facets)
    dofs_p_out = fem.locate_dofs_topological(W.sub(0), fdim, outlet_facets)
    dofs_cw_in = fem.locate_dofs_topological(W.sub(1), fdim, inlet_facets)
    dofs_cgo2_in = fem.locate_dofs_topological(W.sub(2), fdim, inlet_facets)
    bcs = [
        fem.dirichletbc(default_scalar_type((p_in - p_out) / P_scale), dofs_p_in, W.sub(0)),
        fem.dirichletbc(default_scalar_type(0.0), dofs_p_out, W.sub(0)),
        fem.dirichletbc(default_scalar_type(Y_H2O_in), dofs_cw_in, W.sub(1)),
        fem.dirichletbc(default_scalar_type(Y_O2_in), dofs_cgo2_in, W.sub(2)),
    ]

    def s_expr(Cw):
        if single_phase:
            return 0.0 * Cw
        raw = rho_g_const * (Cw - Cg_sat) / (rho_l * (Cl_sat - Cw) + rho_g_const * (Cw - Cg_sat) + 1e-12)
        s_smooth = 0.5 * (raw + ufl.sqrt(raw**2 + s_smooth_eps**2))
        s_smooth = ufl.max_value(s_smooth, 0.0)
        return ufl.min_value(s_smooth, s_max_cap)

    def min_smooth(a, b, eps=s_smooth_eps):
        return 0.5 * (a + b - ufl.sqrt((a - b) ** 2 + eps ** 2))

    hydrophobic = theta_c_deg >= 90.0
    s = s_expr(Cw_sol)
    krl = s ** n_corey
    krg = (1.0 - s) ** n_corey
    lam_l = (krl / mu_l) / (krl / mu_l + krg / mu_g + 1e-30)
    lam_g = 1.0 - lam_l
    rho_mix = rho_g_const * (1.0 - s) + rho_l * s
    nu_g_c, nu_l_c = mu_g / rho_g_const, mu_l / rho_l
    nu_mix = 1.0 / (krg / nu_g_c + krl / nu_l_c + 1e-30)
    mu_mix = nu_mix * rho_mix
    kappa = rho_mix * K_perm / mu_mix
    if hydrophobic:
        dJds = 1.417 - 4.240 * s + 3.789 * s**2
    else:
        u_hp = 1.0 - s
        dJds = -(1.417 - 4.240 * u_hp + 3.789 * u_hp**2)
    Dc = ((lam_l * lam_g * K_perm / (nu_mix + 1e-30))
          * sigma_st * np.cos(theta_c) * (eps_p / K_perm) ** 0.5 * dJds)
    Gamma = Dc * (1.0 - Cg_sat) / (rho_l - rho_g_const * Cg_sat)
    grads = ufl.grad(s)
    u_darcy = -(kappa / rho_mix) * ufl.grad(p_sol)
    CgH2O_local = min_smooth(Cw_sol, Cg_sat)
    Weff_w = lam_l + lam_g * CgH2O_local
    Deff_O2 = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_O2_g
    Diff_w = rho_g_const * (eps_p * (1.0 - s)) ** tau_brug * D_H2O_g + Gamma * rho_mix
    Jl_vec = Dc * grads
    F_conv_w = Weff_w * rho_mix * u_darcy
    W_O2 = rho_mix * lam_g * u_darcy

    eta_const = fem.Constant(domain, default_scalar_type(0.0))
    C_O2_molar = rho_g_const * CgO2_sol / M_O2
    I_local = (1.0 - s) * I0 * (C_O2_molar / C_O2_ref) * ufl.exp(alpha_c * F * eta_const / (R * T_cell))
    N_O2_expr = (M_O2 / (4.0 * F)) * I_local
    N_w_expr = (M_H2O * (1.0 + 2.0 * alpha_w) / (2.0 * F)) * I_local

    F_pres = (kappa * ufl.inner(ufl.grad(p_sol), ufl.grad(v_p)) * dx
              - (N_w_expr - N_O2_expr) * v_p * ds_meas(TAG_MEMBRANE))
    F_water = (Diff_w * ufl.inner(ufl.grad(Cw_sol), ufl.grad(v_cw)) * dx
               - ufl.inner(F_conv_w, ufl.grad(v_cw)) * dx
               + _smooth_outflow(ufl.inner(F_conv_w, nvec)) * v_cw * ds_meas(TAG_OUTLET)
               - N_w_expr * v_cw * ds_meas(TAG_MEMBRANE))
    F_oxy = (-CgO2_sol * ufl.inner(W_O2, ufl.grad(v_cgo2)) * dx
             + Deff_O2 * ufl.inner(ufl.grad(CgO2_sol), ufl.grad(v_cgo2)) * dx
             + CgO2_sol * ufl.inner(Jl_vec, ufl.grad(v_cgo2)) * dx
             + CgO2_sol * _smooth_outflow(ufl.inner(W_O2, nvec)) * v_cgo2 * ds_meas(TAG_OUTLET)
             + N_O2_expr * v_cgo2 * ds_meas(TAG_MEMBRANE))
    F_total = F_pres + F_water + F_oxy

    snes_petsc_options = {
        "snes_type": "newtonls", "snes_rtol": 1e-7, "snes_atol": 1e-9,
        "snes_max_it": 50, "snes_error_if_not_converged": False,
        "ksp_type": "preonly", "pc_type": "lu",
    }
    problem = NonlinearProblem(F_total, w, bcs=bcs, petsc_options=snes_petsc_options,
                                petsc_options_prefix=f"pemfc_secondbranch_{id(overrides)}_",
                                form_compiler_options={"quadrature_degree": 4})

    # Bound-constrained Newton (SNES VI), same reason as every other notebook in this
    # project: keeps Cw/CgO2 in [0,1] at every iterate rather than detecting a bad state
    # after the fact.
    problem.solver.setType("vinewtonrsls")
    Cw_dofs_global = W.sub(1).collapse()[1]
    CgO2_dofs_global = W.sub(2).collapse()[1]
    n_total = len(w.x.array)
    lb_arr = np.full(n_total, -1e20)
    ub_arr = np.full(n_total, 1e20)
    lb_arr[Cw_dofs_global] = 0.0
    ub_arr[Cw_dofs_global] = 1.0
    lb_arr[CgO2_dofs_global] = 0.0
    ub_arr[CgO2_dofs_global] = 1.0
    lb = w.x.petsc_vec.copy()
    ub = w.x.petsc_vec.copy()
    lb.array[:] = lb_arr
    ub.array[:] = ub_arr
    lb.assemble()
    ub.assemble()
    problem.solver.setVariableBounds(lb, ub)

    def measure():
        Jloc_arr = _project_scalar_standalone(I_local, V, dx)
        I_now = float(np.mean(Jloc_arr[membrane_dofs_V]))
        s_arr = _project_scalar_standalone(s, V, dx)
        s_now = float(s_arr.max())
        return I_now, s_now

    def check_physical(I_now, tol=0.01, I_tol=-50.0):
        Cw_arr = w.sub(1).collapse().x.array
        CgO2_arr = w.sub(2).collapse().x.array
        Cw_min, Cw_max = float(Cw_arr.min()), float(Cw_arr.max())
        CgO2_min, CgO2_max = float(CgO2_arr.min()), float(CgO2_arr.max())
        conc_ok = (Cw_min > -tol and Cw_max < 1 + tol and
                   CgO2_min > -tol and CgO2_max < 1 + tol)
        I_ok = I_now > I_tol
        ok = conc_ok and I_ok
        return ok, dict(Cw_min=Cw_min, Cw_max=Cw_max, CgO2_min=CgO2_min, CgO2_max=CgO2_max, I=I_now)

    return w, eta_const, problem, measure, check_physical


def continue_branch(w, eta_const, problem, measure, check_physical,
                     w_init, eta_init, I_init, s_init, eta_target,
                     checkpoint_path, ds_init=0.02, ds_min=5e-4, log_fn=alog,
                     s_max_flooding_thresh=0.05, ds_flooding_cap=0.005,
                     s_max_jump_thresh=0.1, resume_rollback_points=0,
                     I_jump_thresh=3000.0, big_jump_multipliers=(1.0, 2.0, 3.0),
                     big_jump_jump_gate=True):
    """Same adaptive restore-on-failure / step-halving-then-escalating / BIG-JUMP-fallback
    stepper as HighEta_Exploration.ipynb's adaptive_polarization_curve, generalized to (a)
    start from an ARBITRARY given field state (w_init at eta_init) instead of always a fresh
    eta=0.02 solve, and (b) step in EITHER eta direction (eta_target may be above or below
    eta_init) -- the jump gates, flooding step cap, BIG JUMP fallback, and checkpoint/resume
    mechanics are otherwise unchanged, EXCEPT for one deliberate change from the first version
    of this notebook: `big_jump_jump_gate` (default True). Originally, a BIG JUMP candidate was
    accepted purely on Newton convergence + physical-bounds validity, bypassing the s_max/I
    jump-consistency gate entirely -- the reasoning being that a bigger eta step legitimately
    allows a bigger physical change. In practice this let Newton land on a DIFFERENT, merely
    physically-VALID solution (i.e. a different branch entirely) and accept it silently, which
    is exactly what happened on this notebook's first two runs. With the gate turned on, a BIG
    JUMP candidate must ALSO pass the same s_max_jump_thresh/I_jump_thresh consistency check
    used for regular steps before it's accepted; if every multiplier fails that check (or fails
    to converge / fails physical bounds), the leg stops there and reports it plainly as a
    probable fold or isola boundary, rather than quietly continuing on the wrong branch."""
    going_up = eta_target > eta_init
    direction = 1.0 if going_up else -1.0

    resumed = False
    w_field_history = []
    if checkpoint_path is not None and os.path.exists(checkpoint_path):
        d_ckpt = np.load(checkpoint_path)
        if "w_history" in d_ckpt.files:
            w_hist_saved = d_ckpt["w_history"]
            n_avail = w_hist_saved.shape[0]
            back = min(resume_rollback_points, n_avail - 1)
            if back != resume_rollback_points:
                log_fn(f"  Only {n_avail} saved field state(s) available -- rolling back "
                       f"{back} point(s) instead of the requested {resume_rollback_points}")
            pick_idx = n_avail - 1 - back
            w.x.array[:] = w_hist_saved[pick_idx]
            w.x.scatter_forward()
            all_results = [(float(e), float(i), float(s), int(it)) for e, i, s, it in
                            zip(d_ckpt["eta"], d_ckpt["I_avg"], d_ckpt["s_max"], d_ckpt["iters"])]
            keep_n = len(all_results) - back
            results = all_results[:keep_n]
            eta = results[-1][0]
            eta_const.value = eta
            problem.solve()
            if problem.solver.getConvergedReason() > 0:
                I0, s0 = results[-1][1], results[-1][2]
                w_field_history = [w_hist_saved[i] for i in range(pick_idx + 1)]
                log_fn(f"  Resumed from checkpoint at eta={eta:.4f} ({len(results)} prior "
                       f"point(s) preserved, rolled back {back} point(s))")
                resumed = True
            else:
                log_fn(f"  Checkpoint at eta={eta:.4f} did not re-converge cleanly -- "
                       f"restarting this leg from the shared starting point")
    if not resumed:
        w.x.array[:] = w_init
        w.x.scatter_forward()
        eta = eta_init
        eta_const.value = eta
        I0, s0 = I_init, s_init
        results = [(eta, I0, s0, 0)]
        w_field_history = [w.x.array.copy()]

    w_last_good = w.x.array.copy()
    s_last_accepted = s0
    I_last_accepted = I0
    ds = ds_init
    consecutive_I_jump_rejections = 0
    consecutive_failures = 0
    MAX_FIELD_HISTORY = 3

    def write_checkpoint():
        if checkpoint_path is None:
            return
        eta_c = np.array([r[0] for r in results]); I_c = np.array([r[1] for r in results])
        s_c = np.array([r[2] for r in results]); it_c = np.array([r[3] for r in results])
        np.savez(checkpoint_path, eta=eta_c, I_avg=I_c, s_max=s_c, iters=it_c,
                 w_history=np.array(w_field_history[-MAX_FIELD_HISTORY:]), complete=False)

    write_checkpoint()

    def not_done():
        return (eta < eta_target - 1e-9) if going_up else (eta > eta_target + 1e-9)

    while not_done():
        eta_try = eta + direction * ds
        eta_try = min(eta_try, eta_target) if going_up else max(eta_try, eta_target)
        eta_const.value = eta_try
        problem.solve()
        newton_ok = problem.solver.getConvergedReason() > 0
        if newton_ok:
            I_now, s_now = measure()
            phys_ok, phys_info = check_physical(I_now)
        else:
            I_now, s_now, phys_ok, phys_info = None, None, False, None
        s_jump = (s_now - s_last_accepted) if newton_ok and phys_ok else None
        I_jump = (abs(I_now - I_last_accepted)) if newton_ok and phys_ok else None
        s_jump_rejected = newton_ok and phys_ok and s_jump is not None and abs(s_jump) > s_max_jump_thresh
        I_jump_rejected = newton_ok and phys_ok and I_jump is not None and I_jump > I_jump_thresh
        jump_rejected = s_jump_rejected or I_jump_rejected
        at_floor = (ds / 2.0) < ds_min
        I_jump_twice_in_a_row = I_jump_rejected and consecutive_I_jump_rejections >= 2
        force_accept = (jump_rejected and at_floor) or I_jump_twice_in_a_row
        if newton_ok and phys_ok and (not jump_rejected or force_accept):
            results.append((eta_try, I_now, s_now, problem.solver.getIterationNumber()))
            w_last_good = w.x.array.copy()
            w_field_history.append(w.x.array.copy())
            if len(w_field_history) > MAX_FIELD_HISTORY:
                w_field_history.pop(0)
            eta = eta_try
            s_last_accepted = s_now
            I_last_accepted = I_now
            consecutive_failures = 0
            consecutive_I_jump_rejections = 0
            growth_cap = ds_flooding_cap if s_now > s_max_flooding_thresh else ds_init
            ds = min(ds * 1.5, growth_cap)
            flood_note = " (flooding-capped)" if s_now > s_max_flooding_thresh else ""
            note = ""
            if force_accept:
                bits = []
                if s_jump_rejected: bits.append(f"s_max jump of {s_jump:.4f}")
                if I_jump_rejected: bits.append(f"I jump of {I_jump/1e4:.4f} A/cm^2")
                note = f" [accepted despite {' and '.join(bits)} -- at floor or repeated]"
            log_fn(f"  eta={eta:.4f}: I={I_now/1e4:.4f} A/cm^2, s_max={s_now:.4f}, "
                   f"ds now {ds:.4f}{flood_note}{note}")
            write_checkpoint()
        else:
            w.x.array[:] = w_last_good
            w.x.scatter_forward()
            consecutive_failures += 1
            consecutive_I_jump_rejections = (consecutive_I_jump_rejections + 1
                                              if I_jump_rejected else 0)
            shrink_factor = 2.0 if consecutive_failures == 1 else 8.0
            ds /= shrink_factor
            if not newton_ok:
                reason = "Newton FAILED"
            elif not phys_ok:
                reason = f"Newton converged but REJECTED on physical-bounds check ({phys_info})"
            else:
                bits = []
                if s_jump_rejected:
                    bits.append(f"s_max jumped by {s_jump:.4f}, exceeding {s_max_jump_thresh}")
                if I_jump_rejected:
                    bits.append(f"I jumped by {I_jump/1e4:.4f} A/cm^2, exceeding "
                                f"{I_jump_thresh/1e4:.4f} A/cm^2")
                reason = "REJECTED: " + "; ".join(bits)
            log_fn(f"  eta={eta_try:.4f}: {reason} -- shrinking ds by {shrink_factor:.0f}x to "
                   f"{ds:.5f} ({consecutive_failures} consecutive failure(s) here) and "
                   f"retrying from eta={eta:.4f}")
            if ds < ds_min:
                log_fn(f"  ds below {ds_min} -- trying BIG JUMP fallback(s) before giving up")
                jumped = False
                for jump_mult in big_jump_multipliers:
                    eta_try_big = eta + direction * ds_init * jump_mult
                    eta_try_big = min(eta_try_big, eta_target) if going_up else max(eta_try_big, eta_target)
                    if going_up and eta_try_big <= eta + 1e-9:
                        continue
                    if not going_up and eta_try_big >= eta - 1e-9:
                        continue
                    eta_const.value = eta_try_big
                    problem.solve()
                    newton_ok_big = problem.solver.getConvergedReason() > 0
                    if newton_ok_big:
                        I_big, s_big = measure()
                        phys_ok_big, phys_info_big = check_physical(I_big)
                    else:
                        I_big, s_big, phys_ok_big, phys_info_big = None, None, False, None
                    s_jump_big = (s_big - s_last_accepted) if newton_ok_big and phys_ok_big else None
                    I_jump_big = (abs(I_big - I_last_accepted)) if newton_ok_big and phys_ok_big else None
                    jump_ok_big = (not big_jump_jump_gate) or (
                        newton_ok_big and phys_ok_big
                        and s_jump_big is not None and abs(s_jump_big) <= s_max_jump_thresh
                        and I_jump_big is not None and I_jump_big <= I_jump_thresh)
                    if newton_ok_big and phys_ok_big and jump_ok_big:
                        log_fn(f"  BIG JUMP to eta={eta_try_big:.4f} (x{jump_mult:.1f} ds_init) "
                               f"SUCCEEDED: I={I_big/1e4:.4f} A/cm^2, s_max={s_big:.4f}")
                        results.append((eta_try_big, I_big, s_big, problem.solver.getIterationNumber()))
                        w_last_good = w.x.array.copy()
                        eta = eta_try_big
                        s_last_accepted = s_big
                        I_last_accepted = I_big
                        ds = ds_init
                        consecutive_failures = 0
                        write_checkpoint()
                        jumped = True
                        break
                    else:
                        w.x.array[:] = w_last_good
                        w.x.scatter_forward()
                        if not newton_ok_big:
                            reason_big = "Newton FAILED"
                        elif not phys_ok_big:
                            reason_big = f"REJECTED on physical-bounds check ({phys_info_big})"
                        else:
                            reason_big = (f"converged to a PHYSICALLY VALID but INCONSISTENT "
                                          f"solution -- s_max jumped by {s_jump_big:+.4f} "
                                          f"(thresh {s_max_jump_thresh}), I jumped by "
                                          f"{I_jump_big/1e4:+.4f} A/cm^2 (thresh "
                                          f"{I_jump_thresh/1e4:.4f}); REJECTED as a likely "
                                          f"different-branch solution rather than accepted blindly")
                        log_fn(f"  BIG JUMP to eta={eta_try_big:.4f} also failed ({reason_big})")
                if jumped:
                    continue
                log_fn(f"  ds below {ds_min} and all BIG JUMP fallbacks failed -- stopping here "
                       f"(a probable fold OR isola boundary at/near eta={eta:.4f}: no nearby "
                       f"solution could be reached by a small step, and the only solution(s) "
                       f"reachable by a bigger jump were inconsistent with this branch -- NOT "
                       f"further classified by this notebook)")
                break

    eta_arr = np.array([r[0] for r in results])
    I_arr = np.array([r[1] for r in results])
    s_arr = np.array([r[2] for r in results])
    it_arr = np.array([r[3] for r in results])
    return eta_arr, I_arr, s_arr, it_arr


# ============ EDIT THESE ============
OVERRIDES = dict(tau_brug=1.1, Lx=0.2e-3, RH_in=1.0)  # MUST match MinusDirection_Fields.ipynb exactly
ETA_SOURCE = 1.28          # nominal -- overridden immediately below by the checkpoint's
                            # own actually-reached eta (may be lower than 1.28 if the fold/
                            # obstruction at this parameter value sits before 1.28 -- see
                            # Stage 1's backoff-on-stall logic)
CKPT_LABEL_CANDIDATES = ["eta1.28_postfold", "eta1.27_postfold", "eta1.24_prefold"]  # same
    # fold-proximity priority order as Part C's Update 13 search -- keeps ETA_SOURCE
    # consistent with whichever checkpoint Part C actually perturbed around, instead of
    # picking independently by file-existence alone (Update 12's original approach).
_eig_check_path_for_eta = f"{PLUS_DRIVE_BACKUP}/eigenvalue_check.npz"
_d_eig_for_eta = (np.load(_eig_check_path_for_eta)
                  if os.path.exists(_eig_check_path_for_eta) else None)
CKPT_LABEL, _ckpt_fp_for_eta = None, None
for _cand in CKPT_LABEL_CANDIDATES:
    _cand_fp = f"{PLUS_DRIVE_BACKUP}/{_cand}_checkpoint.npz"
    if not os.path.exists(_cand_fp):
        continue
    if _d_eig_for_eta is not None and f"{_cand}__unstable_eigvecs" in _d_eig_for_eta.files:
        CKPT_LABEL, _ckpt_fp_for_eta = _cand, _cand_fp
        break
if CKPT_LABEL is None:
    # Update 13: matches Part C's own requirement above -- ETA_SOURCE has to come from
    # a checkpoint that ALSO has a confirmed unstable eigenvector (not just a checkpoint
    # file that exists), or it would silently disagree with whatever Part C actually
    # perturbed around.
    raise RuntimeError(
        f"None of {CKPT_LABEL_CANDIDATES} currently has both an extracted checkpoint "
        f"and a confirmed unstable eigenvector at this parameter value -- ETA_SOURCE "
        f"cannot be resolved consistently with what Part C would use. Run Part C first "
        f"(it raises a more detailed per-label diagnosis), or check whether TARGETS in "
        f"the extraction cell needs to probe further past the fold.")
_stab_ckpt_for_eta = np.load(_ckpt_fp_for_eta)
ETA_SOURCE = float(_stab_ckpt_for_eta["eta_last"])
print(f"ETA_SOURCE resolved to {ETA_SOURCE:.4f} from the {CKPT_LABEL} checkpoint's own "
      f"actually-reached eta (nominal target was 1.28)")
ETA_TARGET_UP = 1.60
ETA_TARGET_DOWN = 0.02

# Full seed set: the 4 from SecondBranch_Continuation.ipynb (kept here too, so this notebook's
# own plot is self-contained) plus the 2 new ones this update makes possible. Only idx=2/0.032
# and idx=1/0.00025 are actually already continued as of this update -- their checkpoints
# already exist in DRIVE_BACKUP and will just be restored + skipped. idx=0/0.016 and
# idx=2/0.064 were ADDED to SecondBranch_Continuation.ipynb's own SEED_POINTS this session but
# not yet run anywhere -- if this cell runs before that notebook does, it computes them for
# real, same as the 2 brand-new seeds below (and SecondBranch_Continuation.ipynb will then just
# find them already done).
SEED_POINTS = [
    dict(idx=0, fraction=0.032, direction="-", tag="idx0_frac032_minus",
         note="sensitivity run: matches baseline's THIRD-branch seed recipe exactly (idx=0, "
              "fraction=0.032, '-' direction)",
         force_up=True, force_down=True),
    dict(idx=2, fraction=0.032, direction="-", tag="idx2_frac032_minus",
         note="sensitivity run: matches baseline's SECOND-branch seed recipe exactly (idx=2, "
              "fraction=0.032, '-' direction)",
         force_up=True, force_down=True),
]

# Same tightened jump-consistency thresholds used throughout this line of notebooks.
JUMP_THRESH_KWARGS = dict(s_max_jump_thresh=0.03, I_jump_thresh=800.0, ds_min=1e-6)
# =====================================

w, eta_const, problem, measure, check_physical = build_problem(OVERRIDES)


def load_seed_source(idx, fraction, direction):
    """Loads a raw field checkpoint and re-converges it at ETA_SOURCE. Two possible sources,
    selected by `direction`:
    - "-": MinusDirection_Fields.ipynb's w_history[-1] (unchanged from
      SecondBranch_Continuation.ipynb's own load_seed_source).
    - "+": THIS notebook's own Part C basin-sweep npz, w_final for this fraction -- only
      present for fractions (re)computed since the raw-field-save update above."""
    frac_tag = f"{fraction:.5f}".replace(".", "p")
    if direction == "-":
        source_path = (f"{MINUSDIR_DRIVE_BACKUP}/minusdir_fields_frac{frac_tag}_idx{idx}"
                        f"_inprogress.npz")
        assert os.path.exists(source_path), (
            f"{source_path} not found -- check idx/fraction match a completed "
            f"MinusDirection_Fields run, and its _inprogress.npz checkpoint is still in Drive.")
        w_source = np.load(source_path)["w_history"][-1]
    elif direction == "+":
        source_path = f"{PLUS_DRIVE_BACKUP}/basin_sweep_eta1.28_postfold_signplus.npz"
        assert os.path.exists(source_path), (
            f"{source_path} not found -- run HighEta_Stability.ipynb's Part C at least once.")
        d_source = np.load(source_path)
        key = f"traj_{frac_tag}__w_final"
        assert key in d_source.files, (
            f"'{key}' not found in {source_path} -- this fraction's raw field state hasn't been "
            f"(re)computed since the raw-field-save update. In Part C above, set "
            f"FRACTIONS_TO_SWEEP=[{fraction}] and FORCE_RECOMPUTE_SWEEP=True, re-run, then "
            f"retry this cell.")
        w_source = d_source[key]
    else:
        raise ValueError(f"direction must be '-' or '+', got {direction!r}")
    assert len(w_source) == len(w.x.array), (
        "Source checkpoint dof count doesn't match this mesh -- check OVERRIDES (nx/ny/Lx/"
        "tau_brug) match exactly.")
    w.x.array[:] = w_source
    w.x.scatter_forward()
    eta_const.value = ETA_SOURCE
    problem.solve()
    assert problem.solver.getConvergedReason() > 0, (
        f"Source state (idx={idx}, frac={fraction:g}, direction={direction}) did not "
        f"re-converge cleanly at eta={ETA_SOURCE} -- try a different idx/fraction.")
    I_src, s_src = measure()
    alog(f"Seed idx={idx} frac={fraction:g} direction={direction}: re-converged at "
         f"eta={ETA_SOURCE:.4f} -- I={I_src/1e4:.4f} A/cm^2, s_max={s_src:.4f}")
    return w.x.array.copy(), I_src, s_src


def detect_suspicious_jumps(eta_arr, s_arr, factor=4.0, min_abs=0.02):
    """Flags any single accepted step whose |Delta s_max| is much larger than its IMMEDIATE
    neighboring steps' -- same diagnostic SecondBranch_Continuation.ipynb uses, verbatim."""
    if len(eta_arr) < 3:
        return []
    order = np.argsort(eta_arr)
    eta_s, s_s = eta_arr[order], s_arr[order]
    ds_steps = np.diff(s_s)
    n = len(ds_steps)
    flags = []
    for i in range(n):
        neighbors = []
        if i > 0:
            neighbors.append(abs(ds_steps[i - 1]))
        if i < n - 1:
            neighbors.append(abs(ds_steps[i + 1]))
        if not neighbors:
            continue
        neighbor_avg = sum(neighbors) / len(neighbors)
        d = ds_steps[i]
        if abs(d) > min_abs and abs(d) > factor * neighbor_avg:
            flags.append((eta_s[i], eta_s[i + 1], d, neighbor_avg))
    return flags


def run_leg(direction_label, seed_tag, eta_target, checkpoint_path, force_recompute,
            w_seed, eta_seed, I_seed, s_seed, stepper_kwargs=None):
    skip = False
    if os.path.exists(checkpoint_path) and not force_recompute:
        d_existing = np.load(checkpoint_path)
        is_complete = bool(d_existing["complete"]) if "complete" in d_existing else True
        if is_complete:
            reached = d_existing["eta"].max() if direction_label == "UP" else d_existing["eta"].min()
            alog(f"[{seed_tag}] {direction_label} leg SKIPPED (already complete, reached "
                 f"eta={reached:.4f}) -- set this seed's force_{direction_label.lower()} flag "
                 f"to True to redo it")
            skip = True
        else:
            alog(f"[{seed_tag}] {direction_label} leg: found an INCOMPLETE checkpoint -- resuming")
    if not skip:
        if force_recompute and os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)  # start this leg genuinely fresh, not resumed
        alog(f"=== [{seed_tag}] Continuing {direction_label} from eta={eta_seed:.4f} to "
             f"eta={eta_target:.4f} ===")
        eta_arr, I_arr, s_arr, it_arr = continue_branch(
            w, eta_const, problem, measure, check_physical,
            w_seed, eta_seed, I_seed, s_seed, eta_target,
            checkpoint_path=checkpoint_path, log_fn=alog, **(stepper_kwargs or {}))
        np.savez(checkpoint_path, eta=eta_arr, I_avg=I_arr, s_max=s_arr, iters=it_arr, complete=True)
        shutil.copy2(checkpoint_path, os.path.join(DRIVE_BACKUP, os.path.basename(checkpoint_path)))
        reached = eta_arr.max() if direction_label == "UP" else eta_arr.min()
        alog(f"[{seed_tag}] {direction_label} leg done: I range "
             f"{I_arr.min()/1e4:.4f}-{I_arr.max()/1e4:.4f} A/cm^2, reached eta={reached:.4f} "
             f"(target {eta_target:.4f})")
        short = (reached < eta_target - 1e-6) if direction_label == "UP" else (reached > eta_target + 1e-6)
        if short:
            alog(f"[{seed_tag}] {direction_label} leg STOPPED SHORT of its target -- a genuine "
                 f"candidate for follow-up (possible fold/isola boundary on THIS branch too; "
                 f"not classified further here).")
        else:
            alog(f"[{seed_tag}] {direction_label} leg reached its full target cleanly -- no "
                 f"fold or obstruction found on this branch over that range.")
        for eta_a, eta_b, d_s, typical in detect_suspicious_jumps(eta_arr, s_arr):
            alog(f"  ⚠ SUSPICIOUS JUMP in [{seed_tag}] {direction_label} leg: s_max changed "
                 f"by {d_s:+.4f} in one step between eta={eta_a:.4f} and eta={eta_b:.4f} "
                 f"(typical step elsewhere: {typical:.4f}) -- inspect the log around these eta "
                 f"values before trusting it.")


for seed in SEED_POINTS:
    tag = seed["tag"]
    alog(f"\n{'='*70}\nSeed: idx={seed['idx']} fraction={seed['fraction']:g} "
         f"direction={seed['direction']} (tag={tag})\n{seed.get('note', '')}\n{'='*70}")
    # Bug fix: if Part C above skipped this idx (fewer unstable eigenvalues at this
    # parameter's checkpoint than baseline), there is no source field state to continue
    # from -- load_seed_source raises AssertionError in that case. That is a valid,
    # expected outcome (see this notebook's own README), not a bug -- skip this seed with a
    # clear log message instead of letting it abort every OTHER seed in this list too.
    try:
        w_seed, I_seed, s_seed = load_seed_source(seed["idx"], seed["fraction"], seed["direction"])
    except AssertionError as e:
        alog(f"Seed idx={seed['idx']} fraction={seed['fraction']:g} direction={seed['direction']}: "
             f"SKIPPED -- source not available ({e})")
        continue
    up_path = f"{OUT}/secondbranch_up_{tag}.npz"
    down_path = f"{OUT}/secondbranch_down_{tag}.npz"
    run_leg("UP", tag, ETA_TARGET_UP, up_path, seed.get("force_up", False),
            w_seed, ETA_SOURCE, I_seed, s_seed, stepper_kwargs=JUMP_THRESH_KWARGS)
    run_leg("DOWN", tag, ETA_TARGET_DOWN, down_path, seed.get("force_down", True),
            w_seed, ETA_SOURCE, I_seed, s_seed, stepper_kwargs=JUMP_THRESH_KWARGS)


## Plot: do the plateaus' branches coincide?

Same overlay-plus-cross-seed-comparison plot as `SecondBranch_Continuation.ipynb`'s own plot
cell -- reads every `secondbranch_up_<tag>.npz` / `secondbranch_down_<tag>.npz` found (all 6
seeds above, restored from the shared `DRIVE_BACKUP`), overlays their branches, and prints the
min/max $|\Delta I|$ between every pair of seeds over their shared $\eta$ range: consistently
small values across a pair means those two seeds' branches coincide (same branch, reached from
different starting points); a persistent gap means they are genuinely different branches.


In [ ]:
import os, re, glob
import numpy as np
import matplotlib.pyplot as plt

OUT = "/content/pemfc_sensitivity_rh1p0_cont_results"
ORIGINAL_BRANCH_BACKUP = "/content/drive/MyDrive/pemfc_tau1.1_Lx0.2mm_higheta_backup"

up_files = sorted(glob.glob(f"{OUT}/secondbranch_up_*.npz"))
down_files = sorted(glob.glob(f"{OUT}/secondbranch_down_*.npz"))
# Update 15: this used to be a bare `assert up_files or down_files, "..."` -- which crashed
# unhelpfully whenever the compute cell above couldn't produce ANY secondbranch_{up,down}_*.npz
# file (most commonly because every seed in SEED_POINTS got skipped there with "source not
# available", which itself traces back to Part C's own compute cell never producing a perturbed
# field to seed from -- see that cell's RuntimeError/log for the specific reason). That per-seed
# skip is already handled gracefully one cell up -- this plot cell just needs to not crash when
# the end result is zero files. Rest of the cell is now nested under the `else`.
if not (up_files or down_files):
    print(f"No second-branch results found yet in {OUT} -- nothing to plot. This is expected "
          f"(not a bug), see this cell's own Update 15 comment above for why. Re-run this plot "
          f"cell once the compute cell above has produced at least one result file.")
else:
    tag_re = re.compile(r"secondbranch_(?:up|down)_(.+)\.npz$")
    tags = sorted({tag_re.search(fp).group(1) for fp in up_files + down_files})
    print(f"Found {len(tags)} seed(s): {tags}")

    seed_branches = {}  # tag -> (eta_branch, I_branch, s_branch)
    for tag in tags:
        up_p = f"{OUT}/secondbranch_up_{tag}.npz"
        down_p = f"{OUT}/secondbranch_down_{tag}.npz"
        eta_parts, I_parts, s_parts = [], [], []
        for leg_path, label in [(down_p, "DOWN"), (up_p, "UP")]:
            if os.path.exists(leg_path):
                d = np.load(leg_path)
                eta_parts.append(d["eta"]); I_parts.append(d["I_avg"]); s_parts.append(d["s_max"])
                print(f"  [{tag}] {label} leg: {len(d['eta'])} point(s), "
                      f"eta {d['eta'].min():.4f}-{d['eta'].max():.4f}")
            else:
                print(f"  [{tag}] {label} leg: not found yet -- plotting without it")
        if not eta_parts:
            continue
        eta_b = np.concatenate(eta_parts)
        I_b = np.concatenate(I_parts) / 1e4
        s_b = np.concatenate(s_parts)
        order = np.argsort(eta_b)
        seed_branches[tag] = (eta_b[order], I_b[order], s_b[order])

    orig_path = f"{ORIGINAL_BRANCH_BACKUP}/higheta_tau1.1_Lx0.2mm_coarse.npz"
    have_original = os.path.exists(orig_path)
    if have_original:
        d_orig = np.load(orig_path)
        eta_orig, I_orig, s_orig = d_orig["eta"], d_orig["I_avg"] / 1e4, d_orig["s_max"]
        print(f"Original branch: {len(eta_orig)} point(s), eta {eta_orig.min():.4f}-{eta_orig.max():.4f}")
    else:
        print(f"Original branch curve not found at {orig_path} -- plotting the seeded branch(es) alone.")

    ORIGINAL_COLOR = "#2a78d6"
    SEED_COLORS = ["#eb6834", "#1baf7a", "#e87ba4", "#eda100", "#4a3aa7", "#008300"]
    SEED_MARKERS = ["o", "s", "D", "^", "v", "P"]
    ETA_SOURCE = 1.28

    all_seed_etas = np.concatenate([seed_branches[t][0] for t in tags if t in seed_branches])
    zoom_lo = all_seed_etas.min() - 0.02
    zoom_hi = all_seed_etas.max() + 0.02

    fig, axes = plt.subplots(2, 2, figsize=(12, 9.2))
    (ax_I, ax_s), (ax_I_zoom, ax_s_zoom) = axes

    for i, tag in enumerate(tags):
        if tag not in seed_branches:
            continue
        eta_b, I_b, s_b = seed_branches[tag]
        color = SEED_COLORS[i % len(SEED_COLORS)]
        marker = SEED_MARKERS[i % len(SEED_MARKERS)]
        for ax_pair, y_b in [((ax_I, ax_I_zoom), I_b), ((ax_s, ax_s_zoom), s_b)]:
            for ax in ax_pair:
                ax.plot(eta_b, y_b, "-", marker=marker, color=color, ms=6, lw=1.8, label=f"seed {tag}")
        src_idx = int(np.argmin(np.abs(eta_b - ETA_SOURCE)))
        for ax, y_b in [(ax_I, I_b), (ax_I_zoom, I_b)]:
            ax.plot(eta_b[src_idx], y_b[src_idx], "*", color="black", ms=16, zorder=5)
        for ax, y_b in [(ax_s, s_b), (ax_s_zoom, s_b)]:
            ax.plot(eta_b[src_idx], y_b[src_idx], "*", color="black", ms=16, zorder=5)

    if have_original:
        for ax in (ax_I, ax_I_zoom):
            ax.plot(eta_orig, I_orig, "-o", color=ORIGINAL_COLOR, ms=4, lw=1.5, label="original branch")
        for ax in (ax_s, ax_s_zoom):
            ax.plot(eta_orig, s_orig, "-o", color=ORIGINAL_COLOR, ms=4, lw=1.5, label="original branch")

    ax_I.set_title("Current density vs. eta -- full range (black star = each seed's starting point)")
    ax_s.set_title("Saturation vs. eta -- full range")
    ax_I_zoom.set_title(f"Current density vs. eta -- zoomed to [{zoom_lo:.3f}, {zoom_hi:.3f}]")
    ax_s_zoom.set_title(f"Saturation vs. eta -- zoomed to [{zoom_lo:.3f}, {zoom_hi:.3f}]")
    for ax in (ax_I_zoom, ax_s_zoom):
        ax.set_xlim(zoom_lo, zoom_hi)
    for ax, ylab in [(ax_I, r"$I$ (A/cm$^2$)"), (ax_s, r"$s_{max}$"),
                     (ax_I_zoom, r"$I$ (A/cm$^2$)"), (ax_s_zoom, r"$s_{max}$")]:
        ax.set_xlabel(r"$\eta$ (V)")
        ax.set_ylabel(ylab)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    out_png = f"{OUT}/secondbranch_vs_original.png"
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nSaved to {out_png}")

    # Cross-seed comparison -- do the seeds' branches sit close to EACH OTHER (same branch reached
    # from different starting points) or stay apart (genuinely different branches)? This is the
    # direct answer to "are the plateaus the same branch or different ones".
    seed_tags_with_data = [t for t in tags if t in seed_branches]
    if len(seed_tags_with_data) >= 2:
        from numpy import interp
        for i in range(len(seed_tags_with_data)):
            for j in range(i + 1, len(seed_tags_with_data)):
                ta, tb = seed_tags_with_data[i], seed_tags_with_data[j]
                eta_a, I_a, s_a = seed_branches[ta]
                eta_b2, I_b2, s_b2 = seed_branches[tb]
                lo = max(eta_a.min(), eta_b2.min())
                hi = min(eta_a.max(), eta_b2.max())
                if hi > lo:
                    grid = np.linspace(lo, hi, 400)
                    gap = np.abs(interp(grid, eta_a, I_a) - interp(grid, eta_b2, I_b2))
                    verdict = ("SAME branch (small gap everywhere)" if gap.max() < 0.01 else
                               "different branches (persistent gap)" if gap.min() > 0.01 else
                               "inconclusive -- gap varies a lot over the shared range")
                    print(f"\n[{ta}] vs [{tb}]: over shared eta range [{lo:.4f}, {hi:.4f}], "
                          f"min |Delta I|={gap.min():.4f} A/cm^2, max |Delta I|={gap.max():.4f} "
                          f"A/cm^2 -- {verdict}")
                else:
                    print(f"\n[{ta}] vs [{tb}]: no overlapping eta range yet.")


## Download results

In [ ]:
import os, zipfile
from google.colab import files

OUT = "/content/pemfc_sensitivity_rh1p0_cont_results"
zip_path = "/content/pemfc_sensitivity_rh1p0_cont_results.zip"

# Update 17: two bugs fixed here together.
# (1) OUT previously pointed at "/content/pemfc_sensitivity_rh1p0_stability_results" -- Part A/B's own
# output directory, NOT this Part D section's own output ("_cont_results", written by the
# compute cell directly above, whose own OUT already uses that path). Confirmed by comparing
# against that compute cell's OUT: this download cell was silently zipping and handing back
# the WRONG results (Part A/B's checkpoints/eigenvalue_check.npz) instead of Part D's own
# secondbranch_up/down_*.npz and comparison plot -- with no error raised to reveal the mismatch
# (Part A/B's directory reliably exists and is non-empty by the time Part D runs, so the old
# bare assert below never caught it; only the wrong file names inside the downloaded zip
# would have given it away).
# (2) Same graceful-skip fix as Part C's download cell above, for when Part D itself hasn't
# produced anything to zip yet.
if not (os.path.isdir(OUT) and os.listdir(OUT)):
    print(f"No results found in {OUT} -- nothing to download yet. This is expected (not a "
          f"bug) when the compute cell above could not produce any result file for this "
          f"parameter value (see that cell's own diagnostic/log for the specific reason). Run "
          f"the cells above first, or re-run this cell once they've produced at least one "
          f"result file.")
else:
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fname in sorted(os.listdir(OUT)):
            zf.write(os.path.join(OUT, fname), fname)

    print(f"Zipped {len(os.listdir(OUT))} files ({os.path.getsize(zip_path)/1e3:.1f} KB)")
    files.download(zip_path)
